In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

try:
    import xgboost as xgb
except:
    !pip install xgboost
    import xgboost

try:
    import shap
except:
    !pip install shap
    import shap

try:
    from catboost import CatBoostClassifier
except:
    !pip install catboost
    from catboost import CatBoostClassifier
    
try:
    import plotly
except ImportError:
    import subprocess
    print("Installing plotly package...")
    subprocess.check_call(["pip", "install", "plotly"])
    import plotly
    
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
import time
import os
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from tqdm import tqdm

In [18]:
# To do: make this its own module

"""
Flexible stratification module for feature selection
Add this to your feature selection code
"""

import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer


def create_value_bins(y, n_bins=4):
    """
    Create quantile-based bins for continuous target values
    
    Parameters:
    -----------
    y : array-like
        Target values to bin
    n_bins : int
        Number of bins to create
        
    Returns:
    --------
    array : Binned values as integers
    """
    y_array = y.values if hasattr(y, 'values') else np.array(y)
    return KBinsDiscretizer(
        n_bins=n_bins, 
        encode='ordinal', 
        strategy='quantile'
    ).fit_transform(y_array.reshape(-1, 1)).flatten().astype(int)


def is_continuous_target(y, threshold=10):
    """
    Determine if target is continuous or categorical
    
    Parameters:
    -----------
    y : array-like
        Target values
    threshold : int
        Number of unique values above which target is considered continuous
        
    Returns:
    --------
    bool : True if continuous, False if categorical
    """
    n_unique = len(np.unique(y))
    return n_unique > threshold


def create_stratification(y, df=None, stratification_method='auto', 
                         class_column=None, n_bins=4, force_categorical=False):
    """
    Create stratification variable for train/test splitting
    
    Parameters:
    -----------
    y : array-like or pandas Series
        Target variable
    df : pandas DataFrame, optional
        Full dataframe containing class_column if needed
    stratification_method : str or None
        Stratification strategy. Options:
        - 'auto' or 'target': Auto-detect target type and stratify accordingly
          * Continuous targets: bin into quantiles (uses n_bins)
          * Categorical targets: stratify by class labels directly
        - 'value': Force binning into quantiles (treats target as continuous)
          * Uses n_bins to create quantile-based bins
          * Useful for continuous or ordinal targets
        - 'class': Stratify by metadata class column only
          * Requires class_column to be specified
          * Ignores target variable distribution
          * Example: stratify by phylum, organism type, etc.
        - 'class_and_value': Stratify by BOTH metadata class AND target
          * Requires class_column to be specified
          * For continuous targets: bins target first, then combines with class
          * For categorical targets: combines target classes with metadata class
          * Example: ensure both phylum distribution AND target balance in splits
        - None or 'none': NO stratification (simple random split)
          * Useful for benchmarking, small datasets, or already-balanced data
          * Train/test splits will be completely random (subject to random_state)
    class_column : str, optional
        Name of the metadata class column in df (e.g., 'phylum', 'species', 'group')
        Required for 'class' and 'class_and_value' methods
    n_bins : int, default=4
        Number of bins for continuous target stratification
        Only used when target is continuous or when using 'value' method
        Ignored for categorical targets
    force_categorical : bool, default=False
        If True, treats target as categorical even if it appears continuous
        Use when target is integer-encoded classes (0,1,2,3...) but represents
        discrete categories rather than continuous values
        
    Returns:
    --------
    stratify : array-like or None
        Stratification variable for sklearn's train_test_split
        Returns None if stratification_method is None or 'none'
        
    Raises:
    -------
    ValueError : If class-based stratification requested but class_column not provided
    KeyError : If class_column not found in df
    
    Examples:
    ---------
    # Auto-stratify by target (default)
    stratify = create_stratification(y, stratification_method='auto')
    
    # No stratification (random split)
    stratify = create_stratification(y, stratification_method=None)
    
    # Stratify by phylum only
    stratify = create_stratification(y, df, stratification_method='class', 
                                    class_column='phylum')
    
    # Stratify by both phylum and binary target (e.g., 85/15 split preserved per phylum)
    stratify = create_stratification(y, df, stratification_method='class_and_value',
                                    class_column='phylum')
    
    # Force treat numeric target as categorical
    stratify = create_stratification(y, stratification_method='auto',
                                    force_categorical=True)
    """
    if stratification_method is None or stratification_method == 'none':
        return None
    
    # Convert y to array if it's a Series
    y_array = y.values if hasattr(y, 'values') else np.array(y)
    
    # Determine if target is continuous
    is_continuous = is_continuous_target(y_array)
    
    if stratification_method == 'auto' or stratification_method == 'target':
        # Auto-stratify based on target type
        if is_continuous:
            return create_value_bins(y_array, n_bins)
        else:
            return y_array
    
    elif stratification_method == 'value':
        # Force binning regardless of target type
        return create_value_bins(y_array, n_bins)
    
    elif stratification_method == 'class':
        # Stratify by class column only
        if class_column is None:
            raise ValueError(
                "stratification_method='class' requires class_column to be specified"
            )
        if df is None:
            raise ValueError(
                "stratification_method='class' requires df to be provided"
            )
        if class_column not in df.columns:
            raise KeyError(
                f"class_column '{class_column}' not found in dataframe. "
                f"Available columns: {list(df.columns)}"
            )
        return df[class_column].values
    
    elif stratification_method == 'class_and_value':
        # Stratify by both class and target
        if class_column is None:
            raise ValueError(
                "stratification_method='class_and_value' requires class_column to be specified"
            )
        if df is None:
            raise ValueError(
                "stratification_method='class_and_value' requires df to be provided"
            )
        if class_column not in df.columns:
            raise KeyError(
                f"class_column '{class_column}' not found in dataframe. "
                f"Available columns: {list(df.columns)}"
            )
        
        # Get class values
        class_values = df[class_column].values
        
        # For continuous target: bin it first
        if is_continuous:
            value_bins = create_value_bins(y_array, n_bins)
            # Combine using string concatenation (sklearn handles this)
            stratify = np.array([f"{c}_{v}" for c, v in zip(class_values, value_bins)])
        else:
            # For categorical target: combine directly with class
            stratify = np.array([f"{c}_{v}" for c, v in zip(class_values, y_array)])
        
        return stratify
    
    else:
        raise ValueError(
            f"Unknown stratification_method: '{stratification_method}'. "
            f"Must be one of: 'auto', 'target', 'value', 'class', 'class_and_value', None"
        )


def stratified_sample_flexible(X, y, df, n_samples, stratification_method='auto',
                               class_column=None, n_bins=4, force_categorical=False, 
                               random_state=42):
    """
    Sample data while preserving stratification, with minimum samples per stratum
    
    Parameters:
    -----------
    X : array-like
        Feature matrix
    y : array-like
        Target values
    df : pandas DataFrame or None
        Full dataframe (needed if using class-based stratification)
    n_samples : int
        Total number of samples to draw
    stratification_method : str or None
        Stratification method (see create_stratification for options)
        Use None or 'none' for simple random sampling without stratification
    class_column : str, optional
        Name of the metadata class column
    n_bins : int
        Number of bins for continuous target
    force_categorical : bool
        If True, treats target as categorical even if numeric
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    array : Indices of sampled data points
    """
    # Create stratification variable
    stratify = create_stratification(
        y, df, stratification_method, class_column, n_bins
    )
    
    if stratify is None:
        # No stratification: simple random sampling
        np.random.seed(random_state)
        return np.random.choice(len(y), size=min(n_samples, len(y)), replace=False)
    
    # Get unique strata
    unique_strata = np.unique(stratify)
    min_samples_per_stratum = max(3, int(n_samples / (len(unique_strata) * 2)))
    
    sampled_indices = []
    for stratum in unique_strata:
        stratum_indices = np.where(stratify == stratum)[0]
        
        if len(stratum_indices) <= min_samples_per_stratum:
            # Include all samples from small strata
            sampled_indices.extend(stratum_indices)
        else:
            # Sample proportionally from larger strata
            n_to_sample = max(
                min_samples_per_stratum, 
                int(len(stratum_indices) / len(y) * n_samples)
            )
            n_to_sample = min(n_to_sample, len(stratum_indices))
            
            np.random.seed(random_state)
            sampled_from_stratum = np.random.choice(
                stratum_indices, size=n_to_sample, replace=False
            )
            sampled_indices.extend(sampled_from_stratum)
    
    return np.array(sampled_indices)

In [19]:
def create_interactive_gene_network(selected_features, interaction_matrix, scores, output_file="interactive_gene_network.html", threshold_percentile=90):
    """
    Create an interactive gene network visualization using Plotly
    
    Parameters:
    -----------
    selected_features : list
        Names of the selected genes
    interaction_matrix : numpy.ndarray
        Matrix of interaction strengths between genes
    scores : numpy.ndarray
        Pathogenicity scores for each gene
    output_file : str
        Path to save the HTML output file
    threshold_percentile : int
        Percentile threshold for including edges
    """
    import plotly.graph_objects as go
    import networkx as nx
    import numpy as np
    
    # Create a copy of the interaction matrix and set diagonal to 0
    interaction_matrix_copy = interaction_matrix.copy()
    np.fill_diagonal(interaction_matrix_copy, 0)
    
    # Calculate threshold for edges
    positive_interactions = interaction_matrix_copy[interaction_matrix_copy > 0]
    if len(positive_interactions) > 0:
        edge_threshold = np.percentile(positive_interactions, threshold_percentile)
    else:
        edge_threshold = 0.001
        print("No positive interactions found. Using default threshold of 0.001")
    
    # Create a network graph
    G = nx.Graph()
    
    # Add nodes
    for i, gene in enumerate(selected_features):
        # Create a shorter display name for the gene
        display_name = get_short_gene_name(gene)
        G.add_node(i, name=gene, display_name=display_name, score=scores[i])
    
    # Add edges based on interaction strengths
    for i in range(len(selected_features)):
        for j in range(i+1, len(selected_features)):
            interaction = interaction_matrix_copy[i, j]
            if interaction > edge_threshold:
                G.add_edge(i, j, weight=interaction)
    
    # Calculate layout (position of nodes)
    pos = nx.spring_layout(G, seed=42)
    
    # Extract node positions
    node_x = []
    node_y = []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
    
    # Create node trace
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        marker=dict(
            showscale=True,
            colorscale='Viridis',
            size=10,
            color=[G.nodes[node]['score'] for node in G.nodes()],
            colorbar=dict(
                thickness=15,
                #title='Pathogenicity Score',
                #xanchor='left',
                #titleside='right',
                title=dict(text='Pathogenicity Score', side='right'),
                xanchor='left'
            ),
            line=dict(width=2)
        )
    )
    
    # Create node hover text
    node_text = []
    for node in G.nodes():
        full_name = G.nodes[node]['name']
        score = G.nodes[node]['score']
        node_text.append(f"Gene: {full_name}<br>Score: {score:.4f}")
    
    node_trace.hovertext = node_text
    
    # Create edge traces
    edge_x = []
    edge_y = []
    edge_weights = []
    
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        edge_weights.append(G.edges[edge]['weight'])
    
    # Calculate edge width based on weight
    edge_width = [max(1, w * 5) for w in edge_weights]
    
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        mode='lines'
    )
    
    # Create figure
    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(
                        title=dict(
                            text='Gene Interaction Network - Pathogenicity Driving Genes',
                            font=dict(size=16)
                        ),
                        showlegend=False,
                        hovermode='closest',
                        margin=dict(b=20, l=5, r=5, t=40),
                        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                        width=1000,
                        height=800
                    ))
    
    # Save the plot as HTML file
    fig.write_html(output_file)
    print(f"Interactive network saved to {output_file}")

def get_short_gene_name(gene_string):
    """
    Extract a short displayable name from the full gene string
    
    Parameters:
    -----------
    gene_string : str
        Full gene name string with species identifiers
        
    Returns:
    --------
    str : A shorter, more displayable gene name
    """
    # Check if it's a group
    if gene_string.startswith('group_'):
        return gene_string.split('-')[0]
    
    # Check for multiple genes separated by ~~~
    if '~~~' in gene_string:
        parts = gene_string.split('~~~')
        # Take the first part
        gene_string = parts[0]
    
    # Check for species identifier pattern (sp-XXXXX-GENE_SPECIES)
    if gene_string.startswith('sp-'):
        parts = gene_string.split('-')
        if len(parts) >= 3:
            # Extract just the gene name part without species
            gene_part = parts[2].split('_')[0]
            return gene_part
        
    # If no specific pattern matched, return the first 10 characters
    return gene_string[:10]

In [20]:
def importance_driven_feature_selection_gpu(
    df_pa,  # Pandas DataFrame with sparse data
    target_column="Oxygen2",  # Column name containing pathogenicity scores
    n_features_to_select=20,
    scoring_strategy="equal_classes",  # scoring_strategy = "high_class_emphasis" | "equal_classes" | "rare_class_emphasis"
    high_importance_threshold=2,  # Classes >= this value are considered "important" - only works with 'high_class_emphasis'- classes need to be ranked ordinally in order of importance
    class_weights=None,
    random_state=42,
    stratification_method='target',  # 'auto'/'target', 'value', 'class', 'class_and_value', None
    class_column=None,  # Name of metadata class column for stratification
    n_stratification_bins=4,  # Number of bins for value-based stratification
    force_categorical=False,  # ADD THIS
    initial_features_per_model=2373,
    test_size=0.2,
    n_iterations=10,  # Multiple iterations to improve stability
    interaction_analysis=True,  # Enable gene-gene interaction analysis
    max_samples_for_interactions=75,  # Cap for interaction analysis (memory intensive)
    output_dir='./feature_selection_results',  # Directory to save results
    gpu_id=0,  # GPU device to use
    max_gpu_memory_fraction=0.8  # Max fraction of GPU memory to use
):
    """
    GPU-accelerated feature selection pipeline for sparse gene presence/absence data,
    prioritizing genes driving pathogenicity while accounting for gene-gene interactions.
    
    Parameters:
    -----------
    df_pa : pandas DataFrame
        Sparse dataframe with samples as rows and genes as columns
        Should include the target column (pathogenicity)
    target_column : str
        Name of the column containing pathogenicity values
    n_features_to_select : int
        Final number of features to select
    high_importance_threshold : int/float
        Threshold at or above which classes are considered pathogenic
    class_weights : dict, optional
        Custom weights for each class (higher weights for higher pathogenicity)
    random_state : int
        Random seed for reproducibility
    initial_features_per_model : int
        Number of features to pre-select in first round using model-based methods
    test_size : float
        Portion of data to use for validation
    n_iterations : int
        Number of times to repeat the analysis with different random seeds
    interaction_analysis : bool
        Whether to analyze gene-gene interactions (more memory intensive)
    max_samples_for_interactions : int
        Maximum number of samples to use for SHAP interaction analysis
    output_dir : str
        Directory to save output files
    gpu_id : int
        GPU device ID to use
    max_gpu_memory_fraction : float
        Maximum fraction of GPU memory to use
        
    Returns:
    --------
    selected_features : list
        Names of selected features (gene IDs)
    importance_df : DataFrame
        DataFrame with importance scores and interaction information for selected features
    """
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    import shap
    import xgboost as xgb
    from catboost import CatBoostClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.cluster import AgglomerativeClustering
    from sklearn.metrics.pairwise import cosine_similarity
    import time
    import os
    import matplotlib.pyplot as plt
    import seaborn as sns
    import networkx as nx
    import gc
    from tqdm import tqdm

    valid_strategies = ["high_class_emphasis", "equal_classes", "rare_class_emphasis"]
    if scoring_strategy not in valid_strategies:
        raise ValueError(f"scoring_strategy must be one of {valid_strategies}")
    
    # Try to import GPU-specific libraries
    try:
        import cupy as cp
        has_cupy = True
        print("CuPy found. GPU matrix operations enabled.")
    except ImportError:
        has_cupy = False
        print("CuPy not found. Using CPU for matrix operations.")
    
    # Try to import GPU memory monitoring
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(gpu_id)
        info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        total_gpu_memory = info.total / 1e9
        has_pynvml = True
        print(f"GPU memory monitoring enabled. Total GPU memory: {total_gpu_memory:.2f} GB")
    except:
        has_pynvml = False
        total_gpu_memory = 24.0  # Approximate for RTX 3090
        print("GPU memory monitoring not available. Assuming RTX 3090 with 24GB VRAM.")

            
    # Function to check available GPU memory
    def check_gpu_memory():
        """Return available GPU memory in GB"""
        if has_pynvml:
            info = pynvml.nvmlDeviceGetMemoryInfo(handle)
            free_memory_gb = info.free / 1e9
            return free_memory_gb
        else:
            # If monitoring not available, use a conservative estimate
            return 16.0  # Conservative default
    
    # Function to get adaptive sample size based on feature count and available memory
    def get_adaptive_sample_size(n_features, operation_type='shap'):
        """
        Calculate adaptive sample size based on feature count and available memory
        
        Parameters:
        -----------
        n_features : int
            Number of features
        operation_type : str
            Type of operation ('shap', 'interaction', 'similarity')
            
        Returns:
        --------
        int : Recommended sample size
        """
        available_gb = check_gpu_memory() * max_gpu_memory_fraction
        
        if operation_type == 'shap':
            # Basic SHAP calculation
            # Heuristic: each sample with n_features needs about n_features*8 bytes
            # SHAP needs roughly 3x this memory
            max_samples = int((available_gb * 1e9) / (n_features * 8 * 3))
            return max(30, min(max_samples, 200))  # Between 30-200 samples
        
        elif operation_type == 'interaction':
            # SHAP interaction calculation (O(n²) memory)
            # Much more memory intensive
            max_samples = int(np.sqrt((available_gb * 1e9) / (n_features * n_features * 8 * 2)))
            return max(10, min(max_samples, 75))  # Between 10-75 samples
        
        elif operation_type == 'similarity':
            # For similarity matrix (O(n²) memory for n genes)
            # Return whether to use GPU for this operation
            matrix_size_gb = (n_features * n_features * 8) / 1e9
            return matrix_size_gb < available_gb
        
        return 100  # Default fallback
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Setup logging file
    log_file = os.path.join(output_dir, "feature_selection_log.txt")
    def log_message(message):
        """Log message to both console and file"""
        print(message)
        with open(log_file, 'a') as f:
            f.write(message + "\n")
    
    log_message(f"Starting GPU-accelerated pathogenicity-driven feature selection on {df_pa.shape[1]-1} genes...")
    log_message(f"Analysis will include gene-gene interactions: {interaction_analysis}")
    log_message(f"Running {n_iterations} iterations for stability...")
    log_message(f"Available GPU memory: {check_gpu_memory():.2f} GB")
    
    total_start_time = time.time()
    
    # Separate features and target
    y_sparse = df_pa[target_column]
    # Convert target to regular numpy array if sparse
    if hasattr(y_sparse, 'sparse'):
        y = y_sparse.sparse.to_dense().values
    else:
        y = y_sparse.values
        
    X_df = df_pa.drop(columns=[target_column])
    feature_names = X_df.columns.tolist()
    
    # Encode target if it's not already numeric
    if not np.issubdtype(y.dtype, np.number):
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(y)
        class_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
        log_message(f"Encoded {len(label_encoder.classes_)} classes: {class_mapping}")
    else:
        # Create class mapping for numeric values
        unique_classes = sorted(np.unique(y))
        class_mapping = {cls: idx for idx, cls in enumerate(unique_classes)}
        log_message(f"Using {len(unique_classes)} numeric classes: {class_mapping}")
        
        # Create a label encoder for consistency
        label_encoder = LabelEncoder()
        label_encoder.classes_ = np.array(unique_classes)

    is_binary = len(np.unique(y)) == 2
    if is_binary:
        log_message("Binary classification detected. Adjusting analysis approach...")
        
        if scoring_strategy == "high_class_emphasis":
            # Assume class 1 is "high importance"
            if high_importance_threshold is None:
                high_importance_threshold = 1
            log_message("Binary + high_class_emphasis: Focusing on positive class (class 1)")
        
        elif scoring_strategy == "equal_classes":
            log_message("Binary + equal_classes: Treating both classes equally")
        
        elif scoring_strategy == "rare_class_emphasis":
            log_message("Binary + rare_class_emphasis: Emphasizing minority class")
    
    if scoring_strategy == "high_class_emphasis":
        high_importance_classes = [i for i, c in enumerate(label_encoder.classes_) 
                                 if c >= high_importance_threshold]
    else:
        # For other strategies, consider all classes as "important"
        high_importance_classes = list(range(len(label_encoder.classes_)))
    
    # If no custom weights, create default weights that emphasize pathogenic classes
    if class_weights is None:
        if scoring_strategy == "high_class_emphasis":
            if is_binary:
                class_weights = {0: 1.0, 1: 3.0}
            else:
                max_class = len(np.unique(y)) - 1
                class_weights = {i: np.exp(i/max_class * 1.5) for i in range(len(np.unique(y)))}
        
        elif scoring_strategy == "equal_classes":
            class_weights = {i: 1.0 for i in range(len(np.unique(y)))}
        
        elif scoring_strategy == "rare_class_emphasis":
            # Inverse frequency weighting
            class_counts = np.bincount(y)
            total_samples = len(y)
            class_weights = {i: total_samples / (len(class_counts) * count) 
                            for i, count in enumerate(class_counts)}
    
    # Function for stratified sampling that ensures representation of rare classes
    def stratified_sample(X, y, n_samples, random_state=42):
        """Sample data while preserving class distribution, with minimum samples per class"""
        unique_classes = np.unique(y)
        min_samples_per_class = max(5, int(n_samples / (len(unique_classes) * 2)))
        
        sampled_indices = []
        for cls in unique_classes:
            cls_indices = np.where(y == cls)[0]
            
            if len(cls_indices) <= min_samples_per_class:
                sampled_indices.extend(cls_indices)
            else:
                n_to_sample = max(min_samples_per_class, 
                                int(len(cls_indices) / len(y) * n_samples))
                n_to_sample = min(n_to_sample, len(cls_indices))
                
                np.random.seed(random_state)
                sampled_from_cls = np.random.choice(cls_indices, size=n_to_sample, replace=False)
                sampled_indices.extend(sampled_from_cls)
        
        return np.array(sampled_indices)
    
    # Initialize storage for multi-iteration results
    all_iterations_scores = {}
    all_iterations_clusters = {}
    all_iterations_interactions = {}
    
    # Run multiple iterations with different random seeds for stability
    for iteration in range(n_iterations):
        current_seed = random_state + iteration
        log_message(f"\n{'='*80}\nIteration {iteration+1}/{n_iterations} (seed: {current_seed})\n{'='*80}")
        start_time = time.time()
        
        # Create stratification for this iteration
        stratify = create_stratification(
            y, df_pa, stratification_method, class_column, n_stratification_bins, force_categorical
        )
        
        # Split data with current seed
        X_train_df, X_test_df, y_train, y_test = train_test_split(
            X_df, y, test_size=test_size, random_state=current_seed, stratify=stratify
        )
        
        log_message(f"Training set: {X_train_df.shape}, Test set: {X_test_df.shape}")
        log_message(f"Class distribution in training: {np.bincount(y_train)}")
        
        # STAGE 1: Initial feature pre-selection using model-based importance
        log_message(f"STAGE 1: Pre-selecting features...")
        
        # Define models for initial screening
        objective = 'binary:logistic' if is_binary else 'multi:softmax'
        models = {
            'xgb': xgb.XGBClassifier(
                n_estimators=100, 
                learning_rate=0.1,
                max_depth=6,
                objective=objective,
                num_class=None if is_binary else len(np.unique(y)),
                tree_method='hist',      # Updated
                device='cuda',           # Updated - use CUDA instead of gpu_id
                random_state=current_seed
            ),
            'catboost': CatBoostClassifier(
                iterations=100,
                depth=6,
                learning_rate=0.1,
                loss_function='MultiClass',
                task_type='GPU',        
                devices=f'{gpu_id}',    
                random_seed=current_seed,
                verbose=False,
                gpu_ram_part=0.95,      # Controls GPU memory usage
                od_type='Iter',         # Early stopping type
                od_wait=20              # Stop if no improvement for 20 iterations
            ),
            'rf': RandomForestClassifier(
                n_estimators=100,
                max_depth=6,
                n_jobs=-1,
                random_state=current_seed
            )
        }
        
        # Initialize to store feature importance across models
        feature_importances = {}
        
        # Train each model and calculate feature importance
        for name, model in models.items():
            log_message(f"Training {name} model for initial feature screening...")
            
            # Convert sparse dataframe to array for model training
            try:
                # Memory-efficient conversion
                if hasattr(X_train_df, 'sparse'):
                    log_message("Converting sparse matrix to dense for model training...")
                    X_train_array = X_train_df.sparse.to_dense().values
                else:
                    X_train_array = X_train_df.values
                
                # Train model
                model.fit(X_train_array, y_train)
                
                # Clear memory after training
                gc.collect()
                if has_cupy:
                    cp.get_default_memory_pool().free_all_blocks()
                
            except MemoryError:
                log_message(f"Memory error with full conversion. Switching to batch processing...")
                
                # Train in batches
                batch_size = 1000  # Adjust based on your available memory
                total_features = X_train_df.shape[1]
                
                # Train on initial subset
                X_subset = X_train_df.iloc[:, :batch_size]
                if hasattr(X_subset, 'sparse'):
                    X_subset_array = X_subset.sparse.to_dense().values
                else:
                    X_subset_array = X_subset.values
                    
                model.fit(X_subset_array, y_train)
                
                # Continue adding features in batches
                for start_idx in range(batch_size, total_features, batch_size):
                    end_idx = min(start_idx + batch_size, total_features)
                    X_batch = X_train_df.iloc[:, start_idx:end_idx]
                    
                    if hasattr(X_batch, 'sparse'):
                        X_batch_array = X_batch.sparse.to_dense().values
                    else:
                        X_batch_array = X_batch.values
                    
                    # Update model with new features
                    model.fit(np.hstack([X_subset_array, X_batch_array]), y_train)
                    
                    # Update subset
                    X_subset = pd.concat([X_subset, X_batch], axis=1)
                    X_subset_array = X_subset.sparse.to_dense().values if hasattr(X_subset, 'sparse') else X_subset.values
                    
                    # Clear memory
                    gc.collect()
                    if has_cupy:
                        cp.get_default_memory_pool().free_all_blocks()
            
            # Get feature importances
            if name == 'xgb':
                importances = model.feature_importances_
            elif name == 'catboost':
                importances = model.feature_importances_
            else:  # Random Forest
                importances = model.feature_importances_
            
            feature_importances[name] = importances
        
        # Average the feature importances across models
        avg_importance = np.zeros(X_df.shape[1])
        for imp in feature_importances.values():
            avg_importance += imp
        avg_importance /= len(feature_importances)
        
        # Select top features from initial screening
        initial_selected_indices = np.argsort(avg_importance)[-initial_features_per_model:]
        initial_selected_features = [feature_names[i] for i in initial_selected_indices]
        
        log_message(f"Selected {len(initial_selected_features)} features in initial screening")
        
        # Create reduced dataframes with only pre-selected features
        X_train_reduced_df = X_train_df[initial_selected_features]
        X_test_reduced_df = X_test_df[initial_selected_features]
        
        # Convert to dense for further analysis
        if hasattr(X_train_reduced_df, 'sparse'):
            X_train_reduced = X_train_reduced_df.sparse.to_dense().values
        else:
            X_train_reduced = X_train_reduced_df.values
            
        if hasattr(X_test_reduced_df, 'sparse'):
            X_test_reduced = X_test_reduced_df.sparse.to_dense().values
        else:
            X_test_reduced = X_test_reduced_df.values
        
        # Clear memory
        gc.collect()
        if has_cupy:
            cp.get_default_memory_pool().free_all_blocks()
        
        # STAGE 2: Cluster genes to identify potential interaction groups
        log_message("STAGE 2: Clustering genes based on co-occurrence patterns...")
        
        # Calculate gene similarity - decide whether to use GPU based on matrix size
        use_gpu_for_similarity = has_cupy and get_adaptive_sample_size(
            len(initial_selected_features), 'similarity')
        
        if use_gpu_for_similarity:
            log_message("Calculating gene similarity matrix on GPU...")
            # Transfer data to GPU
            X_gpu = cp.array(X_train_reduced.T)
            
            # Calculate similarity matrix
            X_norm = cp.linalg.norm(X_gpu, axis=1)[:, cp.newaxis]
            X_normalized = X_gpu / cp.maximum(X_norm, 1e-10)
            similarity_gpu = cp.dot(X_normalized, X_normalized.T)
            
            # Transfer back to CPU
            gene_similarity = cp.asnumpy(similarity_gpu)
            
            # Free GPU memory
            del X_gpu, X_norm, X_normalized, similarity_gpu
            cp.get_default_memory_pool().free_all_blocks()
        else:
            log_message("Calculating gene similarity matrix on CPU...")
            gene_similarity = cosine_similarity(X_train_reduced.T)
        
        # Save similarity matrix visualization
        plt.figure(figsize=(10, 8))
        sns.heatmap(gene_similarity[:100, :100], cmap='viridis')  # First 100x100 for visibility
        plt.title(f"Gene Similarity Matrix (First 100 Genes) - Iteration {iteration+1}")
        plt.tight_layout()
        plt.savefig(f"{output_dir}/gene_similarity_iter{iteration+1}.png", dpi=150)
        plt.close()
        
        # Cluster genes
        n_clusters = min(50, len(initial_selected_features))
        log_message(f"Clustering genes into {n_clusters} groups...")
        clustering = AgglomerativeClustering(n_clusters=n_clusters)
        gene_clusters = clustering.fit_predict(gene_similarity)
        
        # Map genes to clusters
        gene_cluster_map = {gene: cluster for gene, cluster in zip(initial_selected_features, gene_clusters)}
        
        # STAGE 3: Directional SHAP analysis on pre-selected features
        log_message("STAGE 3: Performing directional SHAP analysis...")
        
        # Get adaptive sample size based on feature count and available memory
        max_shap_samples = get_adaptive_sample_size(len(initial_selected_features), 'shap')
        log_message(f"Using adaptive sample size of {max_shap_samples} for SHAP analysis...")
        
        # Sample test data with stratification
        # For SHAP sampling
        sample_indices = stratified_sample_flexible(
            X_test_reduced, y_test, df_pa.loc[X_test_df.index], max_shap_samples,
            stratification_method, class_column, n_stratification_bins, force_categorical, current_seed
        )
        X_shap_samples = X_test_reduced[sample_indices]
        y_shap_samples = y_test[sample_indices]
        
        # Verify class distribution
        class_counts = np.bincount(y_shap_samples, minlength=len(np.unique(y)))
        log_message(f"Class distribution in SHAP samples: {class_counts}")
        
        # Store interaction information
        interaction_scores = {}
        
        # Store pathogenicity scores
        importance_scores = {}
        
        # Analyze with each model
        for name, model_class in models.items():
            log_message(f"Training {name} model for directional SHAP analysis...")
            
            # Create a new model instance with more trees for stability
            if name == 'catboost':
                model = CatBoostClassifier(
                    iterations=300,
                    depth=6,
                    learning_rate=0.05,
                    loss_function='MultiClass',
                    task_type='GPU',
                    devices=f'{gpu_id}',
                    gpu_ram_part=0.95,     # Add this
                    random_seed=current_seed,
                    verbose=False
                )
            elif name == 'xgb':
                objective = 'binary:logistic' if is_binary else 'multi:softmax'
                model = xgb.XGBClassifier(
                    n_estimators=300,
                    max_depth=6,
                    learning_rate=0.05,
                    objective=objective,
                    num_class=None if is_binary else len(np.unique(y)),
                    tree_method='hist',    # Updated
                    device='cuda',         # Updated
                    random_state=current_seed
                )
            else:  # Random Forest
                model = RandomForestClassifier(
                    n_estimators=300,
                    max_depth=8,
                    n_jobs=-1,
                    random_state=current_seed
                )
            
            # Train the model
            model.fit(X_train_reduced, y_train)
            
            # Calculate SHAP values
            log_message(f"Calculating SHAP values for {name}...")
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_shap_samples)
            # DEBUG: Print SHAP values information
            log_message(f"DEBUG: Raw SHAP values type: {type(shap_values)}")
            if isinstance(shap_values, list):
                log_message(f"DEBUG: SHAP values list length: {len(shap_values)}")
                for i, arr in enumerate(shap_values):
                    log_message(f"DEBUG: SHAP array {i} shape: {arr.shape}")
            else:
                log_message(f"DEBUG: Single SHAP array shape: {shap_values.shape}")
            log_message(f"DEBUG: Expected shape should be: {X_shap_samples.shape}")
            log_message(f"DEBUG: Number of unique classes: {len(np.unique(y))}")
            log_message(f"DEBUG: Is binary: {len(np.unique(y)) == 2}")
            # ROBUST SHAP NORMALIZATION - HANDLE ALL MODEL TYPES
            expected_features = X_shap_samples.shape[1]
            expected_samples = X_shap_samples.shape[0]
            
            if isinstance(shap_values, list):
                # Already a list, validate each array
                valid_shap_arrays = []
                for i, shap_array in enumerate(shap_values):
                    if len(shap_array.shape) == 2 and shap_array.shape == (expected_samples, expected_features):
                        valid_shap_arrays.append(shap_array)
                        log_message(f"DEBUG: Class {i} SHAP array is valid")
                    else:
                        log_message(f"DEBUG: Class {i} SHAP array invalid shape {shap_array.shape}, skipping")
                
                if len(valid_shap_arrays) == 0:
                    log_message("ERROR: No valid SHAP arrays found, skipping this model")
                    continue
                
                shap_values = valid_shap_arrays
            else:
                # Single array case - handle different formats
                if len(shap_values.shape) == 2 and shap_values.shape == (expected_samples, expected_features):
                    # XGBoost binary format: (samples, features)
                    if len(np.unique(y)) == 2:
                        shap_values = [np.zeros_like(shap_values), shap_values]
                        log_message("DEBUG: Converted XGBoost binary SHAP to list format")
                    else:
                        shap_values = [shap_values]
                        log_message("DEBUG: Converted single SHAP to list format")
                
                elif len(shap_values.shape) == 3 and shap_values.shape == (expected_samples, expected_features, len(np.unique(y))):
                    # Random Forest/CatBoost binary format: (samples, features, classes)
                    log_message("DEBUG: Detected 3D SHAP array (RF/CatBoost format)")
                    shap_list = []
                    for class_idx in range(shap_values.shape[2]):
                        shap_list.append(shap_values[:, :, class_idx])
                        log_message(f"DEBUG: Extracted class {class_idx} SHAP array shape: {shap_values[:, :, class_idx].shape}")
                    shap_values = shap_list
                    log_message("DEBUG: Converted 3D SHAP to list format")
                
                else:
                    log_message(f"ERROR: Invalid SHAP array shape {shap_values.shape}, skipping this model")
                    continue
            
            # Final validation
            log_message(f"DEBUG: Final SHAP list length: {len(shap_values)}")
            for i, arr in enumerate(shap_values):
                log_message(f"DEBUG: Final SHAP array {i} shape: {arr.shape}")
            
            # Clear some memory
            gc.collect()
            
            if has_cupy:
                cp.get_default_memory_pool().free_all_blocks()
            
            # Process SHAP values for pathogenicity
            if isinstance(shap_values, list):
                if scoring_strategy == "high_class_emphasis":
                    # Current logic - emphasize high classes
                    high_importance_contribution = np.zeros(X_shap_samples.shape[1])
                    non_high_importance_contribution = np.zeros(X_shap_samples.shape[1])
                    class_contribution = np.zeros(X_shap_samples.shape[1])  # Initialize here
                    
                    for class_idx in range(len(shap_values)):
                        weight = class_weights.get(class_idx, 1.0)
                        positive_contribution = np.where(shap_values[class_idx] > 0, 
                                                      shap_values[class_idx], 0).mean(axis=0)
                        
                        if class_idx in high_importance_classes:
                            high_importance_contribution += positive_contribution
                        else:
                            non_high_importance_contribution += positive_contribution
                        
                        # Add weighted contribution
                        class_contribution += weight * positive_contribution
                    
                    importance_score = high_importance_contribution - non_high_importance_contribution
                
                elif scoring_strategy == "equal_classes":
                    # Sum all positive contributions equally
                    importance_score = np.zeros(X_shap_samples.shape[1])
                    class_contribution = np.zeros(X_shap_samples.shape[1])  # Initialize here
                    
                    for class_idx in range(len(shap_values)):
                        positive_contribution = np.where(shap_values[class_idx] > 0, 
                                                      shap_values[class_idx], 0).mean(axis=0)
                        importance_score += positive_contribution
                        class_contribution += positive_contribution  # Same as importance_score for equal weighting
                
                elif scoring_strategy == "rare_class_emphasis":
                    # Weight by inverse class frequency
                    importance_score = np.zeros(X_shap_samples.shape[1])
                    class_contribution = np.zeros(X_shap_samples.shape[1])  # Initialize here
                    class_counts = np.bincount(y)
                    
                    for class_idx in range(len(shap_values)):
                        weight = len(y) / class_counts[class_idx]  # Inverse frequency
                        positive_contribution = np.where(shap_values[class_idx] > 0, 
                                                      shap_values[class_idx], 0).mean(axis=0)
                        weighted_contribution = weight * positive_contribution
                        importance_score += weighted_contribution
                        class_contribution += weighted_contribution  # Same as importance_score for this strategy
                
                # Store scores
                importance_scores[f"{name}_importance_score"] = importance_score
                importance_scores[f"{name}_weighted_score"] = class_contribution
                
                # Save SHAP summary plot for each model
                plt.figure(figsize=(10, 12))
                shap.summary_plot(shap_values, X_shap_samples, plot_type="bar", 
                                  feature_names=initial_selected_features, show=False)
                plt.title(f"{name.upper()} SHAP Summary - Iteration {iteration+1}")
                plt.tight_layout()
                plt.savefig(f"{output_dir}/{name}_shap_summary_iter{iteration+1}.png", dpi=200)
                plt.close()
                
                # Calculate gene-gene interactions if enabled and memory available
                if interaction_analysis and name in ['xgb', 'rf']:
                    try:
                        # Check if we have enough memory for interaction analysis
                        avail_mem = check_gpu_memory()
                        log_message(f"Available GPU memory before interaction analysis: {avail_mem:.2f} GB")
                        
                        if avail_mem < 5.0 and name == 'xgb':
                            log_message(f"Not enough GPU memory for XGBoost interaction values. Creating CPU model...")
                            # Create CPU version of the model for interaction analysis
                            cpu_model_params = model.get_params()
                            cpu_model_params['tree_method'] = 'hist'  # CPU version
                            if 'gpu_id' in cpu_model_params:
                                del cpu_model_params['gpu_id']
                            if 'predictor' in cpu_model_params:
                                del cpu_model_params['predictor']
                            
                            cpu_model = xgb.XGBClassifier(**cpu_model_params)
                            cpu_model.fit(X_train_reduced, y_train)
                            cpu_explainer = shap.TreeExplainer(cpu_model)
                            use_model_for_interactions = cpu_explainer
                        else:
                            use_model_for_interactions = explainer
                        
                        # Get adaptive sample size for interactions (much more memory intensive)
                        interaction_sample_size = get_adaptive_sample_size(
                            len(initial_selected_features), 'interaction')
                        interaction_sample_size = min(interaction_sample_size, 
                                                    max_samples_for_interactions,
                                                    X_shap_samples.shape[0])
                        
                        log_message(f"Calculating interaction values for {name} with {interaction_sample_size} samples...")
                        
                        # Sample for interaction analysis
                        interaction_indices = np.random.choice(X_shap_samples.shape[0], 
                                                             size=interaction_sample_size, 
                                                             replace=False)
                        interaction_samples = X_shap_samples[interaction_indices]
                        
                        # Calculate SHAP interaction values
                        interaction_values = use_model_for_interactions.shap_interaction_values(interaction_samples)
                        
                        # Process interaction values for pathogenic classes
                        if isinstance(interaction_values, list):
                            # Initialize interaction matrix
                            gene_interactions = np.zeros((X_shap_samples.shape[1], X_shap_samples.shape[1]))
                            
                            # Sum positive interactions toward pathogenic classes
                            if scoring_strategy == "high_class_emphasis":
                                target_classes = high_importance_classes
                            else:
                                target_classes = list(range(len(interaction_values)))
                            
                            for class_idx in target_classes:
                                if class_idx < len(interaction_values):
                                    pos_interactions = np.where(interaction_values[class_idx] > 0, 
                                                              interaction_values[class_idx], 0)
                                    gene_interactions += pos_interactions.mean(axis=0)
                            
                            # Store interaction scores
                            interaction_scores[f"{name}_interactions"] = gene_interactions
                            
                            # Find top interactions
                            np.fill_diagonal(gene_interactions, 0)  # Remove self-interactions
                            interaction_sums = gene_interactions.sum(axis=1)
                            
                            # Top interacting genes
                            top_interacting_idx = np.argsort(interaction_sums)[-50:]
                            top_interacting_genes = [initial_selected_features[i] for i in top_interacting_idx]
                            
                            log_message(f"Top interacting genes for {name}: {top_interacting_genes[:10]}")
                            
                            # Save interaction matrix visualization
                            plt.figure(figsize=(12, 10))
                            
                            # Use top 50 interacting genes for a clearer visualization
                            interaction_subset = gene_interactions[top_interacting_idx][:, top_interacting_idx]
                            
                            sns.heatmap(interaction_subset, cmap='inferno', 
                                        xticklabels=[initial_selected_features[i] for i in top_interacting_idx],
                                        yticklabels=[initial_selected_features[i] for i in top_interacting_idx])
                            plt.title(f"{name.upper()} Gene Interaction Matrix - Top 50 Genes")
                            plt.xticks(rotation=90)
                            plt.yticks(rotation=0)
                            plt.tight_layout()
                            plt.savefig(f"{output_dir}/{name}_interactions_iter{iteration+1}.png", dpi=200)
                            plt.close()
                            
                            # Clear memory
                            del interaction_values, interaction_samples
                            if 'cpu_model' in locals():
                                del cpu_model, cpu_explainer
                            gc.collect()
                            if has_cupy:
                                cp.get_default_memory_pool().free_all_blocks()
                    except Exception as e:
                        log_message(f"Error calculating interaction values for {name}: {str(e)}")
                        log_message("Will continue without interaction values for this model.")
            else:
                log_message(f"Warning: SHAP values for {name} not in expected format. Skipping.")
        
        # Calculate average scores across models
        path_score_cols = [k for k in importance_scores.keys() if k.endswith('_importance_score')]
        weighted_score_cols = [k for k in importance_scores.keys() if k.endswith('_weighted_score')]
        
        # Initialize arrays for averaging
        avg_path_score = np.zeros(len(initial_selected_features))
        avg_weighted_score = np.zeros(len(initial_selected_features))
        
        # Average the scores from different models
        for col in path_score_cols:
            avg_path_score += importance_scores[col]
        avg_path_score /= len(path_score_cols) if path_score_cols else 1
        
        for col in weighted_score_cols:
            avg_weighted_score += importance_scores[col]
        avg_weighted_score /= len(weighted_score_cols) if weighted_score_cols else 1
        
        # Create combined score (both directional and weighted)
        combined_score = (avg_path_score + avg_weighted_score) / 2
        
        # Store scores for this iteration
        all_iterations_scores[iteration] = {
            'importance_score': avg_path_score,
            'weighted_score': avg_weighted_score,
            'combined_score': combined_score
        }
        
        # Store clusters for this iteration
        all_iterations_clusters[iteration] = gene_clusters
        
        # Store interactions if calculated
        if interaction_scores:
            # Average interaction matrix across models
            avg_interaction = np.zeros((X_shap_samples.shape[1], X_shap_samples.shape[1]))
            for matrix in interaction_scores.values():
                avg_interaction += matrix
            avg_interaction /= len(interaction_scores)
            
            all_iterations_interactions[iteration] = avg_interaction
        
        elapsed_time = time.time() - start_time
        log_message(f"Iteration {iteration+1} complete! Time taken: {elapsed_time:.1f} seconds")
        
        # Clear memory between iterations
        gc.collect()
        if has_cupy:
            cp.get_default_memory_pool().free_all_blocks()
    
# STAGE 4: Aggregate results across iterations
    log_message("\nSTAGE 4: Aggregating results across all iterations...")
    
    # Initialize array to store aggregated importance scores
    aggregated_scores = np.zeros(len(initial_selected_features))
    
    # Average scores across iterations
    for iteration_scores in all_iterations_scores.values():
        aggregated_scores += iteration_scores['combined_score']
    aggregated_scores /= n_iterations
    
    # Select final features using aggregated scores
    selected_indices = np.argsort(aggregated_scores)[-n_features_to_select:]
    
    # Get the selected feature names
    selected_features = [initial_selected_features[i] for i in selected_indices]
    
    # Create results DataFrame with detailed scores
    results = pd.DataFrame({
        'Feature': selected_features,
        'Aggregated_Score': aggregated_scores[selected_indices],
    })
    
    # Add individual iteration scores
    for iteration in range(n_iterations):
        results[f'Iteration_{iteration+1}_Score'] = all_iterations_scores[iteration]['combined_score'][selected_indices]
    
    # Add cluster information (from the first iteration for simplicity)
    results['Gene_Cluster'] = [gene_cluster_map.get(gene, -1) for gene in selected_features]
    
    # Add interaction information if available
    if all_iterations_interactions:
        # Calculate average interaction strength for each gene
        avg_interaction_matrix = np.zeros((len(initial_selected_features), len(initial_selected_features)))
        for matrix in all_iterations_interactions.values():
            avg_interaction_matrix += matrix
        avg_interaction_matrix /= len(all_iterations_interactions)
        
        # Calculate total interaction strength for each gene (sum of row)
        interaction_strength = avg_interaction_matrix.sum(axis=1)
        
        # Add to results
        results['Interaction_Strength'] = interaction_strength[selected_indices]
        
        # Find top interaction partners for each selected gene
        top_partners = []
        for idx in selected_indices:
            gene_interactions = avg_interaction_matrix[idx, :]
            partner_indices = np.argsort(gene_interactions)[-5:]  # Top 5 partners
            partners = [initial_selected_features[i] for i in partner_indices if gene_interactions[i] > 0]
            top_partners.append(', '.join(partners) if partners else 'None')
        
        results['Top_Interaction_Partners'] = top_partners
        
        # Create gene interaction network visualization
        log_message("Creating gene interaction network visualization...")
        G = nx.Graph()
        
        # Add nodes for top genes
        for gene in selected_features:
            idx = initial_selected_features.index(gene)
            G.add_node(gene, score=aggregated_scores[idx])
        
        # Add edges based on interaction strengths
        if np.sum(avg_interaction_matrix > 0) > 0:
            # There are positive values, proceed normally
            edge_threshold = np.percentile(avg_interaction_matrix[avg_interaction_matrix > 0], 90)
            log_message(f"Using edge threshold of {edge_threshold:.4f} (90th percentile of non-zero interactions)")
        else:
            # No positive interactions found, use a small positive value
            edge_threshold = 0.001
            log_message("No positive interactions found. Using default threshold of 0.001")
        
        for i, gene1 in enumerate(selected_features):
            idx1 = initial_selected_features.index(gene1)
            for j, gene2 in enumerate(selected_features):
                if i < j:  # Avoid duplicate edges
                    idx2 = initial_selected_features.index(gene2)
                    interaction = avg_interaction_matrix[idx1, idx2]
                    if interaction > edge_threshold:
                        G.add_edge(gene1, gene2, weight=interaction)
        
        # Draw the network if it has edges
        if nx.number_of_edges(G) > 0:
            # Save static version for the report
            plt.figure(figsize=(15, 15))

            # Position nodes using force-directed layout
            pos = nx.spring_layout(G, seed=42)

            # Get node colors based on scores
            node_colors = [G.nodes[n]['score'] for n in G.nodes()]

            # Get edge weights
            edge_weights = [G.edges[e]['weight'] * 5 for e in G.edges()]

            # Draw nodes
            nodes = nx.draw_networkx_nodes(G, pos, node_size=500, node_color=node_colors, 
                                         alpha=0.8, cmap=plt.cm.viridis)

            # Draw edges
            edges = nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.5, 
                                         edge_color='gray')

            # Draw without labels for the static version
            plt.title("Gene Interaction Network - Pathogenicity Driving Genes")
            plt.axis('off')
            plt.colorbar(nodes, label="Pathogenicity Score")
            plt.tight_layout()
            plt.savefig(f"{output_dir}/gene_interaction_network.png", dpi=300)
            plt.close()

            # Create interactive version with hoverable labels
            log_message("Creating interactive gene network visualization...")
            create_interactive_gene_network(
                selected_features=selected_features,
                interaction_matrix=avg_interaction_matrix,
                scores=aggregated_scores[selected_indices],
                output_file=f"{output_dir}/interactive_gene_network.html"
            )
        else:
            log_message("No edges above threshold for interaction network visualization")
        
    
    # Sort by aggregated score
    results = results.sort_values('Aggregated_Score', ascending=False)
    
    # Save final results
    results.to_csv(f"{output_dir}/selected_important_features.csv", index=False)
    
    # Create a separate detailed table with per-class information for top genes
    if n_iterations > 0 and isinstance(shap_values, list):
        log_message("Creating detailed per-class analysis for top genes...")
        
        # Use the last iteration's SHAP values for simplicity
        detailed_results = []
        
        for gene_idx, gene_name in enumerate(initial_selected_features):
            if gene_name in selected_features:
                gene_data = {'Gene': gene_name}
                
                # Add per-class contributions
                for class_idx in range(len(shap_values)):
                    class_name = label_encoder.classes_[class_idx]
                    # Average positive SHAP values for this class
                    positive_contrib = np.where(shap_values[class_idx][:, gene_idx] > 0, 
                                             shap_values[class_idx][:, gene_idx], 0).mean()
                    gene_data[f'Push_to_Class_{class_name}'] = positive_contrib
                
                detailed_results.append(gene_data)
        
        # Convert to DataFrame and save
        detailed_df = pd.DataFrame(detailed_results)
        detailed_df.to_csv(f"{output_dir}/detailed_class_contributions.csv", index=False)
    
    # Create an HTML report with key findings
    try:
        log_message("Creating HTML report with key findings...")
        
        html_report = f"""
        <html>
        <head>
            <title>NTM Pathogenicity Gene Selection Report</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; }}
                h1, h2, h3 {{ color: #2c3e50; }}
                table {{ border-collapse: collapse; width: 100%; margin-bottom: 20px; }}
                th, td {{ text-align: left; padding: 8px; border-bottom: 1px solid #ddd; }}
                th {{ background-color: #f2f2f2; }}
                tr:hover {{ background-color: #f5f5f5; }}
                .summary {{ background-color: #eef7fa; padding: 15px; border-radius: 5px; }}
                img {{ max-width: 100%; border: 1px solid #ddd; margin: 10px 0; }}
            </style>
        </head>
        <body>
            <h1>NTM Pathogenicity Gene Selection Report</h1>
            
            <div class="summary">
                <h2>Summary</h2>
                <p>Analysis completed on {time.strftime('%Y-%m-%d %H:%M:%S')}</p>
                <p>Selected {len(selected_features)} genes from {len(feature_names)} total genes that drive pathogenicity in NTMs.</p>
                <p>Analysis used {n_iterations} iterations with different random seeds to ensure stability.</p>
                <p>Gene selection focused on genes that push specifically toward pathogenic classes (>= {high_importance_threshold}).</p>
            </div>
            
            <h2>Top 20 Pathogenicity-Driving Genes</h2>
            <table>
                <tr>
                    <th>Rank</th>
                    <th>Gene</th>
                    <th>Pathogenicity Score</th>
                    <th>Cluster</th>
        """
        
        # Add top 20 genes to table
        for i, (_, row) in enumerate(results.head(20).iterrows()):
            html_report += f"""
                <tr>
                    <td>{i+1}</td>
                    <td>{row['Feature']}</td>
                    <td>{row['Aggregated_Score']:.4f}</td>
                    <td>{row['Gene_Cluster']}</td>
                </tr>
            """
        
        html_report += """
            </table>
            
            <h2>Visualizations</h2>
            
            <h3>Gene Interaction Network</h3>
            <p>This network shows interactions between genes that drive pathogenicity. Genes with stronger interactions may work together in functional pathways.</p>
            <img src="gene_interaction_network.png" alt="Gene Interaction Network">
            
            <h3>SHAP Summary</h3>
            <p>The SHAP summary plots show the impact of each gene on pathogenicity predictions. Genes at the top have the strongest effect.</p>
        """
        
        # Add SHAP summary images for each model in the final iteration
        for model_name in models.keys():
            html_report += f"""
            <h4>{model_name.upper()} SHAP Summary</h4>
            <img src="{model_name}_shap_summary_iter{n_iterations}.png" alt="{model_name} SHAP Summary">
            """
            
        html_report += """
            <h2>Recommendations</h2>
            <ul>
                <li>Focus experimental investigation on the top 20-30 genes with highest pathogenicity scores</li>
                <li>Consider gene clusters and interactions when designing validation experiments</li>
                <li>For genes with strong interactions, investigate them in combination rather than in isolation</li>
                <li>Refer to the detailed class contributions file for genes specific to high pathogenicity levels</li>
            </ul>
            
            <h2>Files Generated</h2>
            <ul>
                <li><strong>selected_important_features.csv</strong> - Complete list of selected genes with scores</li>
                <li><strong>detailed_class_contributions.csv</strong> - Per-class contribution of each gene</li>
                <li><strong>feature_selection_log.txt</strong> - Detailed log of the analysis process</li>
                <li>Various visualization PNG files</li>
            </ul>
        </body>
        </html>
        """
        
        # Save the HTML report
        with open(f"{output_dir}/pathogenicity_gene_selection_report.html", "w") as f:
            f.write(html_report)
        
        log_message(f"HTML report saved to {output_dir}/pathogenicity_gene_selection_report.html")
    except Exception as e:
        log_message(f"Error creating HTML report: {str(e)}")
    
    total_elapsed_time = time.time() - total_start_time
    log_message(f"\nFeature selection complete! Total time: {total_elapsed_time:.1f} seconds ({total_elapsed_time/60:.1f} minutes)")
    log_message(f"Selected {len(selected_features)} genes based on pathogenicity driving scores.")
    log_message(f"Results saved to {output_dir}/")
    
    # Final tip
    log_message("\nTIP: To filter the selected genes for a specific pathogenicity level, use the detailed_class_contributions.csv")
    log_message("which shows how strongly each gene pushes towards specific pathogenicity classes.")
    
    return selected_features, results

In [21]:
def importance_driven_feature_selection_cpu(
    df_pa,  # Pandas DataFrame with sparse data
    target_column="Oxygen2",  # Column name containing pathogenicity scores
    n_features_to_select=20,
    scoring_strategy="equal_classes",  # scoring_strategy = "high_class_emphasis" | "equal_classes" | "rare_class_emphasis"
    high_importance_threshold=2,  # Classes >= this value are considered "important" - only works with 'high_class_emphasis'- classes need to be ranked ordinally in order of importance
    class_weights=None,
    random_state=42,
    stratification_method='target',  # 'auto'/'target', 'value', 'class', 'class_and_value', None
    class_column=None,  # Name of metadata class column for stratification
    n_stratification_bins=4,  # Number of bins for value-based stratification
    force_categorical=False,
    initial_features_per_model=2374,
    test_size=0.2,
    n_iterations=3,  # Multiple iterations to improve stability
    interaction_analysis=True,  # Enable gene-gene interaction analysis
    max_samples_for_interactions=50,  # Cap for interaction analysis (memory intensive)
    output_dir='./feature_selection_results',  # Directory to save results
    n_jobs=-1,  # Number of CPU cores to use (-1 for all)
    memory_efficient=True  # Use memory-efficient processing
):
    """
    CPU-only feature selection pipeline for sparse gene presence/absence data,
    prioritizing genes driving pathogenicity while accounting for gene-gene interactions.
    
    Parameters:
    -----------
    df_pa : pandas DataFrame
        Sparse dataframe with samples as rows and genes as columns
        Should include the target column (pathogenicity)
    target_column : str
        Name of the column containing pathogenicity values
    n_features_to_select : int
        Final number of features to select
    scoring_strategy : str
        Determines weighting for different classes; available options are "high_class_emphasis" | "equal_classes" | "rare_class_emphasis"
    high_importance_threshold : int/float
        Threshold at or above which classes are considered pathogenic; only used if scoring_strategy == 'high_class_emphasis'
    class_weights : dict, optional
        Custom weights for each class (higher weights for higher pathogenicity); only used if scoring_strategy == 'high_class_emphasis'
    random_state : int
        Random seed for reproducibility
    initial_features_per_model : int
        Number of features to pre-select in first round using model-based methods
    test_size : float
        Portion of data to use for validation
    n_iterations : int
        Number of times to repeat the analysis with different random seeds
    interaction_analysis : bool
        Whether to analyze gene-gene interactions (more memory intensive)
    max_samples_for_interactions : int
        Maximum number of samples to use for SHAP interaction analysis
    output_dir : str
        Directory to save output files
    n_jobs : int
        Number of CPU cores to use (-1 for all available)
    memory_efficient : bool
        Whether to use memory-efficient processing (recommended for large datasets)
        
    Returns:
    --------
    selected_features : list
        Names of selected features (gene IDs)
    importance_df : DataFrame
        DataFrame with importance scores and interaction information for selected features
    """
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    import shap
    import xgboost as xgb
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.cluster import AgglomerativeClustering
    from sklearn.metrics.pairwise import cosine_similarity
    import time
    import os
    import matplotlib.pyplot as plt
    import seaborn as sns
    import networkx as nx
    import gc
    from tqdm import tqdm
    import psutil

    valid_strategies = ["high_class_emphasis", "equal_classes", "rare_class_emphasis"]
    if scoring_strategy not in valid_strategies:
        raise ValueError(f"scoring_strategy must be one of {valid_strategies}")
    
    # Check available system memory
    available_memory_gb = psutil.virtual_memory().available / 1e9
    print(f"Available system memory: {available_memory_gb:.2f} GB")
    print(f"CPU cores available: {psutil.cpu_count()}")
    print(f"Using {n_jobs if n_jobs != -1 else psutil.cpu_count()} CPU cores for parallel processing")


    # Function to estimate memory usage and adjust batch sizes
    def get_memory_efficient_batch_size(n_features, n_samples, operation_type='general'):
        """
        Calculate memory-efficient batch size based on available memory
        
        Parameters:
        -----------
        n_features : int
            Number of features
        n_samples : int
            Number of samples
        operation_type : str
            Type of operation ('general', 'shap', 'interaction', 'similarity')
            
        Returns:
        --------
        int : Recommended batch size or sample size
        """
        # Conservative memory usage (use 70% of available memory)
        available_gb = available_memory_gb * 0.7
        
        if operation_type == 'shap':
            # SHAP calculation memory estimation
            # Each sample with n_features needs roughly n_features*8 bytes * 3 (for SHAP overhead)
            max_samples = int((available_gb * 1e9) / (n_features * 8 * 3))
            return max(20, min(max_samples, 150))  # Between 20-150 samples
        
        elif operation_type == 'interaction':
            # SHAP interaction calculation (O(n²) memory)
            max_samples = int(np.sqrt((available_gb * 1e9) / (n_features * n_features * 8 * 2)))
            return max(10, min(max_samples, max_samples_for_interactions))
        
        elif operation_type == 'similarity':
            # For similarity matrix calculation
            # Each element is 8 bytes, matrix is n_features x n_features
            matrix_size_gb = (n_features * n_features * 8) / 1e9
            return matrix_size_gb < available_gb
        
        elif operation_type == 'batch_features':
            # For batch processing features
            # Estimate based on dense matrix size
            bytes_per_element = 8  # float64
            max_elements = int((available_gb * 1e9) / bytes_per_element)
            max_features = int(max_elements / n_samples)
            return max(100, min(max_features, 2000))  # Between 100-2000 features per batch
        
        return 100  # Default fallback
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Setup logging file
    log_file = os.path.join(output_dir, "feature_selection_log.txt")
    def log_message(message):
        """Log message to both console and file"""
        print(message)
        with open(log_file, 'a') as f:
            f.write(message + "\n")
    
    log_message(f"Starting CPU-based pathogenicity-driven feature selection on {df_pa.shape[1]-1} genes...")
    log_message(f"Analysis will include gene-gene interactions: {interaction_analysis}")
    log_message(f"Running {n_iterations} iterations for stability...")
    log_message(f"Memory-efficient processing: {memory_efficient}")
    log_message(f"Available system memory: {available_memory_gb:.2f} GB")
    
    total_start_time = time.time()
    
    # Separate features and target
    y_sparse = df_pa[target_column]
    # Convert target to regular numpy array if sparse
    if hasattr(y_sparse, 'sparse'):
        y = y_sparse.sparse.to_dense().values
    else:
        y = y_sparse.values
        
    X_df = df_pa.drop(columns=[target_column])
    feature_names = X_df.columns.tolist()
    
    # Encode target if it's not already numeric
    if not np.issubdtype(y.dtype, np.number):
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(y)
        class_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
        log_message(f"Encoded {len(label_encoder.classes_)} classes: {class_mapping}")
    else:
        # Create class mapping for numeric values
        unique_classes = sorted(np.unique(y))
        class_mapping = {cls: idx for idx, cls in enumerate(unique_classes)}
        log_message(f"Using {len(unique_classes)} numeric classes: {class_mapping}")
        
        # Create a label encoder for consistency
        label_encoder = LabelEncoder()
        label_encoder.classes_ = np.array(unique_classes)

    is_binary = len(np.unique(y)) == 2
    if is_binary:
        log_message("Binary classification detected. Adjusting analysis approach...")
        
        if scoring_strategy == "high_class_emphasis":
            # Assume class 1 is "high importance"
            if high_importance_threshold is None:
                high_importance_threshold = 1
            log_message("Binary + high_class_emphasis: Focusing on positive class (class 1)")
        
        elif scoring_strategy == "equal_classes":
            log_message("Binary + equal_classes: Treating both classes equally")
        
        elif scoring_strategy == "rare_class_emphasis":
            log_message("Binary + rare_class_emphasis: Emphasizing minority class")
    
    if scoring_strategy == "high_class_emphasis":
        high_importance_classes = [i for i, c in enumerate(label_encoder.classes_) 
                                 if c >= high_importance_threshold]
    else:
        # For other strategies, consider all classes as "important"
        high_importance_classes = list(range(len(label_encoder.classes_)))
    
    # If no custom weights, create default weights that emphasize pathogenic classes
    if class_weights is None:
        if scoring_strategy == "high_class_emphasis":
            if is_binary:
                class_weights = {0: 1.0, 1: 3.0}
            else:
                max_class = len(np.unique(y)) - 1
                class_weights = {i: np.exp(i/max_class * 1.5) for i in range(len(np.unique(y)))}
        
        elif scoring_strategy == "equal_classes":
            class_weights = {i: 1.0 for i in range(len(np.unique(y)))}
        
        elif scoring_strategy == "rare_class_emphasis":
            # Inverse frequency weighting
            class_counts = np.bincount(y)
            total_samples = len(y)
            class_weights = {i: total_samples / (len(class_counts) * count) 
                            for i, count in enumerate(class_counts)}
    
    # Function for stratified sampling that ensures representation of rare classes
    def stratified_sample(X, y, n_samples, random_state=42):
        """Sample data while preserving class distribution, with minimum samples per class"""
        unique_classes = np.unique(y)
        min_samples_per_class = max(3, int(n_samples / (len(unique_classes) * 2)))
        
        sampled_indices = []
        for cls in unique_classes:
            cls_indices = np.where(y == cls)[0]
            
            if len(cls_indices) <= min_samples_per_class:
                sampled_indices.extend(cls_indices)
            else:
                n_to_sample = max(min_samples_per_class, 
                                int(len(cls_indices) / len(y) * n_samples))
                n_to_sample = min(n_to_sample, len(cls_indices))
                
                np.random.seed(random_state)
                sampled_from_cls = np.random.choice(cls_indices, size=n_to_sample, replace=False)
                sampled_indices.extend(sampled_from_cls)
        
        return np.array(sampled_indices)
    
    # Initialize storage for multi-iteration results
    all_iterations_scores = {}
    all_iterations_clusters = {}
    all_iterations_interactions = {}
    
    # Run multiple iterations with different random seeds for stability
    for iteration in range(n_iterations):
        current_seed = random_state + iteration
        log_message(f"\n{'='*80}\nIteration {iteration+1}/{n_iterations} (seed: {current_seed})\n{'='*80}")
        start_time = time.time()
        
        # Create stratification for this iteration
        stratify = create_stratification(
            y, df_pa, stratification_method, class_column, n_stratification_bins, force_categorical
        )
        
        # Split data with current seed
        X_train_df, X_test_df, y_train, y_test = train_test_split(
            X_df, y, test_size=test_size, random_state=current_seed, stratify=stratify
        )
        
        log_message(f"Training set: {X_train_df.shape}, Test set: {X_test_df.shape}")
        log_message(f"Class distribution in training: {np.bincount(y_train)}")
        
        # STAGE 1: Initial feature pre-selection using model-based importance
        log_message(f"STAGE 1: Pre-selecting features...")
        
        # Define CPU-only models for initial screening
        objective = 'binary:logistic' if is_binary else 'multi:softmax'
        models = {
            'xgb': xgb.XGBClassifier(
                n_estimators=100, 
                learning_rate=0.1,
                max_depth=6,
                objective=objective,
                num_class=None if is_binary else len(np.unique(y)),
                tree_method='hist',      # CPU version
                n_jobs=n_jobs,           # Use multiple cores
                random_state=current_seed
            ),
            'rf': RandomForestClassifier(
                n_estimators=100,
                max_depth=6,
                n_jobs=n_jobs,
                random_state=current_seed
            )
        }
        
        # Try to add CatBoost
        try:
            from catboost import CatBoostClassifier
            models['catboost'] = CatBoostClassifier(
                iterations=100,
                depth=6,
                learning_rate=0.1,
                loss_function='MultiClass',
                task_type='CPU',           # Force CPU usage
                thread_count=n_jobs if n_jobs > 0 else None,
                random_seed=current_seed,
                verbose=False,
                allow_writing_files=False
            )
            log_message("CatBoost added to analysis.")
        except ImportError:
            log_message("CatBoost not available. Install with: pip install catboost")
        
        # Initialize to store feature importance across models
        feature_importances = {}
        
        # Train each model and calculate feature importance
        for name, model in models.items():
            log_message(f"Training {name} model for initial feature screening...")
            
            # Memory-efficient conversion and training
            if memory_efficient and X_train_df.shape[1] > 5000:
                log_message(f"Using memory-efficient batch processing for {name}...")
                
                # Calculate optimal batch size
                batch_size = get_memory_efficient_batch_size(
                    X_train_df.shape[1], X_train_df.shape[0], 'batch_features')
                
                log_message(f"Processing in batches of {batch_size} features...")
                
                # Train on initial subset to get baseline
                initial_features = min(batch_size, X_train_df.shape[1])
                X_subset = X_train_df.iloc[:, :initial_features]
                
                if hasattr(X_subset, 'sparse'):
                    X_subset_array = X_subset.sparse.to_dense().values
                else:
                    X_subset_array = X_subset.values
                
                # Train initial model
                model.fit(X_subset_array, y_train)
                
                # Get initial importances
                if hasattr(model, 'feature_importances_'):
                    importances = np.zeros(X_train_df.shape[1])
                    importances[:initial_features] = model.feature_importances_
                else:
                    importances = np.zeros(X_train_df.shape[1])
                
                # Process remaining features in batches if needed
                if X_train_df.shape[1] > batch_size:
                    for start_idx in range(batch_size, X_train_df.shape[1], batch_size):
                        end_idx = min(start_idx + batch_size, X_train_df.shape[1])
                        
                        # Create combined feature set
                        feature_indices = list(range(initial_features)) + list(range(start_idx, end_idx))
                        X_batch = X_train_df.iloc[:, feature_indices]
                        
                        if hasattr(X_batch, 'sparse'):
                            X_batch_array = X_batch.sparse.to_dense().values
                        else:
                            X_batch_array = X_batch.values
                        
                        # Train model on combined features
                        batch_model = type(model)(**model.get_params())
                        batch_model.fit(X_batch_array, y_train)
                        
                        # Update importances for new features
                        if hasattr(batch_model, 'feature_importances_'):
                            batch_importances = batch_model.feature_importances_
                            # Update importances for the new features (skip initial features)
                            importances[start_idx:end_idx] = batch_importances[initial_features:]
                        
                        # Clear memory
                        del X_batch, X_batch_array, batch_model
                        gc.collect()
                
            else:
                # Standard processing for smaller datasets
                try:
                    if hasattr(X_train_df, 'sparse'):
                        log_message("Converting sparse matrix to dense for model training...")
                        X_train_array = X_train_df.sparse.to_dense().values
                    else:
                        X_train_array = X_train_df.values
                    
                    # Train model
                    model.fit(X_train_array, y_train)
                    
                    # Get feature importances
                    if hasattr(model, 'feature_importances_'):
                        importances = model.feature_importances_
                    else:
                        importances = np.zeros(X_train_df.shape[1])
                    
                    # Clear memory after training
                    del X_train_array
                    gc.collect()
                    
                except MemoryError:
                    log_message(f"Memory error with {name}. Skipping this model.")
                    importances = np.zeros(X_train_df.shape[1])
            
            feature_importances[name] = importances
        
        # Average the feature importances across models
        avg_importance = np.zeros(X_df.shape[1])
        for imp in feature_importances.values():
            avg_importance += imp
        avg_importance /= len(feature_importances)
        
        # Select top features from initial screening
        initial_selected_indices = np.argsort(avg_importance)[-initial_features_per_model:]
        initial_selected_features = [feature_names[i] for i in initial_selected_indices]
        
        log_message(f"Selected {len(initial_selected_features)} features in initial screening")
        
        # Create reduced dataframes with only pre-selected features
        X_train_reduced_df = X_train_df[initial_selected_features]
        X_test_reduced_df = X_test_df[initial_selected_features]
        
        # Convert to dense for further analysis
        if hasattr(X_train_reduced_df, 'sparse'):
            X_train_reduced = X_train_reduced_df.sparse.to_dense().values
        else:
            X_train_reduced = X_train_reduced_df.values
            
        if hasattr(X_test_reduced_df, 'sparse'):
            X_test_reduced = X_test_reduced_df.sparse.to_dense().values
        else:
            X_test_reduced = X_test_reduced_df.values
        
        # Clear memory
        gc.collect()
        
        # STAGE 2: Cluster genes to identify potential interaction groups
        log_message("STAGE 2: Clustering genes based on co-occurrence patterns...")
        
        # Calculate gene similarity matrix
        log_message("Calculating gene similarity matrix on CPU...")
        
        # Check if we should use memory-efficient similarity calculation
        use_efficient_similarity = not get_memory_efficient_batch_size(
            len(initial_selected_features), X_train_reduced.shape[0], 'similarity')
        
        if use_efficient_similarity and len(initial_selected_features) > 1000:
            log_message("Using memory-efficient similarity calculation...")
            # Calculate similarity in chunks
            chunk_size = 500
            gene_similarity = np.zeros((len(initial_selected_features), len(initial_selected_features)))
            
            for i in range(0, len(initial_selected_features), chunk_size):
                end_i = min(i + chunk_size, len(initial_selected_features))
                for j in range(0, len(initial_selected_features), chunk_size):
                    end_j = min(j + chunk_size, len(initial_selected_features))
                    
                    # Calculate similarity for this chunk
                    chunk_sim = cosine_similarity(
                        X_train_reduced[:, i:end_i].T, 
                        X_train_reduced[:, j:end_j].T
                    )
                    gene_similarity[i:end_i, j:end_j] = chunk_sim
                    
                # Clear memory periodically
                if i % (chunk_size * 4) == 0:
                    gc.collect()
        else:
            gene_similarity = cosine_similarity(X_train_reduced.T)
        
        # Save similarity matrix visualization
        plt.figure(figsize=(10, 8))
        viz_size = min(100, len(initial_selected_features))
        sns.heatmap(gene_similarity[:viz_size, :viz_size], cmap='viridis')
        plt.title(f"Gene Similarity Matrix (First {viz_size} Genes) - Iteration {iteration+1}")
        plt.tight_layout()
        plt.savefig(f"{output_dir}/gene_similarity_iter{iteration+1}.png", dpi=150)
        plt.close()
        
        # Cluster genes
        n_clusters = min(50, len(initial_selected_features) // 10)  # Reasonable number of clusters
        log_message(f"Clustering genes into {n_clusters} groups...")
        clustering = AgglomerativeClustering(n_clusters=n_clusters)
        gene_clusters = clustering.fit_predict(gene_similarity)
        
        # Map genes to clusters
        gene_cluster_map = {gene: cluster for gene, cluster in zip(initial_selected_features, gene_clusters)}
        
        # STAGE 3: Directional SHAP analysis on pre-selected features
        log_message("STAGE 3: Performing directional SHAP analysis...")
        
        # Get memory-efficient sample size
        max_shap_samples = get_memory_efficient_batch_size(
            len(initial_selected_features), X_test_reduced.shape[0], 'shap')
        log_message(f"Using sample size of {max_shap_samples} for SHAP analysis...")
        
        # Sample test data with stratification
        # For SHAP sampling
        sample_indices = stratified_sample_flexible(
            X_test_reduced, y_test, df_pa.loc[X_test_df.index], max_shap_samples,
            stratification_method, class_column, n_stratification_bins, force_categorical, current_seed
        )
        X_shap_samples = X_test_reduced[sample_indices]
        y_shap_samples = y_test[sample_indices]
        
        # Verify class distribution
        class_counts = np.bincount(y_shap_samples, minlength=len(np.unique(y)))
        log_message(f"Class distribution in SHAP samples: {class_counts}")
        
        # Store interaction information
        interaction_scores = {}
        
        # Store pathogenicity scores
        importance_scores = {}
        
        # Analyze with each model (CPU versions)
        for name, model_class in models.items():
            log_message(f"Training {name} model for directional SHAP analysis...")
            
            # Create a new model instance with more trees for stability
            if name == 'catboost':
                try:
                    model = CatBoostClassifier(
                        iterations=200,
                        depth=6,
                        learning_rate=0.05,
                        loss_function='MultiClass',
                        task_type='CPU',
                        thread_count=n_jobs if n_jobs > 0 else None,
                        random_seed=current_seed,
                        verbose=False,
                        allow_writing_files=False
                    )
                except NameError:
                    log_message("CatBoost not available for SHAP analysis")
                    continue
            elif name == 'xgb':
                objective = 'binary:logistic' if is_binary else 'multi:softmax'
                model = xgb.XGBClassifier(
                    n_estimators=200,  # Reduced from 300 for memory efficiency
                    max_depth=6,
                    learning_rate=0.05,
                    objective=objective,
                    num_class=None if is_binary else len(np.unique(y)),
                    tree_method='hist',
                    n_jobs=n_jobs,
                    random_state=current_seed
                )
            else:  # Random Forest
                model = RandomForestClassifier(
                    n_estimators=200,  # Reduced from 300 for memory efficiency
                    max_depth=8,
                    n_jobs=n_jobs,
                    random_state=current_seed
                )
            
            # Train the model
            model.fit(X_train_reduced, y_train)
            
            # Calculate SHAP values
            log_message(f"Calculating SHAP values for {name}...")
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_shap_samples)
            # DEBUG: Print SHAP values information
            log_message(f"DEBUG: Raw SHAP values type: {type(shap_values)}")
            if isinstance(shap_values, list):
                log_message(f"DEBUG: SHAP values list length: {len(shap_values)}")
                for i, arr in enumerate(shap_values):
                    log_message(f"DEBUG: SHAP array {i} shape: {arr.shape}")
            else:
                log_message(f"DEBUG: Single SHAP array shape: {shap_values.shape}")
            log_message(f"DEBUG: Expected shape should be: {X_shap_samples.shape}")
            log_message(f"DEBUG: Number of unique classes: {len(np.unique(y))}")
            log_message(f"DEBUG: Is binary: {len(np.unique(y)) == 2}")
            # ROBUST SHAP NORMALIZATION - HANDLE ALL MODEL TYPES
            expected_features = X_shap_samples.shape[1]
            expected_samples = X_shap_samples.shape[0]
            
            if isinstance(shap_values, list):
                # Already a list, validate each array
                valid_shap_arrays = []
                for i, shap_array in enumerate(shap_values):
                    if len(shap_array.shape) == 2 and shap_array.shape == (expected_samples, expected_features):
                        valid_shap_arrays.append(shap_array)
                        log_message(f"DEBUG: Class {i} SHAP array is valid")
                    else:
                        log_message(f"DEBUG: Class {i} SHAP array invalid shape {shap_array.shape}, skipping")
                
                if len(valid_shap_arrays) == 0:
                    log_message("ERROR: No valid SHAP arrays found, skipping this model")
                    continue
                
                shap_values = valid_shap_arrays
            else:
                # Single array case - handle different formats
                if len(shap_values.shape) == 2 and shap_values.shape == (expected_samples, expected_features):
                    # XGBoost binary format: (samples, features)
                    if len(np.unique(y)) == 2:
                        shap_values = [np.zeros_like(shap_values), shap_values]
                        log_message("DEBUG: Converted XGBoost binary SHAP to list format")
                    else:
                        shap_values = [shap_values]
                        log_message("DEBUG: Converted single SHAP to list format")
                
                elif len(shap_values.shape) == 3 and shap_values.shape == (expected_samples, expected_features, len(np.unique(y))):
                    # Random Forest/CatBoost binary format: (samples, features, classes)
                    log_message("DEBUG: Detected 3D SHAP array (RF/CatBoost format)")
                    shap_list = []
                    for class_idx in range(shap_values.shape[2]):
                        shap_list.append(shap_values[:, :, class_idx])
                        log_message(f"DEBUG: Extracted class {class_idx} SHAP array shape: {shap_values[:, :, class_idx].shape}")
                    shap_values = shap_list
                    log_message("DEBUG: Converted 3D SHAP to list format")
                
                else:
                    log_message(f"ERROR: Invalid SHAP array shape {shap_values.shape}, skipping this model")
                    continue
            
            # Final validation
            log_message(f"DEBUG: Final SHAP list length: {len(shap_values)}")
            for i, arr in enumerate(shap_values):
                log_message(f"DEBUG: Final SHAP array {i} shape: {arr.shape}")
            
            # Clear some memory
            gc.collect()
            
            # Process SHAP values for pathogenicity
            if isinstance(shap_values, list):
                if scoring_strategy == "high_class_emphasis":
                    # Current logic - emphasize high classes
                    high_importance_contribution = np.zeros(X_shap_samples.shape[1])
                    non_high_importance_contribution = np.zeros(X_shap_samples.shape[1])
                    class_contribution = np.zeros(X_shap_samples.shape[1])  # Initialize here
                    
                    for class_idx in range(len(shap_values)):
                        weight = class_weights.get(class_idx, 1.0)
                        positive_contribution = np.where(shap_values[class_idx] > 0, 
                                                      shap_values[class_idx], 0).mean(axis=0)
                        
                        if class_idx in high_importance_classes:
                            high_importance_contribution += positive_contribution
                        else:
                            non_high_importance_contribution += positive_contribution
                        
                        # Add weighted contribution
                        class_contribution += weight * positive_contribution
                    
                    importance_score = high_importance_contribution - non_high_importance_contribution
                
                elif scoring_strategy == "equal_classes":
                    # Sum all positive contributions equally
                    importance_score = np.zeros(X_shap_samples.shape[1])
                    class_contribution = np.zeros(X_shap_samples.shape[1])  # Initialize here
                    
                    for class_idx in range(len(shap_values)):
                        positive_contribution = np.where(shap_values[class_idx] > 0, 
                                                      shap_values[class_idx], 0).mean(axis=0)
                        importance_score += positive_contribution
                        class_contribution += positive_contribution  # Same as importance_score for equal weighting
                
                elif scoring_strategy == "rare_class_emphasis":
                    # Weight by inverse class frequency
                    importance_score = np.zeros(X_shap_samples.shape[1])
                    class_contribution = np.zeros(X_shap_samples.shape[1])  # Initialize here
                    class_counts = np.bincount(y)
                    
                    for class_idx in range(len(shap_values)):
                        weight = len(y) / class_counts[class_idx]  # Inverse frequency
                        positive_contribution = np.where(shap_values[class_idx] > 0, 
                                                      shap_values[class_idx], 0).mean(axis=0)
                        weighted_contribution = weight * positive_contribution
                        importance_score += weighted_contribution
                        class_contribution += weighted_contribution  # Same as importance_score for this strategy
                
                # Store scores
                importance_scores[f"{name}_importance_score"] = importance_score
                importance_scores[f"{name}_weighted_score"] = class_contribution
                
                # Save SHAP summary plot for each model
                plt.figure(figsize=(10, 12))
                shap.summary_plot(shap_values, X_shap_samples, plot_type="bar", 
                                  feature_names=initial_selected_features, show=False)
                plt.title(f"{name.upper()} SHAP Summary - Iteration {iteration+1}")
                plt.tight_layout()
                plt.savefig(f"{output_dir}/{name}_shap_summary_iter{iteration+1}.png", dpi=200)
                plt.close()
                
                # Calculate gene-gene interactions if enabled and memory available
                if interaction_analysis and len(initial_selected_features) <= 2000:  # Limit for memory
                    try:
                        log_message(f"Calculating interaction values for {name}...")
                        
                        # Get memory-efficient sample size for interactions
                        interaction_sample_size = get_memory_efficient_batch_size(
                            len(initial_selected_features), X_shap_samples.shape[0], 'interaction')
                        interaction_sample_size = min(interaction_sample_size, X_shap_samples.shape[0])
                        
                        log_message(f"Using {interaction_sample_size} samples for interaction analysis...")
                        
                        # Sample for interaction analysis
                        if interaction_sample_size < X_shap_samples.shape[0]:
                            interaction_indices = np.random.choice(X_shap_samples.shape[0], 
                                                                 size=interaction_sample_size, 
                                                                 replace=False)
                            interaction_samples = X_shap_samples[interaction_indices]
                        else:
                            interaction_samples = X_shap_samples
                        
                        # Calculate SHAP interaction values
                        interaction_values = explainer.shap_interaction_values(interaction_samples)
                        
                        # Process interaction values for pathogenic classes
                        if isinstance(interaction_values, list):
                            # Initialize interaction matrix
                            gene_interactions = np.zeros((X_shap_samples.shape[1], X_shap_samples.shape[1]))
                            
                            # Sum positive interactions toward pathogenic classes
                            if scoring_strategy == "high_class_emphasis":
                                target_classes = high_importance_classes
                            else:
                                target_classes = list(range(len(interaction_values)))
                            
                            for class_idx in target_classes:
                                if class_idx < len(interaction_values):
                                    pos_interactions = np.where(interaction_values[class_idx] > 0, 
                                                              interaction_values[class_idx], 0)
                                    gene_interactions += pos_interactions.mean(axis=0)
                            
                            # Store interaction scores
                            interaction_scores[f"{name}_interactions"] = gene_interactions
                            
                            # Find top interactions
                            np.fill_diagonal(gene_interactions, 0)  # Remove self-interactions
                            interaction_sums = gene_interactions.sum(axis=1)
                            
                            # Top interacting genes
                            top_interacting_idx = np.argsort(interaction_sums)[-min(50, len(interaction_sums)):]
                            top_interacting_genes = [initial_selected_features[i] for i in top_interacting_idx]
                            
                            log_message(f"Top interacting genes for {name}: {top_interacting_genes[:10]}")
                            
                            # Save interaction matrix visualization (subset for visibility)
                            if len(top_interacting_idx) > 5:
                                plt.figure(figsize=(12, 10))
                                
                                # Use top interacting genes for visualization
                                viz_size = min(30, len(top_interacting_idx))
                                interaction_subset = gene_interactions[top_interacting_idx[-viz_size:]][:, top_interacting_idx[-viz_size:]]
                                
                                sns.heatmap(interaction_subset, cmap='inferno', 
                                            xticklabels=[initial_selected_features[i] for i in top_interacting_idx[-viz_size:]],
                                            yticklabels=[initial_selected_features[i] for i in top_interacting_idx[-viz_size:]])
                                plt.title(f"{name.upper()} Gene Interaction Matrix - Top {viz_size} Genes")
                                plt.xticks(rotation=90)
                                plt.yticks(rotation=0)
                                plt.tight_layout()
                                plt.savefig(f"{output_dir}/{name}_interactions_iter{iteration+1}.png", dpi=200)
                                plt.close()
                            
                            # Clear memory
                            del interaction_values, interaction_samples
                            gc.collect()
                            
                    except Exception as e:
                        log_message(f"Error calculating interaction values for {name}: {str(e)}")
                        log_message("Will continue without interaction values for this model.")
                elif interaction_analysis:
                    log_message(f"Skipping interaction analysis for {name} - too many features for available memory")
            else:
                log_message(f"Warning: SHAP values for {name} not in expected format. Skipping.")
        
        # Calculate average scores across models
        path_score_cols = [k for k in importance_scores.keys() if k.endswith('_importance_score')]
        weighted_score_cols = [k for k in importance_scores.keys() if k.endswith('_weighted_score')]
        
        # Initialize arrays for averaging
        avg_path_score = np.zeros(len(initial_selected_features))
        avg_weighted_score = np.zeros(len(initial_selected_features))
        
        # Average the scores from different models
        for col in path_score_cols:
            avg_path_score += importance_scores[col]
        avg_path_score /= len(path_score_cols) if path_score_cols else 1
        
        for col in weighted_score_cols:
            avg_weighted_score += importance_scores[col]
        avg_weighted_score /= len(weighted_score_cols) if weighted_score_cols else 1
        
        # Create combined score (both directional and weighted)
        combined_score = (avg_path_score + avg_weighted_score) / 2
        
        # Store scores for this iteration
        all_iterations_scores[iteration] = {
            'importance_score': avg_path_score,
            'weighted_score': avg_weighted_score,
            'combined_score': combined_score
        }
        
        # Store clusters for this iteration
        all_iterations_clusters[iteration] = gene_clusters
        
        # Store interactions if calculated
        if interaction_scores:
            # Average interaction matrix across models
            avg_interaction = np.zeros((X_shap_samples.shape[1], X_shap_samples.shape[1]))
            for matrix in interaction_scores.values():
                avg_interaction += matrix
            avg_interaction /= len(interaction_scores)
            
            all_iterations_interactions[iteration] = avg_interaction
        
        elapsed_time = time.time() - start_time
        log_message(f"Iteration {iteration+1} complete! Time taken: {elapsed_time:.1f} seconds")
        
        # Clear memory between iterations
        gc.collect()
    
    # STAGE 4: Aggregate results across iterations
    log_message("\nSTAGE 4: Aggregating results across all iterations...")
    
    # Initialize array to store aggregated importance scores
    aggregated_scores = np.zeros(len(initial_selected_features))
    
    # Average scores across iterations
    for iteration_scores in all_iterations_scores.values():
        aggregated_scores += iteration_scores['combined_score']
    aggregated_scores /= n_iterations
    
    # Select final features using aggregated scores
    selected_indices = np.argsort(aggregated_scores)[-n_features_to_select:]
    
    # Get the selected feature names
    selected_features = [initial_selected_features[i] for i in selected_indices]
    
    # Create results DataFrame with detailed scores
    results = pd.DataFrame({
        'Feature': selected_features,
        'Aggregated_Score': aggregated_scores[selected_indices],
    })
    
    # Add individual iteration scores
    for iteration in range(n_iterations):
        results[f'Iteration_{iteration+1}_Score'] = all_iterations_scores[iteration]['combined_score'][selected_indices]
    
    # Add cluster information (from the first iteration for simplicity)
    results['Gene_Cluster'] = [gene_cluster_map.get(gene, -1) for gene in selected_features]
    
    # Add interaction information if available
    if all_iterations_interactions:
        # Calculate average interaction strength for each gene
        avg_interaction_matrix = np.zeros((len(initial_selected_features), len(initial_selected_features)))
        for matrix in all_iterations_interactions.values():
            avg_interaction_matrix += matrix
        avg_interaction_matrix /= len(all_iterations_interactions)
        
        # Calculate total interaction strength for each gene (sum of row)
        interaction_strength = avg_interaction_matrix.sum(axis=1)
        
        # Add to results
        results['Interaction_Strength'] = interaction_strength[selected_indices]
        
        # Find top interaction partners for each selected gene
        top_partners = []
        for idx in selected_indices:
            gene_interactions = avg_interaction_matrix[idx, :]
            partner_indices = np.argsort(gene_interactions)[-5:]  # Top 5 partners
            partners = [initial_selected_features[i] for i in partner_indices if gene_interactions[i] > 0]
            top_partners.append(', '.join(partners) if partners else 'None')
        
        results['Top_Interaction_Partners'] = top_partners
        
        # Create gene interaction network visualization
        log_message("Creating gene interaction network visualization...")
        G = nx.Graph()
        
        # Add nodes for top genes
        for gene in selected_features:
            idx = initial_selected_features.index(gene)
            G.add_node(gene, score=aggregated_scores[idx])
        
        # Add edges based on interaction strengths
        if np.sum(avg_interaction_matrix > 0) > 0:
            # There are positive values, proceed normally
            edge_threshold = np.percentile(avg_interaction_matrix[avg_interaction_matrix > 0], 90)
            log_message(f"Using edge threshold of {edge_threshold:.4f} (90th percentile of non-zero interactions)")
        else:
            # No positive interactions found, use a small positive value
            edge_threshold = 0.001
            log_message("No positive interactions found. Using default threshold of 0.001")
        
        for i, gene1 in enumerate(selected_features):
            idx1 = initial_selected_features.index(gene1)
            for j, gene2 in enumerate(selected_features):
                if i < j:  # Avoid duplicate edges
                    idx2 = initial_selected_features.index(gene2)
                    interaction = avg_interaction_matrix[idx1, idx2]
                    if interaction > edge_threshold:
                        G.add_edge(gene1, gene2, weight=interaction)
        
        # Draw the network if it has edges
        if nx.number_of_edges(G) > 0:
            # Save static version for the report
            plt.figure(figsize=(15, 15))

            # Position nodes using force-directed layout
            pos = nx.spring_layout(G, seed=42)

            # Get node colors based on scores
            node_colors = [G.nodes[n]['score'] for n in G.nodes()]

            # Get edge weights
            edge_weights = [G.edges[e]['weight'] * 5 for e in G.edges()]

            # Draw nodes
            nodes = nx.draw_networkx_nodes(G, pos, node_size=500, node_color=node_colors, 
                                         alpha=0.8, cmap=plt.cm.viridis)

            # Draw edges
            edges = nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.5, 
                                         edge_color='gray')

            # Draw without labels for the static version
            plt.title("Gene Interaction Network - Pathogenicity Driving Genes")
            plt.axis('off')
            if nodes is not None:
                plt.colorbar(nodes, label="Pathogenicity Score")
            plt.tight_layout()
            plt.savefig(f"{output_dir}/gene_interaction_network.png", dpi=300)
            plt.close()

            # Create interactive version with hoverable labels
            log_message("Creating interactive gene network visualization...")
            try:
                create_interactive_gene_network(
                    selected_features=selected_features,
                    interaction_matrix=avg_interaction_matrix,
                    scores=aggregated_scores[selected_indices],
                    output_file=f"{output_dir}/interactive_gene_network.html"
                )
            except NameError:
                log_message("Interactive network function not available. Creating simple network file...")
                # Create a simple HTML file with network information
                with open(f"{output_dir}/gene_network_info.html", "w") as f:
                    f.write(f"<html><body><h1>Gene Network Information</h1>")
                    f.write(f"<p>Network has {nx.number_of_nodes(G)} nodes and {nx.number_of_edges(G)} edges</p>")
                    f.write(f"<p>Edge threshold: {edge_threshold:.4f}</p>")
                    f.write("</body></html>")
        else:
            log_message("No edges above threshold for interaction network visualization")
        
    
    # Sort by aggregated score
    results = results.sort_values('Aggregated_Score', ascending=False)
    
    # Save final results
    results.to_csv(f"{output_dir}/selected_important_features.csv", index=False)
    
    # Create a separate detailed table with per-class information for top genes
    if n_iterations > 0 and 'shap_values' in locals() and isinstance(shap_values, list):
        log_message("Creating detailed per-class analysis for top genes...")
        
        # Use the last iteration's SHAP values for simplicity
        detailed_results = []
        
        for gene_idx, gene_name in enumerate(initial_selected_features):
            if gene_name in selected_features:
                gene_data = {'Gene': gene_name}
                
                # Add per-class contributions
                for class_idx in range(len(shap_values)):
                    class_name = label_encoder.classes_[class_idx]
                    # Average positive SHAP values for this class
                    positive_contrib = np.where(shap_values[class_idx][:, gene_idx] > 0, 
                                             shap_values[class_idx][:, gene_idx], 0).mean()
                    gene_data[f'Push_to_Class_{class_name}'] = positive_contrib
                
                detailed_results.append(gene_data)
        
        # Convert to DataFrame and save
        if detailed_results:
            detailed_df = pd.DataFrame(detailed_results)
            detailed_df.to_csv(f"{output_dir}/detailed_class_contributions.csv", index=False)
    
    # Create an HTML report with key findings
    try:
        log_message("Creating HTML report with key findings...")
        
        html_report = f"""
        <html>
        <head>
            <title>NTM Pathogenicity Gene Selection Report (CPU Version)</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; }}
                h1, h2, h3 {{ color: #2c3e50; }}
                table {{ border-collapse: collapse; width: 100%; margin-bottom: 20px; }}
                th, td {{ text-align: left; padding: 8px; border-bottom: 1px solid #ddd; }}
                th {{ background-color: #f2f2f2; }}
                tr:hover {{ background-color: #f5f5f5; }}
                .summary {{ background-color: #eef7fa; padding: 15px; border-radius: 5px; }}
                img {{ max-width: 100%; border: 1px solid #ddd; margin: 10px 0; }}
                .cpu-note {{ background-color: #fff3cd; padding: 10px; border-radius: 5px; margin: 10px 0; }}
            </style>
        </head>
        <body>
            <h1>NTM Pathogenicity Gene Selection Report (CPU Version)</h1>
            
            <div class="cpu-note">
                <h3>Analysis Method</h3>
                <p>This analysis was performed using CPU-only processing for maximum compatibility.</p>
                <p>Available system memory: {available_memory_gb:.2f} GB</p>
                <p>CPU cores used: {n_jobs if n_jobs != -1 else psutil.cpu_count()}</p>
                <p>Memory-efficient processing: {memory_efficient}</p>
            </div>
            
            <div class="summary">
                <h2>Summary</h2>
                <p>Analysis completed on {time.strftime('%Y-%m-%d %H:%M:%S')}</p>
                <p>Selected {len(selected_features)} genes from {len(feature_names)} total genes that drive pathogenicity in NTMs.</p>
                <p>Analysis used {n_iterations} iterations with different random seeds to ensure stability.</p>
                <p>Gene selection focused on genes that push specifically toward pathogenic classes (>= {high_importance_threshold}).</p>
            </div>
            
            <h2>Top 20 Pathogenicity-Driving Genes</h2>
            <table>
                <tr>
                    <th>Rank</th>
                    <th>Gene</th>
                    <th>Pathogenicity Score</th>
                    <th>Cluster</th>
        """
        
        # Add top 20 genes to table
        for i, (_, row) in enumerate(results.head(20).iterrows()):
            html_report += f"""
                <tr>
                    <td>{i+1}</td>
                    <td>{row['Feature']}</td>
                    <td>{row['Aggregated_Score']:.4f}</td>
                    <td>{row['Gene_Cluster']}</td>
                </tr>
            """
        
        html_report += """
            </table>
            
            <h2>Visualizations</h2>
        """
        
        # Check if interaction network was created
        if all_iterations_interactions and os.path.exists(f"{output_dir}/gene_interaction_network.png"):
            html_report += """
            <h3>Gene Interaction Network</h3>
            <p>This network shows interactions between genes that drive pathogenicity. Genes with stronger interactions may work together in functional pathways.</p>
            <img src="gene_interaction_network.png" alt="Gene Interaction Network">
            """
        
        html_report += """
            <h3>SHAP Summary</h3>
            <p>The SHAP summary plots show the impact of each gene on pathogenicity predictions. Genes at the top have the strongest effect.</p>
        """
        
        # Add SHAP summary images for each model in the final iteration
        for model_name in ['xgb', 'rf']:
            shap_file = f"{model_name}_shap_summary_iter{n_iterations}.png"
            if os.path.exists(f"{output_dir}/{shap_file}"):
                html_report += f"""
                <h4>{model_name.upper()} SHAP Summary</h4>
                <img src="{shap_file}" alt="{model_name} SHAP Summary">
                """
            
        html_report += """
            <h2>Recommendations</h2>
            <ul>
                <li>Focus experimental investigation on the top 20-30 genes with highest pathogenicity scores</li>
                <li>Consider gene clusters and interactions when designing validation experiments</li>
                <li>For genes with strong interactions, investigate them in combination rather than in isolation</li>
                <li>Refer to the detailed class contributions file for genes specific to high pathogenicity levels</li>
            </ul>
            
            <h2>Performance Notes</h2>
            <ul>
                <li>This CPU-only version provides the same analysis quality as the GPU version</li>
                <li>Processing time may be longer for very large datasets</li>
                <li>Memory-efficient processing was used to handle large gene sets</li>
                <li>Interaction analysis was limited based on available system memory</li>
            </ul>
            
            <h2>Files Generated</h2>
            <ul>
                <li><strong>selected_important_features.csv</strong> - Complete list of selected genes with scores</li>
                <li><strong>detailed_class_contributions.csv</strong> - Per-class contribution of each gene (if available)</li>
                <li><strong>feature_selection_log.txt</strong> - Detailed log of the analysis process</li>
                <li>Various visualization PNG files</li>
            </ul>
        </body>
        </html>
        """
        
        # Save the HTML report
        with open(f"{output_dir}/pathogenicity_gene_selection_report.html", "w") as f:
            f.write(html_report)
        
        log_message(f"HTML report saved to {output_dir}/pathogenicity_gene_selection_report.html")
    except Exception as e:
        log_message(f"Error creating HTML report: {str(e)}")
    
    total_elapsed_time = time.time() - total_start_time
    log_message(f"\nFeature selection complete! Total time: {total_elapsed_time:.1f} seconds ({total_elapsed_time/60:.1f} minutes)")
    log_message(f"Selected {len(selected_features)} genes based on pathogenicity driving scores.")
    log_message(f"Results saved to {output_dir}/")
    
    # Final tip
    log_message("\nTIP: To filter the selected genes for a specific pathogenicity level, use the detailed_class_contributions.csv")
    log_message("which shows how strongly each gene pushes towards specific pathogenicity classes.")
    
    return selected_features, results


def create_interactive_gene_network_cpu(selected_features, interaction_matrix, scores, output_file="interactive_gene_network.html", threshold_percentile=90):
    """
    Create an interactive gene network visualization using basic HTML/CSS/JS (no external dependencies)
    
    Parameters:
    -----------
    selected_features : list
        Names of the selected genes
    interaction_matrix : numpy.ndarray
        Matrix of interaction strengths between genes
    scores : numpy.ndarray
        Pathogenicity scores for each gene
    output_file : str
        Path to save the HTML output file
    threshold_percentile : int
        Percentile threshold for including edges
    """
    import numpy as np
    
    # Create a copy of the interaction matrix and set diagonal to 0
    interaction_matrix_copy = interaction_matrix.copy()
    np.fill_diagonal(interaction_matrix_copy, 0)
    
    # Calculate threshold for edges
    positive_interactions = interaction_matrix_copy[interaction_matrix_copy > 0]
    if len(positive_interactions) > 0:
        edge_threshold = np.percentile(positive_interactions, threshold_percentile)
    else:
        edge_threshold = 0.001
        print("No positive interactions found. Using default threshold of 0.001")
    
    # Create nodes and edges data
    nodes = []
    edges = []
    
    # Add nodes
    for i, gene in enumerate(selected_features):
        # Create a shorter display name for the gene
        display_name = get_short_gene_name_cpu(gene)
        nodes.append({
            'id': i,
            'name': gene,
            'display_name': display_name,
            'score': float(scores[i])
        })
    
    # Add edges based on interaction strengths
    for i in range(len(selected_features)):
        for j in range(i+1, len(selected_features)):
            interaction = interaction_matrix_copy[i, j]
            if interaction > edge_threshold:
                edges.append({
                    'source': i,
                    'target': j,
                    'weight': float(interaction)
                })
    
    # Create HTML content
    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Gene Interaction Network - Pathogenicity Driving Genes</title>
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 20px;
                background-color: #f5f5f5;
            }}
            .container {{
                max-width: 1200px;
                margin: 0 auto;
                background-color: white;
                padding: 20px;
                border-radius: 10px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.1);
            }}
            h1 {{
                color: #2c3e50;
                text-align: center;
            }}
            .network-info {{
                background-color: #e8f4fd;
                padding: 15px;
                border-radius: 5px;
                margin-bottom: 20px;
            }}
            .controls {{
                margin-bottom: 20px;
                padding: 10px;
                background-color: #f8f9fa;
                border-radius: 5px;
            }}
            .gene-list {{
                max-height: 400px;
                overflow-y: auto;
                border: 1px solid #ddd;
                padding: 10px;
                border-radius: 5px;
            }}
            .gene-item {{
                padding: 5px;
                margin: 2px 0;
                background-color: #f0f0f0;
                border-radius: 3px;
                cursor: pointer;
            }}
            .gene-item:hover {{
                background-color: #e0e0e0;
            }}
            .high-score {{
                background-color: #d4edda;
                border-left: 4px solid #28a745;
            }}
            .medium-score {{
                background-color: #fff3cd;
                border-left: 4px solid #ffc107;
            }}
            .low-score {{
                background-color: #f8d7da;
                border-left: 4px solid #dc3545;
            }}
            .stats {{
                display: flex;
                justify-content: space-around;
                margin-top: 20px;
                padding: 15px;
                background-color: #f8f9fa;
                border-radius: 5px;
            }}
            .stat {{
                text-align: center;
            }}
            .stat-number {{
                font-size: 24px;
                font-weight: bold;
                color: #2c3e50;
            }}
            .search-box {{
                width: 100%;
                padding: 8px;
                margin-bottom: 10px;
                border: 1px solid #ddd;
                border-radius: 4px;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <h1>Gene Interaction Network - Pathogenicity Driving Genes</h1>
            
            <div class="network-info">
                <h3>Network Summary</h3>
                <p><strong>Total Genes:</strong> {len(selected_features)}</p>
                <p><strong>Total Interactions:</strong> {len(edges)}</p>
                <p><strong>Edge Threshold:</strong> {edge_threshold:.4f} ({threshold_percentile}th percentile)</p>
                <p>This network shows genes that drive pathogenicity in NTMs. Gene scores represent their importance in pushing toward pathogenic classes.</p>
            </div>
            
            <div class="controls">
                <h3>Gene Search and Filter</h3>
                <input type="text" id="searchBox" class="search-box" placeholder="Search genes by name..." onkeyup="filterGenes()">
                <label>
                    <input type="checkbox" id="showHighScore" checked onchange="filterGenes()"> 
                    Show High Score Genes (top 33%)
                </label>
                <label>
                    <input type="checkbox" id="showMediumScore" checked onchange="filterGenes()"> 
                    Show Medium Score Genes (middle 33%)
                </label>
                <label>
                    <input type="checkbox" id="showLowScore" checked onchange="filterGenes()"> 
                    Show Low Score Genes (bottom 33%)
                </label>
            </div>
            
            <div class="gene-list" id="geneList">
                <!-- Genes will be populated by JavaScript -->
            </div>
            
            <div class="stats">
                <div class="stat">
                    <div class="stat-number" id="visibleGenes">{len(selected_features)}</div>
                    <div>Visible Genes</div>
                </div>
                <div class="stat">
                    <div class="stat-number">{len(edges)}</div>
                    <div>Total Interactions</div>
                </div>
                <div class="stat">
                    <div class="stat-number">{max(scores):.3f}</div>
                    <div>Max Score</div>
                </div>
                <div class="stat">
                    <div class="stat-number">{np.mean(scores):.3f}</div>
                    <div>Avg Score</div>
                </div>
            </div>
        </div>
        
        <script>
            // Gene data
            const nodes = {nodes};
            const edges = {edges};
            
            // Calculate score thresholds
            const scores = nodes.map(n => n.score).sort((a, b) => b - a);
            const highThreshold = scores[Math.floor(scores.length / 3)];
            const lowThreshold = scores[Math.floor(2 * scores.length / 3)];
            
            function getScoreClass(score) {{
                if (score >= highThreshold) return 'high-score';
                if (score >= lowThreshold) return 'medium-score';
                return 'low-score';
            }}
            
            function getInteractionCount(nodeId) {{
                return edges.filter(e => e.source === nodeId || e.target === nodeId).length;
            }}
            
            function renderGenes() {{
                const geneList = document.getElementById('geneList');
                const searchTerm = document.getElementById('searchBox').value.toLowerCase();
                const showHigh = document.getElementById('showHighScore').checked;
                const showMedium = document.getElementById('showMediumScore').checked;
                const showLow = document.getElementById('showLowScore').checked;
                
                let visibleCount = 0;
                geneList.innerHTML = '';
                
                // Sort nodes by score (descending)
                const sortedNodes = [...nodes].sort((a, b) => b.score - a.score);
                
                sortedNodes.forEach(node => {{
                    const scoreClass = getScoreClass(node.score);
                    const shouldShow = (
                        (scoreClass === 'high-score' && showHigh) ||
                        (scoreClass === 'medium-score' && showMedium) ||
                        (scoreClass === 'low-score' && showLow)
                    ) && (
                        searchTerm === '' || 
                        node.name.toLowerCase().includes(searchTerm) ||
                        node.display_name.toLowerCase().includes(searchTerm)
                    );
                    
                    if (shouldShow) {{
                        const interactionCount = getInteractionCount(node.id);
                        const geneItem = document.createElement('div');
                        geneItem.className = `gene-item ${{scoreClass}}`;
                        geneItem.innerHTML = `
                            <strong>${{node.display_name}}</strong><br>
                            <small>Score: ${{node.score.toFixed(4)}} | Interactions: ${{interactionCount}}</small><br>
                            <small style="color: #666;">${{node.name}}</small>
                        `;
                        geneItem.onclick = () => showGeneDetails(node);
                        geneList.appendChild(geneItem);
                        visibleCount++;
                    }}
                }});
                
                document.getElementById('visibleGenes').textContent = visibleCount;
            }}
            
            function filterGenes() {{
                renderGenes();
            }}
            
            function showGeneDetails(node) {{
                const interactions = edges.filter(e => e.source === node.id || e.target === node.id);
                const partners = interactions.map(e => {{
                    const partnerId = e.source === node.id ? e.target : e.source;
                    const partner = nodes[partnerId];
                    return `${{partner.display_name}} (strength: ${{e.weight.toFixed(4)}})`;
                }}).join('\\n');
                
                alert(`Gene Details:\\n\\nName: ${{node.name}}\\nDisplay Name: ${{node.display_name}}\\nPathogenicity Score: ${{node.score.toFixed(4)}}\\nInteraction Count: ${{interactions.length}}\\n\\nTop Interaction Partners:\\n${{partners || 'None'}}`);
            }}
            
            // Initialize
            renderGenes();
        </script>
    </body>
    </html>
    """
    
    # Save the HTML file
    with open(output_file, 'w') as f:
        f.write(html_content)
    
    print(f"Interactive network saved to {output_file}")


def get_short_gene_name_cpu(gene_string):
    """
    Extract a short displayable name from the full gene string
    
    Parameters:
    -----------
    gene_string : str
        Full gene name string with species identifiers
        
    Returns:
    --------
    str : A shorter, more displayable gene name
    """
    # Check if it's a group
    if gene_string.startswith('group_'):
        return gene_string.split('-')[0]
    
    # Check for multiple genes separated by ~~~
    if '~~~' in gene_string:
        parts = gene_string.split('~~~')
        # Take the first part
        gene_string = parts[0]
    
    # Check for species identifier pattern (sp-XXXXX-GENE_SPECIES)
    if gene_string.startswith('sp-'):
        parts = gene_string.split('-')
        if len(parts) >= 3:
            # Extract just the gene name part without species
            gene_part = parts[2].split('_')[0]
            return gene_part
        
    # If no specific pattern matched, return the first 10 characters
    return gene_string[:15]  # Slightly longer for better readability

In [22]:
def prefilter_genes(df_pa, target_column, n_top_genes=5000, min_prevalence=0.05, max_prevalence=0.95):
    """
    Pre-filter genes based on simple criteria before running expensive feature selection.
    This helps with extremely large gene sets by removing genes that are too rare or too common.
    
    Parameters:
    -----------
    df_pa : pandas DataFrame
        DataFrame with genes as columns and samples as rows
    target_column : str
        Name of the column containing pathogenicity values
    n_top_genes : int
        Number of genes to keep after initial filtering
    min_prevalence : float
        Minimum prevalence (fraction of samples where gene is present)
    max_prevalence : float
        Maximum prevalence (fraction of samples where gene is present)
        
    Returns:
    --------
    pandas DataFrame with reduced gene set
    """
    import pandas as pd
    import numpy as np
    from sklearn.feature_selection import chi2, SelectKBest
    import time
    
    start_time = time.time()
    print(f"Starting gene pre-filtering process...")
    
    # Check if we're actually filtering
    if n_top_genes >= df_pa.shape[1] - 1 and min_prevalence <= 0 and max_prevalence >= 1.0:
        print(f"No filtering requested (keeping all {df_pa.shape[1]-1} genes)")
        return df_pa
    
    # Separate features and target
    X_df = df_pa.drop(columns=[target_column])
    y = df_pa[target_column].values
    
    print(f"Original gene count: {X_df.shape[1]}")
    
    # If prevalence filtering is requested
    if min_prevalence > 0 or max_prevalence < 1.0:
        print(f"Filtering genes by prevalence ({min_prevalence}-{max_prevalence})...")
        
        # Calculate gene prevalence (fraction of samples where gene is present)
        gene_prevalence = X_df.mean().values
        
        # Filter genes based on prevalence
        prevalence_mask = (gene_prevalence >= min_prevalence) & (gene_prevalence <= max_prevalence)
        candidate_genes = X_df.columns[prevalence_mask].tolist()
        
        print(f"Filtered to {len(candidate_genes)} genes based on prevalence")
    else:
        candidate_genes = X_df.columns.tolist()
    
    # If we need to further reduce based on univariate statistics
    if len(candidate_genes) > n_top_genes:
        print(f"Further reducing to {n_top_genes} genes using univariate statistics...")
        
        # Convert target to categorical if it's continuous
        if np.issubdtype(type(y[0]), np.number):
            y_cat = np.array([int(val) for val in y])
        else:
            y_cat = y
        
        # Apply chi-square feature selection
        X_filtered = X_df[candidate_genes]
        
        # Handle sparse matrices properly
        if hasattr(X_filtered, 'sparse'):
            print("Converting sparse matrix to dense for chi-square test...")
            X_filtered_array = X_filtered.sparse.to_dense().values
        else:
            X_filtered_array = X_filtered.values
        
        # Run chi-square selection
        selector = SelectKBest(chi2, k=min(n_top_genes, len(candidate_genes)))
        selector.fit(X_filtered_array, y_cat)
        
        # Get the selected gene indices and names
        selected_indices = selector.get_support(indices=True)
        selected_genes = [candidate_genes[i] for i in selected_indices]
        
        print(f"Selected {len(selected_genes)} genes after chi-square filtering")
    else:
        selected_genes = candidate_genes
    
    # Create filtered dataframe
    filtered_df = df_pa[selected_genes + [target_column]]
    
    elapsed_time = time.time() - start_time
    print(f"Pre-filtering complete in {elapsed_time:.1f} seconds. Kept {len(selected_genes)} genes.")
    
    return filtered_df

In [23]:
def analyze_class_transition_drivers(
    df_pa, selected_genes, target_column="Oxygen2", 
    from_class=3, to_class=4, n_samples=100, output_dir="./transition_analysis"
):
    """
    Identify genes that specifically drive transitions between pathogenicity classes.
    
    Parameters:
    -----------
    df_pa : pandas DataFrame
        DataFrame with genes and pathogenicity scores
    selected_genes : list
        Pre-selected genes to analyze
    target_column : str
        Column containing pathogenicity values
    from_class : float/int
        Source pathogenicity class
    to_class : float/int
        Target (higher) pathogenicity class
    n_samples : int
        Number of samples to use for analysis
    output_dir : str
        Directory to save results
        
    Returns:
    --------
    DataFrame with genes ranked by transition importance
    """
    import shap
    import xgboost as xgb
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import os
    
    # Create output directory if needed
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Analyzing genes driving transition from class {from_class} to {to_class}...")
    
    # Create dataset with only the specified classes
    class_mask = (df_pa[target_column] == from_class) | (df_pa[target_column] == to_class)
    df_filtered = df_pa.loc[class_mask, selected_genes + [target_column]]
    
    if len(df_filtered) < 10:
        print(f"Warning: Not enough samples for classes {from_class} and {to_class}")
        return pd.DataFrame()
    
    print(f"Using {len(df_filtered)} samples for transition analysis ({np.sum(df_filtered[target_column] == from_class)} in class {from_class}, {np.sum(df_filtered[target_column] == to_class)} in class {to_class})")
    
    # Convert to binary classification problem
    y = (df_filtered[target_column] == to_class).astype(int).values
    X = df_filtered[selected_genes]
    
    # Convert to dense if sparse
    X_array = X.sparse.to_dense().values if hasattr(X, 'sparse') else X.values
    
    # Train a binary classifier using GPU
    model = xgb.XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        objective='binary:logistic',
        tree_method='hist',      # Updated
        device='cuda',           # Updated
        random_state=42
    )
    
    model.fit(X_array, y)
    
    # Calculate feature importance
    importances = model.feature_importances_
    importance_df = pd.DataFrame({
        'Gene': selected_genes,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    # Calculate SHAP values for deeper analysis
    explainer = shap.TreeExplainer(model)
    
    # Sample data if needed
    if len(X) > n_samples:
        np.random.seed(42)
        sample_indices = np.random.choice(len(X), size=n_samples, replace=False)
        X_sampled = X_array[sample_indices]
        y_sampled = y[sample_indices]
    else:
        X_sampled = X_array
        y_sampled = y
    
    # Get SHAP values
    shap_values = explainer.shap_values(X_sampled)
    
    # For binary classification, we want the positive class SHAP values
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # Positive class (transition to higher path)
    
    # Calculate mean positive SHAP values (pushing toward higher class)
    pos_shap = np.where(shap_values > 0, shap_values, 0).mean(axis=0)
    
    # Create results DataFrame
    results = pd.DataFrame({
        'Gene': selected_genes,
        f'Transition_{from_class}_to_{to_class}': pos_shap,
        'XGB_Importance': importances
    })
    
    # Sort by transition importance
    results = results.sort_values(f'Transition_{from_class}_to_{to_class}', ascending=False)
    
    # Save results
    results.to_csv(f"{output_dir}/transition_{from_class}_to_{to_class}_drivers.csv", index=False)
    
    # Create SHAP summary plot
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values, X_sampled, feature_names=selected_genes, show=False)
    plt.title(f"SHAP Summary for Class Transition {from_class} → {to_class}")
    plt.tight_layout()
    plt.savefig(f"{output_dir}/transition_{from_class}_to_{to_class}_shap.png", dpi=200)
    plt.close()
    
    print(f"Top 10 genes driving transition from {from_class} to {to_class}:")
    for i, gene in enumerate(results[f'Transition_{from_class}_to_{to_class}'].head(10).index):
        print(f"{i+1}. {results.loc[gene, 'Gene']} (score: {results.loc[gene, f'Transition_{from_class}_to_{to_class}']:.4f})")
    
    return results

In [24]:
# Create output directory
output_dir = './outputs/for_cliff'
os.makedirs(output_dir, exist_ok=True)

In [25]:
# Start timing
start_time = time.time()
print(f"Starting analysis pipeline at {time.strftime('%Y-%m-%d %H:%M:%S')}")

Starting analysis pipeline at 2025-10-22 17:40:43


In [26]:
# Modify doubled values to be consecutive integers - memory managment and XGBoost, don't ask
df_pa = pd.read_csv('./data/oxygen_pfams_phylogeny_all.csv', index_col=0)
cols_to_drop = ['Accession', 'phylum']  # put any columns you want to drop here
df_pa = df_pa.drop(columns=cols_to_drop, errors='ignore')
target_column = 'Oxygen2'
df_pa = df_pa.dropna(subset=[target_column])
# Create a mapping of original values to consecutive integers
original_values = sorted(df_pa[target_column].unique())
consecutive_mapping = {val: i for i, val in enumerate(original_values)}
# Apply the mapping
df_pa[target_column] = df_pa[target_column].map(consecutive_mapping)
# Verify the mapping
print("Original values:", original_values)
print("Mapped to:", list(consecutive_mapping.values()))
print("New unique values:", sorted(df_pa[target_column].unique()))
# Convert to sparse format
df_pa = df_pa.astype('Sparse[int]')

Original values: ['aerobic', 'anaerobic']
Mapped to: [0, 1]
New unique values: [0, 1]


In [27]:
# 2. Pre-filter the large gene set (or skip filtering with these parameters)
print("\n=== PRE-FILTERING GENES ===")
# To keep all genes: n_top_genes=df_pa.shape[1], min_prevalence=0.0, max_prevalence=1.0
filtered_df = prefilter_genes(
    df_pa=df_pa,
    target_column="Oxygen2",
    n_top_genes=df_pa.shape[1], # Alter based on memory contraints
    min_prevalence=0.0, # Probably don't want to mess with this as I'm guessing there will be fairly few genes driving pathogenicity compared to the size of the dataset and how unbalanced it is
    max_prevalence=1.0 # I don't think there are any ubiquitous genes, but might be worth playing around with this
)

# Clear memory
gc.collect()


=== PRE-FILTERING GENES ===
Starting gene pre-filtering process...
No filtering requested (keeping all 2373 genes)


320128

In [28]:
filtered_df

,PF00011.27,PF00012.26,PF00013.35,PF00015.26,PF00022.25,PF00025.27,PF00027.35,PF00034.26,PF00035.32,PF00037.32,...,PF04297.20,PF04472.17,PF05167.17,PF06949.16,PF22527.2,PF04116.19,PF05721.19,PF08818.16,PF21926.2,Oxygen2
Accession,,,,,,,,,,,,,,,,,,,,,
GCA_000006945.2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
GCA_000006985.1,1,1,1,0,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,1
GCA_000007145.1,1,1,1,1,1,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
GCA_000007985.2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,1
GCA_000008185.1,1,1,1,1,1,1,1,0,1,1,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCA_905397435.1,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,1,1,1,0,0
GCA_914271545.1,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,1,1,1,1,0
GCA_919592095.1,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,1,1,0,1,0


In [29]:
# 3. Run the feature selection with interactions

print("\n=== PATHOGENICITY-DRIVEN FEATURE SELECTION ===")

'''
selected_genes, gene_importance = importance_driven_feature_selection_gpu(
    df_pa=filtered_df,
    target_column="Pathogenicity_scale",
    n_features_to_select=300, # 300 seems reasonable, but you can of course change this
    scoring_strategy='high_class_emphasis',
    high_importance_threshold=4, # Since the purpose of this function is to find genes that drive a genotype towards being potentially pathogenic, use this to set a threshold of what you want to consider "clinically interesting"
    interaction_analysis=True, # Computationally intensive; There is some cluster-based logic in the function that tries to capture cases where two or more genes work together to drive pathogenicity; no idea if there are any cases of this at all, but it's potentially useful
    n_iterations=3, # Or increase it if you're feeling particularly concerned about validity of the feature subset
    output_dir=output_dir,
    gpu_id=0,                  # Assuming first GPU
    max_gpu_memory_fraction=0.8 # Adjust based on your needs
)
'''

selected_genes, gene_importance = importance_driven_feature_selection_cpu(
    df_pa=filtered_df,
    target_column=target_column,
    scoring_strategy='equal_classes',
    n_features_to_select=20, # 300 seems reasonable, but you can of course change this
    high_importance_threshold=4, # Since the purpose of this function is to find genes that drive a genotype towards being potentially pathogenic, use this to set a threshold of what you want to consider "clinically interesting"
    interaction_analysis=True, # Computationally intensive; There is some cluster-based logic in the function that tries to capture cases where two or more genes work together to drive pathogenicity; no idea if there are any cases of this at all, but it's potentially useful
    n_iterations=10, # Or increase it if you're feeling particularly concerned about validity of the feature subset
    output_dir=output_dir,
    memory_efficient=True,
    n_jobs=-1,  # Set to -1 use all CPU cores
    initial_features_per_model=2373
)

# Clear memory
gc.collect()


=== PATHOGENICITY-DRIVEN FEATURE SELECTION ===
Available system memory: 15.68 GB
CPU cores available: 16
Using 16 CPU cores for parallel processing
Starting CPU-based pathogenicity-driven feature selection on 2373 genes...
Analysis will include gene-gene interactions: True
Running 10 iterations for stability...
Memory-efficient processing: True
Available system memory: 15.68 GB
Using 2 numeric classes: {0: 0, 1: 1}
Binary classification detected. Adjusting analysis approach...
Binary + equal_classes: Treating both classes equally

Iteration 1/10 (seed: 42)
Training set: (4416, 2373), Test set: (1104, 2373)
Class distribution in training: [3382 1034]
STAGE 1: Pre-selecting features...
CatBoost added to analysis.
Training xgb model for initial feature screening...
Converting sparse matrix to dense for model training...
Training rf model for initial feature screening...
Converting sparse matrix to dense for model training...
Training catboost model for initial feature screening...
Conver

201770

In [ ]:
# 4. Analyze specific class transitions
print("\n=== ANALYZING CLASS TRANSITIONS ===")
transitions_to_analyze = [
    (0, 1),
    (1, 0)
]

transition_dir = os.path.join(output_dir, "transitions")
os.makedirs(transition_dir, exist_ok=True)

for from_class, to_class in transitions_to_analyze:
    transition_df = analyze_class_transition_drivers(
        df_pa=df_pa, 
        selected_genes=selected_genes,
        target_column=target_column,
        from_class=from_class,
        to_class=to_class,
        output_dir=transition_dir
    )


In [30]:
# 5. Create a dataset with only the selected genes for model training
print("\n=== CREATING FINAL DATASET ===")
final_df = df_pa[selected_genes + ['Oxygen2']]
final_df.to_csv(f"{output_dir}/selected_genes_dataset.csv")

total_time = time.time() - start_time
print(f"\n=== ANALYSIS COMPLETE ===")
print(f"Total time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"Results saved to {output_dir}/")
print(f"Selected {len(selected_genes)} oxygen-predicting genes")
print(f"Top 10 oxygen-predicting genes:")
print(gene_importance.head(10)[['Feature', 'Aggregated_Score']])


=== CREATING FINAL DATASET ===

=== ANALYSIS COMPLETE ===
Total time: 2710.8 seconds (45.2 minutes)
Results saved to ./outputs/for_cliff/
Selected 20 oxygen-predicting genes
Top 10 oxygen-predicting genes:
       Feature  Aggregated_Score
19  PF00510.24          0.166574
18  PF01871.22          0.078706
17  PF00296.26          0.057669
16  PF10371.14          0.054521
15   PF17910.6          0.044074
14  PF02579.22          0.040418
13  PF16870.10          0.039624
12  PF00116.26          0.036855
11  PF13597.11          0.032397
10  PF03063.25          0.028553


In [31]:
selected_genes

['PF05721.19',
 'PF00115.25',
 'PF01152.27',
 'PF08530.16',
 'PF00042.28',
 'PF02906.19',
 'PF00916.26',
 'PF17773.7',
 'PF05425.18',
 'PF01521.26',
 'PF03063.25',
 'PF13597.11',
 'PF00116.26',
 'PF16870.10',
 'PF02579.22',
 'PF17910.6',
 'PF10371.14',
 'PF00296.26',
 'PF01871.22',
 'PF00510.24']

In [32]:
trimmed_genes = [gene.split('.')[0] for gene in selected_genes]
trimmed_genes

['PF05721',
 'PF00115',
 'PF01152',
 'PF08530',
 'PF00042',
 'PF02906',
 'PF00916',
 'PF17773',
 'PF05425',
 'PF01521',
 'PF03063',
 'PF13597',
 'PF00116',
 'PF16870',
 'PF02579',
 'PF17910',
 'PF10371',
 'PF00296',
 'PF01871',
 'PF00510']

In [33]:
genes_new = ['PF10371',
 'PF12641',
 'PF00494',
 'PF16177',
 'PF04134',
 'PF13564',
 'PF00115',
 'PF03441',
 'PF01329',
 'PF01521',
 'PF16870',
 'PF03063',
 'PF17769',
 'PF10418',
 'PF05163',
 'PF16078',
 'PF21349',
 'PF04241',
 'PF17910',
 'PF02628',
 'PF00970',
 'PF04108',
 'PF00916',
 'PF01867',
 'PF17179',
 'PF02817',
 'PF02256',
 'PF01155',
 'PF02678',
 'PF05425',
 'PF01871',
 'PF12392',
 'PF16901',
 'PF09989',
 'PF13597',
 'PF00296',
 'PF00116',
 'PF00510']


genes_old = ['PF00042',
 'PF05721',
 'PF02906',
 'PF01152',
 'PF08530',
 'PF00916',
 'PF00115',
 'PF17773',
 'PF05425',
 'PF01521',
 'PF03063',
 'PF13597',
 'PF00116',
 'PF16870',
 'PF02579',
 'PF17910',
 'PF10371',
 'PF00296',
 'PF01871',
 'PF00510']
# Assuming you already have two Python lists:
# genes_old = [...]
# genes_new = [...]

old_set = set(genes_old)
new_set = set(genes_new)

overlap      = sorted(old_set & new_set)
only_in_old  = sorted(old_set - new_set)   # in old, not in new
only_in_new  = sorted(new_set - old_set)   # in new, not in old

print(f"Overlap ({len(overlap)}):", overlap[:20])
print(f"Only in old ({len(only_in_old)}):", only_in_old[:20])
print(f"Only in new ({len(only_in_new)}):", only_in_new[:20])



Overlap (20): ['PF00042', 'PF00115', 'PF00116', 'PF00296', 'PF00510', 'PF00916', 'PF01152', 'PF01521', 'PF01871', 'PF02579', 'PF02906', 'PF03063', 'PF05425', 'PF05721', 'PF08530', 'PF10371', 'PF13597', 'PF16870', 'PF17773', 'PF17910']
Only in old (0): []
Only in new (0): []


In [34]:
# Need to get the gene names for those 20 genes
# Read the tab-delimited file with no header
pfam_full = pd.read_csv("pfamA.txt", sep="\t", header=None, dtype=str)

# Select columns 0 to 3 (V1–V4 in R)
pfam_full = pfam_full[[0, 1, 2, 3]]

# Rename the columns
pfam_full.columns = ["Pfam", "Name", "Code", "Description"]

# Filter to just get the 20 genes
pfam_subset = pfam_full[pfam_full["Pfam"].isin(trimmed_genes)]

FileNotFoundError: [Errno 2] No such file or directory: 'pfamA.txt'

In [86]:
pfam_subset

,Pfam,Name,Code,Description
41,PF00042,Globin,globin;,Globin
112,PF00115,COX1,NaN,Cytochrome C and Quinol oxidase polypeptide I
113,PF00116,COX2,NaN,"Cytochrome C oxidase subunit II, periplasmic d..."
286,PF00296,Bac_luciferase,bac_luciferase;,Luciferase-like monooxygenase
489,PF00510,COX3,NaN,Cytochrome c oxidase subunit III
878,PF00916,Sulfate_transp,NaN,Sulfate permease family
1100,PF01152,Bac_globin,Globin;,Bacterial-like globin
1440,PF01521,Fe-S_biosyn,HesB-like;HesB;,Iron-sulphur cluster biosynthesis
1757,PF01871,AMMECR1,DUF51;,AMMECR1
2389,PF02579,Nitro_FeMo-Co,DUF153;,Dinitrogenase iron-molybdenum cofactor


## Main figure: Top 20 genes from feature selection

In [76]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

def get_short_gene_name(gene_string):
    """
    Extract a short displayable name from the full gene string
    """
    # Check if it's a group
    if gene_string.startswith('group_'):
        return gene_string.split('-')[0]
    
    # Check for multiple genes separated by ~~~
    if '~~~' in gene_string:
        parts = gene_string.split('~~~')
        # Take the first part
        gene_string = parts[0]
    
    # Check for species identifier pattern (sp-XXXXX-GENE_SPECIES)
    if gene_string.startswith('sp-'):
        parts = gene_string.split('-')
        if len(parts) >= 3:
            # Extract just the gene name part without species
            gene_part = parts[2].split('_')[0]
            return gene_part
        
    # If no specific pattern matched, return the first 10 characters
    return gene_string[:10]

def format_gene_names(gene_list, use_simple_names=True, max_length=18):
    """
    Format a list of gene names consistently
    """
    formatted_names = []
    
    for gene in gene_list:
        if use_simple_names:
            # Cut at first ~~~
            formatted = gene.split('~~~')[0]
        else:
            # Use sophisticated logic
            formatted = get_short_gene_name(gene)
        
        # Truncate if too long
        if len(formatted) > max_length:
            formatted = formatted[:max_length-3] + "..."
            
        formatted_names.append(formatted)
    
    return formatted_names

def create_top_selected_genes_figure(
    feature_selection_file="./May/ntm_pathogenicity_model/selected_important_features.csv",
    output_dir="./conference_figures",
    top_n=20,
    figure_type="horizontal_bar",
    style="publication",
    use_simple_names=True,
    max_gene_name_length=20,
    font_size_small=8,
    font_size_medium=10,
    font_size_large=12,
    font_size_title=14,
    dpi=300
):
    """
    Create visualization of top selected genes from feature selection process
    
    Parameters:
    -----------
    feature_selection_file : str
        Path to the selected_important_features.csv file
    output_dir : str
        Directory to save the figure
    top_n : int
        Number of top genes to show
    figure_type : str
        Type of visualization:
        - 'horizontal_bar': Horizontal bar plot (recommended)
        - 'vertical_bar': Vertical bar plot
        - 'lollipop': Lollipop plot
        - 'heatmap': Heatmap showing scores across iterations
    style : str
        Figure style ('publication', 'poster', 'presentation')
    use_simple_names : bool
        If True, cut gene names at first '~~~'. If False, use sophisticated logic
    max_gene_name_length : int
        Maximum length for gene names
    font_size_small : int
        Font size for small text
    font_size_medium : int
        Font size for medium text
    font_size_large : int
        Font size for large text
    font_size_title : int
        Font size for title
    dpi : int
        Resolution for saved figure
        
    Returns:
    --------
    str : Path to saved figure
    """
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Set style
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        figure_size = (10, 8)
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        figure_size = (12, 10)
    else:  # presentation
        plt.style.use('seaborn-v0_8-darkgrid')
        figure_size = (11, 9)
    
    plt.rcParams.update({
        'font.size': font_size_medium,
        'axes.titlesize': font_size_large,
        'axes.labelsize': font_size_medium,
        'xtick.labelsize': font_size_small,
        'ytick.labelsize': font_size_small,
        'legend.fontsize': font_size_medium,
        'figure.titlesize': font_size_title
    })
    
    # Load the feature selection results
    print(f"Loading feature selection results from {feature_selection_file}...")
    df = pd.read_csv(feature_selection_file)
    
    # Sort by aggregated score and get top N
    df_sorted = df.sort_values('Aggregated_Score', ascending=False).head(top_n)
    
    # Format gene names
    formatted_names = format_gene_names(
        df_sorted['Feature'].tolist(),
        use_simple_names=use_simple_names,
        max_length=max_gene_name_length
    )
    
    print(f"Creating {figure_type} plot for top {top_n} genes...")
    
    if figure_type == "horizontal_bar":
        # Horizontal bar plot (recommended for gene names)
        plt.figure(figsize=figure_size)
        
        # Create color gradient
        colors = plt.cm.viridis(np.linspace(0, 1, len(df_sorted)))
        
        bars = plt.barh(range(len(df_sorted)), df_sorted['Aggregated_Score'], 
                       color=colors)
        
        plt.yticks(range(len(df_sorted)), formatted_names, fontsize=font_size_small)
        plt.xlabel("Aggregated Feature Selection Score", fontsize=font_size_medium)
        plt.title(f"Top {top_n} Genes from Multi-Model Feature Selection\n(Aggregated across all models and iterations)", 
                 fontsize=font_size_large, pad=20)
        
        # Add value labels on bars
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width + 0.002, bar.get_y() + bar.get_height()/2, 
                    f'{width:.3f}', ha='left', va='center', fontsize=font_size_small)
        
        # Invert y-axis so highest score is at top
        plt.gca().invert_yaxis()
        plt.tight_layout()
        
        filename = f"top_{top_n}_selected_genes_horizontal.png"
        
    elif figure_type == "vertical_bar":
        # Vertical bar plot
        plt.figure(figsize=(figure_size[0], figure_size[1]))
        
        colors = plt.cm.viridis(np.linspace(0, 1, len(df_sorted)))
        
        bars = plt.bar(range(len(df_sorted)), df_sorted['Aggregated_Score'], 
                      color=colors)
        
        plt.xticks(range(len(df_sorted)), formatted_names, 
                  rotation=45, ha='right', fontsize=font_size_small)
        plt.ylabel("Aggregated Feature Selection Score", fontsize=font_size_medium)
        plt.title(f"Top {top_n} Genes from Multi-Model Feature Selection", 
                 fontsize=font_size_large, pad=20)
        
        # Add value labels on bars
        for i, bar in enumerate(bars):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2, height + 0.002, 
                    f'{height:.3f}', ha='center', va='bottom', 
                    fontsize=font_size_small, rotation=90)
        
        plt.tight_layout()
        filename = f"top_{top_n}_selected_genes_vertical.png"
        
    elif figure_type == "lollipop":
        # Lollipop plot
        plt.figure(figsize=figure_size)
        
        y_pos = range(len(df_sorted))
        scores = df_sorted['Aggregated_Score']
        
        # Create stems
        plt.hlines(y=y_pos, xmin=0, xmax=scores, color='skyblue', alpha=0.7, linewidth=2)
        
        # Create lollipops
        plt.scatter(scores, y_pos, color='crimson', s=60, alpha=0.8, zorder=10)
        
        plt.yticks(y_pos, formatted_names, fontsize=font_size_small)
        plt.xlabel("Aggregated Feature Selection Score", fontsize=font_size_medium)
        plt.title(f"Top {top_n} Genes from Multi-Model Feature Selection", 
                 fontsize=font_size_large, pad=20)
        
        # Add value labels
        for i, score in enumerate(scores):
            plt.text(score + 0.002, i, f'{score:.3f}', 
                    va='center', fontsize=font_size_small)
        
        plt.gca().invert_yaxis()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        filename = f"top_{top_n}_selected_genes_lollipop.png"
        
    elif figure_type == "heatmap":
        # Heatmap showing scores across iterations
        plt.figure(figsize=(figure_size[0], figure_size[1]*0.8))
        
        # Prepare data for heatmap
        iteration_cols = ['Iteration_1_Score', 'Iteration_2_Score', 'Iteration_3_Score']
        heatmap_data = df_sorted[iteration_cols + ['Aggregated_Score']].values
        
        # Create heatmap
        sns.heatmap(heatmap_data, 
                   xticklabels=['Iter 1', 'Iter 2', 'Iter 3', 'Aggregated'],
                   yticklabels=formatted_names,
                   cmap='YlOrRd',
                   annot=True,
                   fmt='.3f',
                   cbar_kws={'label': 'Feature Selection Score'})
        
        plt.title(f"Top {top_n} Genes: Scores Across Iterations", 
                 fontsize=font_size_large, pad=20)
        plt.xlabel("Analysis", fontsize=font_size_medium)
        plt.ylabel("Gene", fontsize=font_size_medium)
        plt.xticks(fontsize=font_size_small)
        plt.yticks(fontsize=font_size_small)
        plt.tight_layout()
        
        filename = f"top_{top_n}_selected_genes_heatmap.png"
    
    # Save figure
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"Top selected genes figure saved to {filepath}")
    
    # Print summary statistics
    print(f"\nTop {top_n} genes summary:")
    print(f"Highest score: {df_sorted['Aggregated_Score'].iloc[0]:.4f}")
    print(f"Lowest score: {df_sorted['Aggregated_Score'].iloc[-1]:.4f}")
    print(f"Mean score: {df_sorted['Aggregated_Score'].mean():.4f}")
    
    # Reset matplotlib params
    plt.rcParams.update(plt.rcParamsDefault)
    
    return filepath

def create_comprehensive_gene_analysis(
    feature_selection_file="./May/ntm_pathogenicity_model/selected_important_features.csv",
    output_dir="./conference_figures",
    top_n=20,
    style="publication",
    use_simple_names=True,
    font_size_small=10,
    font_size_medium=12,
    font_size_large=14,
    font_size_title=16
):
    """
    Create a comprehensive analysis of the top selected genes including:
    1. Main ranking plot
    2. Score consistency across iterations
    3. Interaction strength analysis
    """
    
    print("Creating comprehensive gene selection analysis...")
    
    # Load data
    df = pd.read_csv(feature_selection_file)
    df_top = df.sort_values('Aggregated_Score', ascending=False).head(top_n)
    
    # Set up the figure
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        fig_size = (15, 10)
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        fig_size = (18, 12)
    else:
        plt.style.use('seaborn-v0_8-darkgrid')
        fig_size = (16, 11)
    
    plt.rcParams.update({
        'font.size': font_size_medium,
        'axes.titlesize': font_size_large,
        'axes.labelsize': font_size_medium,
        'xtick.labelsize': font_size_small,
        'ytick.labelsize': font_size_small,
        'legend.fontsize': font_size_small
    })
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=fig_size)
    
    # Format gene names
    formatted_names = format_gene_names(
        df_top['Feature'].tolist(),
        use_simple_names=use_simple_names,
        max_length=15  # Shorter for multi-panel
    )
    
    # Panel 1: Main ranking (top 15 for readability)
    top_15 = df_top.head(15)
    colors = plt.cm.viridis(np.linspace(0, 1, len(top_15)))
    
    bars = ax1.barh(range(len(top_15)), top_15['Aggregated_Score'], color=colors)
    ax1.set_yticks(range(len(top_15)))
    ax1.set_yticklabels(formatted_names[:15], fontsize=font_size_small)
    ax1.set_xlabel("Aggregated Score", fontsize=font_size_medium)
    ax1.set_title("A. Top 15 Selected Genes", fontweight='bold', fontsize=font_size_large)
    ax1.invert_yaxis()
    
    # Panel 2: Score consistency across iterations
    iteration_cols = ['Iteration_1_Score', 'Iteration_2_Score', 'Iteration_3_Score']
    
    for i, col in enumerate(iteration_cols):
        ax2.scatter(df_top[col], df_top['Aggregated_Score'], 
                   alpha=0.7, label=f'Iteration {i+1}', s=30)
    
    ax2.set_xlabel("Individual Iteration Score", fontsize=font_size_medium)
    ax2.set_ylabel("Aggregated Score", fontsize=font_size_medium)
    ax2.set_title("B. Score Consistency", fontweight='bold', fontsize=font_size_large)
    ax2.legend(fontsize=font_size_small)
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Interaction strength analysis
    # Remove NaN values for interaction strength
    df_interactions = df_top.dropna(subset=['Interaction_Strength'])
    
    if len(df_interactions) > 0:
        ax3.scatter(df_interactions['Aggregated_Score'], 
                   df_interactions['Interaction_Strength'],
                   alpha=0.7, color='coral', s=40)
        ax3.set_xlabel("Aggregated Score", fontsize=font_size_medium)
        ax3.set_ylabel("Interaction Strength", fontsize=font_size_medium)
        ax3.set_title("C. Score vs Interactions", fontweight='bold', fontsize=font_size_large)
        ax3.grid(True, alpha=0.3)
    else:
        ax3.text(0.5, 0.5, 'No interaction data\navailable', 
                ha='center', va='center', transform=ax3.transAxes,
                fontsize=font_size_medium)
        ax3.set_title("C. Interactions (N/A)", fontweight='bold', fontsize=font_size_large)
    
    # Panel 4: Gene clusters distribution
    if 'Gene_Cluster' in df_top.columns:
        cluster_counts = df_top['Gene_Cluster'].value_counts().head(8)
        ax4.bar(range(len(cluster_counts)), cluster_counts.values, 
               color=plt.cm.Set3(np.linspace(0, 1, len(cluster_counts))))
        ax4.set_xticks(range(len(cluster_counts)))
        ax4.set_xticklabels([f'C{c}' for c in cluster_counts.index], fontsize=font_size_small)
        ax4.set_xlabel("Gene Cluster", fontsize=font_size_medium)
        ax4.set_ylabel("Number of Genes", fontsize=font_size_medium)
        ax4.set_title("D. Cluster Distribution", fontweight='bold', fontsize=font_size_large)
    else:
        ax4.text(0.5, 0.5, 'No cluster data\navailable', 
                ha='center', va='center', transform=ax4.transAxes,
                fontsize=font_size_medium)
        ax4.set_title("D. Clusters (N/A)", fontweight='bold', fontsize=font_size_large)
    
    # Add main title
    fig.suptitle(f"Comprehensive Analysis of Top {top_n} Selected Pathogenicity Genes", 
                fontsize=font_size_title, fontweight='bold', y=0.95)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.92])
    
    # Save figure
    filename = f"comprehensive_gene_selection_analysis_top{top_n}.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Comprehensive analysis saved to {filepath}")
    
    # Reset matplotlib params
    plt.rcParams.update(plt.rcParamsDefault)
    
    return filepath
'''
# Example usage
if __name__ == "__main__":
    feature_file = "./May/ntm_pathogenicity_model/selected_important_features.csv"
    output_dir = "./conference_figures"
    
    # Create different visualizations
    print("Creating gene selection visualizations...")
    
    # 1. Horizontal bar plot (recommended)
    create_top_selected_genes_figure(
        feature_selection_file=feature_file,
        output_dir=output_dir,
        top_n=20,
        figure_type="horizontal_bar",
        style="publication",
        use_simple_names=True,
        max_gene_name_length=20
    )
    
    # 2. Heatmap showing iteration consistency
    create_top_selected_genes_figure(
        feature_selection_file=feature_file,
        output_dir=output_dir,
        top_n=15,
        figure_type="heatmap",
        style="publication",
        use_simple_names=True
    )
    
    # 3. Comprehensive analysis
    create_comprehensive_gene_analysis(
        feature_selection_file=feature_file,
        output_dir=output_dir,
        top_n=20,
        style="publication",
        use_simple_names=True
    )
    
    print("All gene selection figures created!")
'''

'\n# Example usage\nif __name__ == "__main__":\n    feature_file = "./May/ntm_pathogenicity_model/selected_important_features.csv"\n    output_dir = "./conference_figures"\n    \n    # Create different visualizations\n    print("Creating gene selection visualizations...")\n    \n    # 1. Horizontal bar plot (recommended)\n    create_top_selected_genes_figure(\n        feature_selection_file=feature_file,\n        output_dir=output_dir,\n        top_n=20,\n        figure_type="horizontal_bar",\n        style="publication",\n        use_simple_names=True,\n        max_gene_name_length=20\n    )\n    \n    # 2. Heatmap showing iteration consistency\n    create_top_selected_genes_figure(\n        feature_selection_file=feature_file,\n        output_dir=output_dir,\n        top_n=15,\n        figure_type="heatmap",\n        style="publication",\n        use_simple_names=True\n    )\n    \n    # 3. Comprehensive analysis\n    create_comprehensive_gene_analysis(\n        feature_selection_fil

In [77]:
# Main figure: Top 20 genes from feature selection
create_top_selected_genes_figure(
    feature_selection_file="selected_important_features.csv",
    output_dir="./conference_figures",
    top_n=20,
    figure_type="horizontal_bar",  # Best for gene names
    style="publication",
    use_simple_names=True,
    max_gene_name_length=24
)

Loading feature selection results from selected_important_features.csv...
Creating horizontal_bar plot for top 20 genes...
Top selected genes figure saved to ./conference_figures/top_20_selected_genes_horizontal.png

Top 20 genes summary:
Highest score: 0.1548
Lowest score: 0.0164
Mean score: 0.0384


'./conference_figures/top_20_selected_genes_horizontal.png'

## Conference Figure 2

In [91]:
final_df = pd.read_csv(f"{output_dir}/selected_genes_dataset.csv",index_col=0)
shap_output_dir = os.path.join(output_dir, "final_shap_analysis")
os.makedirs(shap_output_dir, exist_ok=True)
selected_genes = [col for col in final_df.columns if col != 'Oxygen2']

In [92]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import os

def create_comprehensive_shap_plots(
    df_pa, 
    selected_genes, 
    target_column="Pathogenicity_scale",
    output_dir="./shap_analysis",
    top_n_features=30,
    high_importance_threshold=4,
    max_samples=100,
    random_state=42,
    use_simple_names=True,
    font_size_small=8,
    font_size_medium=10,
    font_size_large=12,
    font_size_title=14,
    n_class_specific=15,
    n_comparison=20,
    n_interactions=20
):
    """
    Create comprehensive SHAP visualizations for the top selected features.
    
    Parameters:
    -----------
    df_pa : pandas DataFrame
        DataFrame with selected genes and target
    selected_genes : list
        List of selected gene names
    target_column : str
        Name of target column
    output_dir : str
        Directory to save plots
    top_n_features : int
        Number of top features for main plots (summary bar and beeswarm)
    high_importance_threshold : int
        Threshold for pathogenic classes
    max_samples : int
        Maximum samples for SHAP calculation
    random_state : int
        Random seed for reproducibility
    use_simple_names : bool
        If True, cut gene names at first '~~~'. If False, use original short name logic
    font_size_small : int
        Font size for small text (axis labels, tick labels)
    font_size_medium : int
        Font size for medium text (axis titles)
    font_size_large : int
        Font size for large text (plot titles)
    font_size_title : int
        Font size for main titles
    n_class_specific : int
        Number of features to show in each class-specific subplot
    n_comparison : int
        Number of features to show in pathogenic vs non-pathogenic comparison
    n_interactions : int
        Number of features to include in interaction heatmap
    """
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Set matplotlib font sizes
    plt.rcParams.update({
        'font.size': font_size_medium,
        'axes.titlesize': font_size_large,
        'axes.labelsize': font_size_medium,
        'xtick.labelsize': font_size_small,
        'ytick.labelsize': font_size_small,
        'legend.fontsize': font_size_medium,
        'figure.titlesize': font_size_title
    })
    
    print(f"Creating comprehensive SHAP plots for top {top_n_features} features...")
    
    # Define gene name function based on toggle
    def format_gene_name(gene_string):
        if use_simple_names:
            # Cut at first ~~~
            return gene_string.split('~~~')[0]
        else:
            # Use original logic
            return get_short_gene_name(gene_string)
    
    # Prepare data
    X = df_pa[selected_genes]
    y = df_pa[target_column].values
    
    # Convert sparse to dense if needed
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X_array, y, test_size=0.3, random_state=random_state, stratify=y
    )
    
    # Train model for SHAP analysis
    print("Training XGBoost model for SHAP analysis...")
    objective = 'binary:logistic' if is_binary else 'multi:softmax'
    model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective=objective,
        num_class=None if is_binary else len(np.unique(y)),
        tree_method='hist',
        device='cuda',
        random_state=random_state
    )
    
    model.fit(X_train, y_train)
    
    # Sample data for SHAP calculation
    if len(X_test) > max_samples:
        np.random.seed(random_state)
        sample_indices = np.random.choice(len(X_test), size=max_samples, replace=False)
        X_shap = X_test[sample_indices]
        y_shap = y_test[sample_indices]
    else:
        X_shap = X_test
        y_shap = y_test
    
    # Calculate SHAP values
    print("Calculating SHAP values...")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)
    
    # Determine pathogenic classes
    unique_classes = np.unique(y)
    high_importance_classes = [i for i in unique_classes if i >= high_importance_threshold]
    
    # 1. SHAP Summary Plot (Bar) - Overall Feature Importance
    print("Creating SHAP summary bar plot...")
    plt.figure(figsize=(12, 10))
    
    if isinstance(shap_values, list):
        # For multi-class, sum absolute SHAP values ONLY for pathogenic classes
        pathogenic_shap_combined = None
        pathogenic_count = 0
        
        for class_idx in high_importance_classes:
            if class_idx < len(shap_values):
                if pathogenic_shap_combined is None:
                    pathogenic_shap_combined = np.abs(shap_values[class_idx])
                else:
                    pathogenic_shap_combined += np.abs(shap_values[class_idx])
                pathogenic_count += 1
        
        if pathogenic_shap_combined is not None and pathogenic_count > 0:
            # Calculate mean importance per feature (focused on pathogenic classes)
            feature_importance = np.mean(pathogenic_shap_combined, axis=0)
        else:
            # Fallback to all classes if no pathogenic classes found
            print("Warning: No pathogenic classes found, using all classes for ranking")
            combined_shap = np.abs(shap_values[0])
            for i in range(1, len(shap_values)):
                combined_shap += np.abs(shap_values[i])
            feature_importance = np.mean(combined_shap, axis=0)
        
        # Get top features
        top_indices = np.argsort(feature_importance)[-top_n_features:][::-1]
        top_features = [selected_genes[i] for i in top_indices]
        top_importance = feature_importance[top_indices]
        
        print(f"Ranking features based on importance for pathogenic classes: {high_importance_classes}")
        
        # Create bar plot
        y_pos = np.arange(len(top_features))
        plt.barh(y_pos, top_importance, color='skyblue')
        plt.yticks(y_pos, [format_gene_name(gene) for gene in top_features], fontsize=font_size_small)
        plt.xlabel('Mean |SHAP Value| (Pathogenic Classes)', fontsize=font_size_medium)
        plt.title(f'Top {top_n_features} Features by Pathogenic Class Importance', fontsize=font_size_large)
        plt.gca().invert_yaxis()
        
    plt.tight_layout()
    plt.savefig(f"{output_dir}/shap_summary_bar.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. SHAP Summary Plot (Beeswarm) for Top Features
    print("Creating SHAP beeswarm plot...")
    plt.figure(figsize=(12, 10))
    
    if isinstance(shap_values, list):
        # Focus on pathogenic classes
        pathogenic_shap = None
        for class_idx in high_importance_classes:
            if class_idx < len(shap_values):
                if pathogenic_shap is None:
                    pathogenic_shap = shap_values[class_idx].copy()
                else:
                    pathogenic_shap += shap_values[class_idx]
        
        if pathogenic_shap is not None:
            # Select top features for visualization
            X_shap_top = X_shap[:, top_indices]
            pathogenic_shap_top = pathogenic_shap[:, top_indices]
            top_feature_names = [format_gene_name(gene) for gene in top_features]
            
            shap.summary_plot(
                pathogenic_shap_top, 
                X_shap_top, 
                feature_names=top_feature_names,
                show=False,
                max_display=top_n_features
            )
            plt.title('SHAP Values for Pathogenic Class Prediction', fontsize=font_size_large)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/shap_beeswarm.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 3. Class-Specific SHAP Analysis
    print("Creating class-specific SHAP plots...")
    if isinstance(shap_values, list):
        # Create a figure with subplots for each class
        n_classes = len(shap_values)
        fig, axes = plt.subplots(2, (n_classes + 1) // 2, figsize=(15, 10))
        axes = axes.flatten() if n_classes > 1 else [axes]
        
        for class_idx in range(n_classes):
            ax = axes[class_idx]
            
            # Calculate mean SHAP values for this class
            class_shap_mean = np.mean(np.abs(shap_values[class_idx]), axis=0)
            
            # Get top features for this class (using n_class_specific parameter)
            class_top_indices = np.argsort(class_shap_mean)[-n_class_specific:][::-1]
            class_top_features = [format_gene_name(selected_genes[i]) for i in class_top_indices]
            class_top_values = class_shap_mean[class_top_indices]
            
            # Create horizontal bar plot
            y_pos = np.arange(len(class_top_features))
            bars = ax.barh(y_pos, class_top_values, 
                          color=plt.cm.viridis(class_idx / n_classes))
            ax.set_yticks(y_pos)
            ax.set_yticklabels(class_top_features, fontsize=font_size_small)
            ax.set_xlabel('Mean |SHAP Value|', fontsize=font_size_medium)
            ax.set_title(f'Class {class_idx} Top Features', fontsize=font_size_medium)
            ax.invert_yaxis()
        
        # Remove empty subplots
        for i in range(n_classes, len(axes)):
            fig.delaxes(axes[i])
        
        plt.suptitle('Class-Specific Feature Importance', fontsize=font_size_title)
        plt.tight_layout()
        plt.savefig(f"{output_dir}/shap_class_specific.png", dpi=300, bbox_inches='tight')
        plt.close()
    
    # 4. Pathogenic vs Non-Pathogenic Feature Comparison
    print("Creating pathogenic vs non-pathogenic comparison...")
    if isinstance(shap_values, list):
        plt.figure(figsize=(14, 8))
        
        # Calculate mean SHAP values for pathogenic and non-pathogenic classes
        pathogenic_shap_mean = np.zeros(len(selected_genes))
        non_pathogenic_shap_mean = np.zeros(len(selected_genes))
        
        pathogenic_count = 0
        non_pathogenic_count = 0
        
        for class_idx in range(len(shap_values)):
            class_mean = np.mean(np.abs(shap_values[class_idx]), axis=0)
            
            if class_idx in high_importance_classes:
                pathogenic_shap_mean += class_mean
                pathogenic_count += 1
            else:
                non_pathogenic_shap_mean += class_mean
                non_pathogenic_count += 1
        
        if pathogenic_count > 0:
            pathogenic_shap_mean /= pathogenic_count
        if non_pathogenic_count > 0:
            non_pathogenic_shap_mean /= non_pathogenic_count
        
        # Calculate difference (pathogenic - non-pathogenic)
        shap_difference = pathogenic_shap_mean - non_pathogenic_shap_mean
        
        # Get top differentiating features (using n_comparison parameter)
        diff_indices = np.argsort(np.abs(shap_difference))[-n_comparison:][::-1]
        diff_features = [format_gene_name(selected_genes[i]) for i in diff_indices]
        diff_values = shap_difference[diff_indices]
        
        # Create diverging bar plot
        colors = ['red' if x > 0 else 'blue' for x in diff_values]
        y_pos = np.arange(len(diff_features))
        
        plt.barh(y_pos, diff_values, color=colors, alpha=0.7)
        plt.yticks(y_pos, diff_features, fontsize=font_size_small)
        plt.xlabel('SHAP Difference (Pathogenic - Non-Pathogenic)', fontsize=font_size_medium)
        plt.title('Features Distinguishing Pathogenic from Non-Pathogenic Classes', fontsize=font_size_large)
        plt.axvline(x=0, color='black', linestyle='--', alpha=0.5)
        
        # Add legend
        import matplotlib.patches as mpatches
        red_patch = mpatches.Patch(color='red', alpha=0.7, label='More important for pathogenic')
        blue_patch = mpatches.Patch(color='blue', alpha=0.7, label='More important for non-pathogenic')
        plt.legend(handles=[red_patch, blue_patch], fontsize=font_size_medium)
        
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(f"{output_dir}/shap_pathogenic_comparison.png", dpi=300, bbox_inches='tight')
        plt.close()
    
    # 5. Feature Interaction Heatmap (if we have interaction data)
    print("Creating feature interaction heatmap...")
    if isinstance(shap_values, list) and len(selected_genes) <= max(50, n_interactions * 2):
        try:
            # Calculate interaction values for top features only (using n_interactions parameter)
            interaction_top_indices = np.argsort(feature_importance)[-n_interactions:]
            X_shap_small = X_shap[:, interaction_top_indices]
            
            # Sample even smaller for interaction calculation
            if len(X_shap_small) > 30:
                interact_indices = np.random.choice(len(X_shap_small), size=30, replace=False)
                X_interact = X_shap_small[interact_indices]
            else:
                X_interact = X_shap_small
            
            interaction_values = explainer.shap_interaction_values(X_interact)
            
            if isinstance(interaction_values, list):
                # Sum interactions across pathogenic classes
                combined_interactions = np.zeros((len(interaction_top_indices), len(interaction_top_indices)))
                for class_idx in high_importance_classes:
                    if class_idx < len(interaction_values):
                        combined_interactions += np.mean(np.abs(interaction_values[class_idx]), axis=0)
                
                # Create heatmap
                plt.figure(figsize=(12, 10))
                interaction_names = [format_gene_name(selected_genes[i]) for i in interaction_top_indices]
                
                sns.heatmap(combined_interactions, 
                           xticklabels=interaction_names,
                           yticklabels=interaction_names,
                           cmap='viridis',
                           annot=False,
                           square=True)
                plt.title(f'Feature Interaction Strength (Top {n_interactions} Features)', fontsize=font_size_large)
                plt.xticks(rotation=45, ha='right', fontsize=font_size_small)
                plt.yticks(rotation=0, fontsize=font_size_small)
                plt.tight_layout()
                plt.savefig(f"{output_dir}/shap_interactions.png", dpi=300, bbox_inches='tight')
                plt.close()
        
        except Exception as e:
            print(f"Could not create interaction plot: {e}")
    
    # 6. Create a combined summary report
    print("Creating summary statistics...")
    
    # Calculate summary statistics for top features
    summary_stats = []
    
    if isinstance(shap_values, list):
        for i, gene in enumerate(selected_genes[:top_n_features]):
            stats = {
                'Gene': gene,
                'Short_Name': format_gene_name(gene),
                'Overall_Importance': feature_importance[i] if i < len(feature_importance) else 0,
                'Pathogenic_Importance': pathogenic_shap_mean[i] if i < len(pathogenic_shap_mean) else 0,
                'Non_Pathogenic_Importance': non_pathogenic_shap_mean[i] if i < len(non_pathogenic_shap_mean) else 0,
                'Difference_Score': shap_difference[i] if i < len(shap_difference) else 0
            }
            
            # Add class-specific importance
            for class_idx in range(len(shap_values)):
                class_importance = np.mean(np.abs(shap_values[class_idx][:, i])) if i < shap_values[class_idx].shape[1] else 0
                stats[f'Class_{class_idx}_Importance'] = class_importance
            
            summary_stats.append(stats)
    
    # Save summary statistics
    summary_df = pd.DataFrame(summary_stats)
    summary_df.to_csv(f"{output_dir}/shap_feature_summary.csv", index=False)
    
    # Reset matplotlib settings to default
    plt.rcParams.update(plt.rcParamsDefault)
    
    print(f"SHAP analysis complete! Results saved to {output_dir}/")
    print(f"Generated plots:")
    print(f"  - shap_summary_bar.png: Pathogenic class feature importance")
    print(f"  - shap_beeswarm.png: Detailed SHAP value distribution")
    print(f"  - shap_class_specific.png: Class-specific importance")
    print(f"  - shap_pathogenic_comparison.png: Pathogenic vs non-pathogenic")
    print(f"  - shap_interactions.png: Feature interactions (if applicable)")
    print(f"  - shap_feature_summary.csv: Detailed statistics")
    print(f"Note: Primary ranking based on pathogenic classes {high_importance_classes} only")
    
    return summary_df

def get_short_gene_name(gene_string):
    """
    Extract a short displayable name from the full gene string
    (Same function as in your original code)
    """
    # Check if it's a group
    if gene_string.startswith('group_'):
        return gene_string.split('-')[0]
    
    # Check for multiple genes separated by ~~~
    if '~~~' in gene_string:
        parts = gene_string.split('~~~')
        # Take the first part
        gene_string = parts[0]
    
    # Check for species identifier pattern (sp-XXXXX-GENE_SPECIES)
    if gene_string.startswith('sp-'):
        parts = gene_string.split('-')
        if len(parts) >= 3:
            # Extract just the gene name part without species
            gene_part = parts[2].split('_')[0]
            return gene_part
        
    # If no specific pattern matched, return the first 10 characters
    return gene_string[:10]

# Example usage with your existing pipeline:
"""
# After running your feature selection pipeline, you can call:

output_dir = './May/ntm_pathogenicity_analysis'
os.makedirs(output_dir, exist_ok=True)
final_df = pd.read_csv(f"{output_dir}/selected_genes_dataset.csv", index_col=0)
shap_output_dir = os.path.join(output_dir, "final_shap_analysis")
os.makedirs(shap_output_dir, exist_ok=True)
selected_genes = [col for col in final_df.columns if col != 'Pathogenicity_scale']

# With full control over number of features in each plot
summary_stats = create_comprehensive_shap_plots(
    df_pa=final_df,
    selected_genes=selected_genes,
    target_column="Pathogenicity_scale",
    output_dir=shap_output_dir,
    top_n_features=30,      # Main summary and beeswarm plots
    high_importance_threshold=4,
    max_samples=100,
    use_simple_names=True,
    font_size_small=10,
    font_size_medium=12,
    font_size_large=14,
    font_size_title=16,
    n_class_specific=20,    # Features per class in class-specific plots
    n_comparison=25,        # Features in pathogenic vs non-pathogenic plot
    n_interactions=15       # Features in interaction heatmap
)

print("\\nTop 10 most important features by SHAP analysis:")
print(summary_stats.head(10)[['Short_Name', 'Overall_Importance', 'Difference_Score']])
"""

'\n# After running your feature selection pipeline, you can call:\n\noutput_dir = \'./May/ntm_pathogenicity_analysis\'\nos.makedirs(output_dir, exist_ok=True)\nfinal_df = pd.read_csv(f"{output_dir}/selected_genes_dataset.csv", index_col=0)\nshap_output_dir = os.path.join(output_dir, "final_shap_analysis")\nos.makedirs(shap_output_dir, exist_ok=True)\nselected_genes = [col for col in final_df.columns if col != \'Pathogenicity_scale\']\n\n# With full control over number of features in each plot\nsummary_stats = create_comprehensive_shap_plots(\n    df_pa=final_df,\n    selected_genes=selected_genes,\n    target_column="Pathogenicity_scale",\n    output_dir=shap_output_dir,\n    top_n_features=30,      # Main summary and beeswarm plots\n    high_importance_threshold=4,\n    max_samples=100,\n    use_simple_names=True,\n    font_size_small=10,\n    font_size_medium=12,\n    font_size_large=14,\n    font_size_title=16,\n    n_class_specific=20,    # Features per class in class-specific plot

In [94]:
is_binary=True

In [95]:
final_df = pd.read_csv(f"{output_dir}/selected_genes_dataset.csv", index_col=0)
shap_output_dir = os.path.join(output_dir, "final_shap_analysis")
os.makedirs(shap_output_dir, exist_ok=True)
selected_genes = [col for col in final_df.columns if col != 'Oxygen2']

# Example for comprehensive analysis with many features
summary_stats_detailed = create_comprehensive_shap_plots(
    df_pa=final_df,
    selected_genes=selected_genes,
    target_column=target_column,
    output_dir=shap_output_dir + "_detailed",
    top_n_features=20,      # More main features
    high_importance_threshold=4,
    max_samples=100,
    use_simple_names=True, # Detailed names
    font_size_small=8,      # Smaller fonts to fit more
    font_size_medium=10,
    font_size_large=12,
    font_size_title=14,
    n_class_specific=25,    # More per class
    n_comparison=30,        # More comparison features
    n_interactions=25       # Larger interaction matrix
)

Creating comprehensive SHAP plots for top 20 features...
Training XGBoost model for SHAP analysis...
Calculating SHAP values...
Creating SHAP summary bar plot...


/projects/clbd1748/software/anaconda/envs/hpc-light-py310/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [16:03:04] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1748293041487/work/src/context.cc:203: XGBoost is not compiled with CUDA support.
  bst.update(dtrain, iteration=i, fobj=obj)


Creating SHAP beeswarm plot...
Creating class-specific SHAP plots...
Creating pathogenic vs non-pathogenic comparison...
Creating feature interaction heatmap...
Creating summary statistics...
SHAP analysis complete! Results saved to /scratch/alpine/clbd1748/Oxygen/final_shap_analysis_detailed/
Generated plots:
  - shap_summary_bar.png: Pathogenic class feature importance
  - shap_beeswarm.png: Detailed SHAP value distribution
  - shap_class_specific.png: Class-specific importance
  - shap_pathogenic_comparison.png: Pathogenic vs non-pathogenic
  - shap_interactions.png: Feature interactions (if applicable)
  - shap_feature_summary.csv: Detailed statistics
Note: Primary ranking based on pathogenic classes [] only


In [96]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import os
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb

def get_short_gene_name(gene_string):
    """
    Extract a short displayable name from the full gene string
    (Original sophisticated logic)
    """
    # Check if it's a group
    if gene_string.startswith('group_'):
        return gene_string.split('-')[0]
    
    # Check for multiple genes separated by ~~~
    if '~~~' in gene_string:
        parts = gene_string.split('~~~')
        # Take the first part
        gene_string = parts[0]
    
    # Check for species identifier pattern (sp-XXXXX-GENE_SPECIES)
    if gene_string.startswith('sp-'):
        parts = gene_string.split('-')
        if len(parts) >= 3:
            # Extract just the gene name part without species
            gene_part = parts[2].split('_')[0]
            return gene_part
        
    # If no specific pattern matched, return the first 10 characters
    return gene_string[:10]

def format_gene_names(gene_list, use_simple_names=True, max_length=18):
    """
    Format a list of gene names consistently
    
    Parameters:
    -----------
    gene_list : list
        List of gene names to format
    use_simple_names : bool
        If True, cut gene names at first '~~~'. If False, use sophisticated logic
    max_length : int
        Maximum length for displayed names (will add "..." if longer)
        
    Returns:
    --------
    list : Formatted gene names
    """
    formatted_names = []
    
    for gene in gene_list:
        if use_simple_names:
            # Cut at first ~~~
            formatted = gene.split('~~~')[0]
        else:
            # Use sophisticated logic
            formatted = get_short_gene_name(gene)
        
        # Truncate if too long
        if len(formatted) > max_length:
            formatted = formatted[:max_length-3] + "..."
            
        formatted_names.append(formatted)
    
    return formatted_names

def create_conference_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    figure_type="transition_beeswarm",
    from_class=0,
    to_class=1,
    dpi=300,
    style="publication",
    use_simple_names=True,
    max_gene_name_length=18,
    font_size_small=8,
    font_size_medium=10,
    font_size_large=12,
    font_size_title=14
):
    """
    Create publication-quality figures for conference presentation using PROPER CV predictions
    
    Parameters:
    -----------
    model_dir : str
        Directory containing trained models
    output_dir : str
        Directory to save figures
    figure_type : str
        Type of figure to create:
        - 'transition_beeswarm': SHAP beeswarm for class transition
        - 'overall_beeswarm': SHAP beeswarm for overall pathogenicity
        - 'pathway_heatmap': Gene cluster importance heatmap
        - 'prediction_confidence': Model confidence across classes
        - 'feature_importance': Top genes driving pathogenicity
    from_class : int
        Source class for transition analysis
    to_class : int
        Target class for transition analysis
    dpi : int
        Resolution for saved figures
    style : str
        Figure style ('publication', 'poster', 'presentation')
    use_simple_names : bool
        If True, cut gene names at first '~~~'. If False, use sophisticated logic
    max_gene_name_length : int
        Maximum length for gene names in plots
    font_size_small : int
        Font size for small text (tick labels)
    font_size_medium : int
        Font size for medium text (axis labels)
    font_size_large : int
        Font size for large text (plot titles)
    font_size_title : int
        Font size for main titles
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set matplotlib style based on figure purpose
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        default_font_size = 12
        default_title_size = 14
        default_label_size = 10
        figure_size = (10, 8)
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        default_font_size = 16
        default_title_size = 20
        default_label_size = 14
        figure_size = (12, 10)
    else:  # presentation
        plt.style.use('seaborn-v0_8-darkgrid')
        default_font_size = 14
        default_title_size = 18
        default_label_size = 12
        figure_size = (11, 9)
    
    # Use custom font sizes if provided, otherwise use style defaults
    final_font_size = font_size_medium if font_size_medium != 10 else default_font_size
    final_title_size = font_size_title if font_size_title != 14 else default_title_size
    final_label_size = font_size_medium if font_size_medium != 10 else default_label_size
    
    plt.rcParams.update({
        'font.size': final_font_size,
        'axes.titlesize': font_size_large,
        'axes.labelsize': final_label_size,
        'xtick.labelsize': font_size_small,
        'ytick.labelsize': font_size_small,
        'legend.fontsize': final_font_size,
        'figure.titlesize': final_title_size
    })
    
    # Load models and data
    print("Loading original models, data, and CV results...")
    
    # Load the ORIGINAL CV results (this is key!)
    results_path = os.path.join(model_dir, "results.pkl")
    with open(results_path, 'rb') as f:
        original_results = pickle.load(f)
    
    # Load the dataset
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load trained models (for SHAP and feature importance, NOT for evaluation)
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load the ORIGINAL explainers
    with open(os.path.join(model_dir, "shap_explainers.pkl"), 'rb') as f:
        explainers = pickle.load(f)
    
    # Get data from original results
    feature_names = original_results['feature_names']
    class_names = original_results['class_names']
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    
    # Convert sparse to dense if needed
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use ORIGINAL XGBoost model and explainer for SHAP/importance (NOT for evaluation)
    model = models['xgb']
    explainer = explainers['xgb']
    
    # Get the PROPER CV predictions (this is the key fix!)
    y_pred_cv = original_results['cv_results']['xgb']['predictions']
    
    print(f"Using LOOCV predictions with accuracy: {original_results['cv_results']['xgb']['accuracy']:.4f}")
    
    if figure_type == "transition_beeswarm":
        print(f"Creating transition beeswarm plot for classes {from_class} → {to_class}...")
        
        # Filter data for the specific transition
        transition_mask = (y == from_class) | (y == to_class)
        X_transition = X_array[transition_mask]
        y_transition = y[transition_mask]
        
        # Convert to binary classification problem
        y_binary = (y_transition == to_class).astype(int)
        
        # Train a binary classifier for this transition (this is still needed)
        transition_model = xgb.XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            tree_method='hist',
            device='cpu',  # Use CPU for compatibility
            objective='binary:logistic',
            random_state=42
        )
        
        transition_model.fit(X_transition, y_binary)
        
        # Create SHAP explainer for transition
        transition_explainer = shap.TreeExplainer(transition_model)
        
        # Calculate SHAP values (use subset for speed)
        n_samples = min(100, len(X_transition))
        sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
        X_sample = X_transition[sample_indices]
        
        shap_values = transition_explainer.shap_values(X_sample)
        
        # Create the beeswarm plot
        plt.figure(figsize=figure_size)
        
        # Format feature names consistently
        short_feature_names = format_gene_names(
            feature_names, 
            use_simple_names=use_simple_names, 
            max_length=max_gene_name_length
        )
        
        shap.summary_plot(
            shap_values, 
            X_sample, 
            feature_names=short_feature_names,
            max_display=20,
            show=False
        )
        
        plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
                 fontsize=font_size_large, pad=20)
        plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=font_size_medium)
        plt.tight_layout()
        
        # Save figure
        filename = f"transition_beeswarm_{from_class}_to_{to_class}_corrected.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "overall_beeswarm":
        print("Creating overall pathogenicity beeswarm plot using ORIGINAL explainer...")
        
        # Use the ORIGINAL explainer (no need to create new one)
        # Use a representative sample
        n_samples = min(150, len(X_array))
        sample_indices = np.random.choice(len(X_array), n_samples, replace=False)
        X_sample = X_array[sample_indices]
        y_sample = y[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)

        # Process SHAP values
        # For multi-class, focus on high pathogenicity classes
        if isinstance(shap_values, list):
            if scoring_strategy == "high_class_emphasis":
                target_classes = high_importance_classes
            else:
                target_classes = list(range(len(shap_values)))
            
            combined_shap = np.zeros_like(shap_values[0])
            for class_idx in target_classes:
                if class_idx < len(shap_values):
                    combined_shap += shap_values[class_idx]
            shap_values_plot = combined_shap
        else:
            shap_values_plot = shap_values
        
        # Format feature names consistently
        short_feature_names = format_gene_names(
            feature_names, 
            use_simple_names=use_simple_names, 
            max_length=max_gene_name_length
        )
        
        shap.summary_plot(
            shap_values_plot,
            X_sample,
            feature_names=short_feature_names,
            max_display=25,
            show=False
        )
        
        plt.title("Gene Impact on NTM Pathogenicity\n(Combined effect on pathogenic classes)", 
                 fontsize=font_size_large, pad=20)
        plt.xlabel("SHAP Value (Impact on Pathogenicity)", fontsize=font_size_medium)
        plt.tight_layout()
        
        filename = "overall_pathogenicity_beeswarm_corrected.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "feature_importance":
        print("Creating feature importance plot using ORIGINAL model...")
        
        # Get feature importances from the ORIGINAL model
        importances = model.feature_importances_
        
        # Get top 20 features
        top_indices = np.argsort(importances)[-20:]
        top_features = [feature_names[i] for i in top_indices]
        top_importances = importances[top_indices]
        
        # Format feature names consistently
        short_names = format_gene_names(
            top_features, 
            use_simple_names=use_simple_names, 
            max_length=max_gene_name_length
        )
        
        # Create horizontal bar plot
        plt.figure(figsize=figure_size)
        bars = plt.barh(range(len(top_features)), top_importances, 
                       color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
        
        plt.yticks(range(len(top_features)), short_names, fontsize=font_size_small)
        plt.xlabel("Feature Importance", fontsize=font_size_medium)
        plt.title("Top Genes Driving NTM Pathogenicity Prediction\n(From Original Model)", 
                 fontsize=font_size_large, pad=20)
        
        # Add value labels on bars
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                    f'{width:.3f}', ha='left', va='center', fontsize=font_size_small)
        
        plt.tight_layout()
        
        filename = "feature_importance_top20_corrected.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "prediction_confidence":
        print("Creating prediction confidence plot using ORIGINAL CV results...")
        
        # Use the CV probabilities if available
        if 'all_probabilities' in original_results and 'xgb' in original_results['all_probabilities']:
            y_prob_cv = original_results['all_probabilities']['xgb']
            confidence = np.max(y_prob_cv, axis=1)
        else:
            # Fallback: calculate from final model (but this won't be as accurate)
            print("Warning: Using final model for probabilities - not ideal!")
            y_prob = model.predict_proba(X_array)
            confidence = np.max(y_prob, axis=1)
        
        # Create confidence vs class plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(figure_size[0]*1.5, figure_size[1]))
        
        # Plot 1: Confidence distribution by predicted class
        for class_idx in np.unique(y_pred_cv):
            class_confidence = confidence[y_pred_cv == class_idx]
            ax1.hist(class_confidence, alpha=0.6, label=f'{class_names[class_idx]}', 
                    bins=20, density=True)
        
        ax1.set_xlabel("Prediction Confidence", fontsize=font_size_medium)
        ax1.set_ylabel("Density", fontsize=font_size_medium)
        ax1.set_title("Model Confidence by Predicted Class", fontsize=font_size_large)
        ax1.legend(fontsize=font_size_small)
        ax1.grid(True, alpha=0.3)
        ax1.tick_params(labelsize=font_size_small)
        
        # Plot 2: Confusion matrix with PROPER CV predictions
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y, y_pred_cv)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=[cn.replace('Class ', '') for cn in class_names],
                   yticklabels=[cn.replace('Class ', '') for cn in class_names],
                   ax=ax2)
        
        ax2.set_xlabel("Predicted Class", fontsize=font_size_medium)
        ax2.set_ylabel("True Class", fontsize=font_size_medium)
        ax2.set_title("LOOCV Prediction Accuracy", fontsize=font_size_large)
        ax2.tick_params(labelsize=font_size_small)
        
        plt.suptitle("NTM Pathogenicity Prediction Performance (LOOCV)", 
                    fontsize=font_size_title, y=1.02)
        plt.tight_layout()
        
        filename = "prediction_confidence_analysis_corrected.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "pathway_heatmap":
        print("Creating gene cluster/pathway heatmap...")
        
        # Load or create gene clusters
        clusters_dir = os.path.join(model_dir, "gene_clusters")
        cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
        
        if os.path.exists(cluster_file):
            with open(cluster_file, 'rb') as f:
                cluster_results = pickle.load(f)
        else:
            # Create clusters if they don't exist
            from analysis_functions import analyze_gene_clusters
            cluster_results = analyze_gene_clusters(model_dir, n_clusters=10)
        
        if cluster_results and 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Create heatmap data
            if isinstance(list(importance_data.keys())[0], int):  # Multi-class case
                n_classes = len(importance_data)
                n_clusters = len(importance_data[0])
                
                heatmap_data = np.zeros((n_clusters, n_classes))
                
                for class_idx in range(n_classes):
                    for cluster_idx in range(n_clusters):
                        if cluster_idx in importance_data[class_idx]:
                            heatmap_data[cluster_idx, class_idx] = importance_data[class_idx][cluster_idx]['mean_abs_shap']
                
                # Create the heatmap
                plt.figure(figsize=figure_size)
                
                sns.heatmap(heatmap_data, 
                           annot=True, 
                           fmt='.3f',
                           cmap='YlOrRd',
                           xticklabels=[cn.replace('Class ', '') for cn in class_names],
                           yticklabels=[f'Cluster {i}' for i in range(n_clusters)],
                           cbar_kws={'label': 'Mean |SHAP| Value'})
                
                plt.title("Gene Cluster Importance Across Pathogenicity Classes", 
                         fontsize=font_size_large, pad=20)
                plt.xlabel("Pathogenicity Class", fontsize=font_size_medium)
                plt.ylabel("Gene Cluster", fontsize=font_size_medium)
                plt.xticks(fontsize=font_size_small)
                plt.yticks(fontsize=font_size_small)
                plt.tight_layout()
                
                filename = "gene_cluster_importance_heatmap.png"
                plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
                plt.close()
        else:
            print("Warning: Gene cluster data not available. Creating feature importance instead.")
            # Fall back to feature importance
            return create_conference_figure(model_dir, output_dir, "feature_importance", 
                                          from_class, to_class, dpi, style, 
                                          use_simple_names, max_gene_name_length,
                                          font_size_small, font_size_medium, 
                                          font_size_large, font_size_title)
    
    print(f"Conference figure saved to {output_dir}/{filename}")
    return os.path.join(output_dir, filename)

def create_multi_panel_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    dpi=300,
    style="publication",
    use_simple_names=True,
    max_gene_name_length=12,
    font_size_small=8,
    font_size_medium=10,
    font_size_large=12,
    font_size_title=14
):
    """
    Create a comprehensive multi-panel figure using PROPER CV predictions
    """
    print("Creating comprehensive multi-panel conference figure using PROPER CV predictions...")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set style defaults
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        default_font_size = 10
        default_title_size = 12
        default_label_size = 9
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        default_font_size = 14
        default_title_size = 16
        default_label_size = 12
    
    # Use custom font sizes if provided, otherwise use style defaults
    final_font_size = font_size_medium if font_size_medium != 10 else default_font_size
    final_title_size = font_size_title if font_size_title != 14 else default_title_size
    final_label_size = font_size_medium if font_size_medium != 10 else default_label_size
    
    plt.rcParams.update({
        'font.size': final_font_size,
        'axes.titlesize': font_size_large,
        'axes.labelsize': final_label_size,
        'xtick.labelsize': font_size_small,
        'ytick.labelsize': font_size_small,
        'legend.fontsize': font_size_small,
    })
    
    # Load the ORIGINAL CV results (this is key!)
    results_path = os.path.join(model_dir, "results.pkl")
    with open(results_path, 'rb') as f:
        original_results = pickle.load(f)
    
    # Load data
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load ORIGINAL trained models (for feature importance only)
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load the ORIGINAL explainers
    with open(os.path.join(model_dir, "shap_explainers.pkl"), 'rb') as f:
        explainers = pickle.load(f)
    
    # Get data from original results
    feature_names = original_results['feature_names']
    class_names = original_results['class_names']
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use ORIGINAL model and explainer
    model = models['xgb']
    explainer = explainers['xgb']
    
    # Get the PROPER CV predictions
    y_pred_cv = original_results['cv_results']['xgb']['predictions']
    cv_results = original_results['cv_results']
    
    # Create figure with subplots (2x3 layout)
    fig = plt.figure(figsize=(18, 12))
    
    # Panel A: PROPER performance using CV predictions
    ax1 = plt.subplot(2, 3, 1)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y, y_pred_cv)  # Use CV predictions!
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=[cn.replace('Class ', '') for cn in class_names],
               yticklabels=[cn.replace('Class ', '') for cn in class_names],
               ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_title("A. LOOCV Performance", fontweight='bold', fontsize=font_size_large)
    ax1.set_xlabel("Predicted", fontsize=font_size_medium)
    ax1.set_ylabel("Actual", fontsize=font_size_medium)
    ax1.tick_params(labelsize=font_size_small)
    
    # Panel B: Feature importance from ORIGINAL model
    ax2 = plt.subplot(2, 3, 2)
    importances = model.feature_importances_
    top_indices = np.argsort(importances)[-15:]
    top_features = [feature_names[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    # Format names consistently
    short_names = format_gene_names(
        top_features, 
        use_simple_names=use_simple_names, 
        max_length=max_gene_name_length
    )
    
    bars = ax2.barh(range(len(top_features)), top_importances,
                   color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
    ax2.set_yticks(range(len(top_features)))
    ax2.set_yticklabels(short_names, fontsize=font_size_small)
    ax2.set_xlabel("Importance", fontsize=font_size_medium)
    ax2.set_title("B. Top Genes", fontweight='bold', fontsize=font_size_large)
    ax2.tick_params(labelsize=font_size_small)
    
    # Panel C: Class distribution
    ax3 = plt.subplot(2, 3, 3)
    class_counts = np.bincount(y, minlength=len(class_names))
    bars = ax3.bar(range(len(class_names)), class_counts, 
                  color=plt.cm.RdYlBu_r(np.linspace(0, 1, len(class_names))))
    ax3.set_xticks(range(len(class_names)))
    ax3.set_xticklabels([cn.replace('Class ', '') for cn in class_names], 
                        rotation=45, fontsize=font_size_small)
    ax3.set_ylabel("Number of Strains", fontsize=font_size_medium)
    ax3.set_title("C. Dataset Distribution", fontweight='bold', fontsize=font_size_large)
    ax3.tick_params(labelsize=font_size_small)
    
    # Panel D: Prediction confidence using CV probabilities
    ax4 = plt.subplot(2, 3, 4)
    
    if 'all_probabilities' in original_results and 'xgb' in original_results['all_probabilities']:
        y_prob_cv = original_results['all_probabilities']['xgb']
        confidence = np.max(y_prob_cv, axis=1)
    else:
        # Fallback to model predictions (not ideal)
        y_prob = model.predict_proba(X_array)
        confidence = np.max(y_prob, axis=1)
    
    ax4.hist(confidence, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax4.axvline(np.mean(confidence), color='red', linestyle='--', 
               label=f'Mean: {np.mean(confidence):.2f}')
    ax4.set_xlabel("Prediction Confidence", fontsize=font_size_medium)
    ax4.set_ylabel("Frequency", fontsize=font_size_medium)
    ax4.set_title("D. LOOCV Confidence", fontweight='bold', fontsize=font_size_large)
    ax4.legend(fontsize=font_size_small)
    ax4.tick_params(labelsize=font_size_small)
    
    # Panel E: Model comparison from ORIGINAL CV results
    ax5 = plt.subplot(2, 3, 5)
    
    model_names = list(cv_results.keys())
    accuracies = [cv_results[model]['accuracy'] for model in model_names]
    f1_scores = [cv_results[model]['f1_weighted'] for model in model_names]
    
    x = np.arange(len(model_names))
    width = 0.35
    
    ax5.bar(x - width/2, accuracies, width, label='Accuracy', alpha=0.8)
    ax5.bar(x + width/2, f1_scores, width, label='F1-Score', alpha=0.8)
    
    ax5.set_xlabel('Model', fontsize=font_size_medium)
    ax5.set_ylabel('Score', fontsize=font_size_medium)
    ax5.set_title('E. Model Comparison', fontweight='bold', fontsize=font_size_large)
    ax5.set_xticks(x)
    ax5.set_xticklabels([m.upper() for m in model_names], fontsize=font_size_small)
    ax5.legend(fontsize=font_size_small)
    ax5.set_ylim(0, 1)
    ax5.tick_params(labelsize=font_size_small)
    
    # Panel F: Gene cluster importance or top genes from ORIGINAL analysis
    ax6 = plt.subplot(2, 3, 6)
    
    # Load gene clusters if available
    clusters_dir = os.path.join(model_dir, "gene_clusters")
    cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
    
    if os.path.exists(cluster_file):
        with open(cluster_file, 'rb') as f:
            cluster_results = pickle.load(f)
        
        if 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Get importance for the most pathogenic class (class 6 or 7)
            target_class = 6 if 6 in importance_data else 7 if 7 in importance_data else max(importance_data.keys())
            
            cluster_ids = list(importance_data[target_class].keys())
            cluster_importances = [importance_data[target_class][cid]['mean_abs_shap'] 
                                 for cid in cluster_ids]
            
            bars = ax6.bar([f'C{cid}' for cid in cluster_ids], cluster_importances,
                          color=plt.cm.plasma(np.linspace(0, 1, len(cluster_ids))))
            ax6.set_xlabel('Gene Cluster', fontsize=font_size_medium)
            ax6.set_ylabel('Importance', fontsize=font_size_medium)
            ax6.set_title('F. Gene Cluster Impact', fontweight='bold', fontsize=font_size_large)
            ax6.tick_params(axis='x', rotation=45, labelsize=font_size_small)
            ax6.tick_params(axis='y', labelsize=font_size_small)
        else:
            # Fallback: show top feature importances in a pie chart
            top_5_indices = np.argsort(importances)[-5:]
            top_5_features = [feature_names[i] for i in top_5_indices]
            top_5_importances = importances[top_5_indices]
            
            # Format names for pie chart
            top_5_names = format_gene_names(
                top_5_features, 
                use_simple_names=use_simple_names, 
                max_length=8
            )
            
            ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
            ax6.set_title('F. Top 5 Genes', fontweight='bold', fontsize=font_size_large)
    else:
        # Fallback: show top feature importances as pie chart
        top_5_indices = np.argsort(importances)[-5:]
        top_5_features = [feature_names[i] for i in top_5_indices]
        top_5_importances = importances[top_5_indices]
        
        # Format names for pie chart
        top_5_names = format_gene_names(
            top_5_features, 
            use_simple_names=use_simple_names, 
            max_length=8
        )
        
        ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
        ax6.set_title('F. Top 5 Genes', fontweight='bold', fontsize=font_size_large)
    
    # Add main title
    fig.suptitle("Machine Learning-Based Prediction of NTM Pathogenicity from Genomic Data\n(Leave-One-Out Cross-Validation Results)", 
                fontsize=font_size_title, fontweight='bold', y=0.96)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    
    filename = "comprehensive_conference_figure_corrected.png"
    plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    # Create a separate SHAP transition plot (still needs new model for transition)
    print("Creating separate SHAP transition plot...")
    
    # Create transition data for SHAP (this still needs a new binary model)
    from_class, to_class = 2, 4
    transition_mask = (y == from_class) | (y == to_class)
    X_transition = X_array[transition_mask]
    y_transition = y[transition_mask]
    y_binary = (y_transition == to_class).astype(int)
    
    # Train transition model (this is still needed for binary classification)
    transition_model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=5,
        tree_method='hist', device='cpu', objective='binary:logistic',
        random_state=42
    )
    transition_model.fit(X_transition, y_binary)
    
    # SHAP analysis for transition
    transition_explainer = shap.TreeExplainer(transition_model)
    n_samples = min(80, len(X_transition))
    sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
    X_sample = X_transition[sample_indices]
    shap_values = transition_explainer.shap_values(X_sample)
    
    # Create standalone SHAP plot
    plt.figure(figsize=(10, 8))
    
    # Format feature names consistently
    short_feature_names = format_gene_names(
        feature_names, 
        use_simple_names=use_simple_names, 
        max_length=18
    )
    
    shap.summary_plot(shap_values, X_sample, feature_names=short_feature_names,
                     max_display=20, show=False)
    plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
             fontsize=font_size_title, pad=20)
    plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=font_size_medium)
    plt.tick_params(labelsize=font_size_small)
    plt.tight_layout()
    
    shap_filename = "transition_shap_standalone_corrected.png"
    plt.savefig(os.path.join(output_dir, shap_filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"Comprehensive conference figure saved to {output_dir}/{filename}")
    print(f"Standalone SHAP plot saved to {output_dir}/{shap_filename}")
    print(f"LOOCV Accuracy used: {original_results['cv_results']['xgb']['accuracy']:.4f}")
    return os.path.join(output_dir, filename)

def create_all_conference_figures(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    use_simple_names=True,
    style="publication",
    from_class=0,
    to_class=6,
    max_gene_name_length=18,
    font_size_small=8,
    font_size_medium=10,
    font_size_large=12,
    font_size_title=14
):
    """
    Create all types of conference figures with consistent naming and font control
    
    Parameters:
    -----------
    model_dir : str
        Directory containing trained models
    output_dir : str
        Directory to save figures
    use_simple_names : bool
        If True, cut gene names at first '~~~'. If False, use sophisticated logic
    style : str
        Figure style ('publication', 'poster', 'presentation')
    from_class : int
        Source class for transition analysis
    to_class : int
        Target class for transition analysis
    max_gene_name_length : int
        Maximum length for gene names in plots
    font_size_small : int
        Font size for small text (tick labels)
    font_size_medium : int
        Font size for medium text (axis labels)
    font_size_large : int
        Font size for large text (plot titles)
    font_size_title : int
        Font size for main titles
    """
    
    print("Creating all conference figures with consistent gene naming and font control...")
    
    figures_created = []
    
    # 1. Transition beeswarm (recommended for showing biological insight)
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="transition_beeswarm",
            from_class=from_class,
            to_class=to_class,
            style=style,
            use_simple_names=use_simple_names,
            max_gene_name_length=max_gene_name_length,
            font_size_small=font_size_small,
            font_size_medium=font_size_medium,
            font_size_large=font_size_large,
            font_size_title=font_size_title
        )
        figures_created.append(fig_path)
        print(f"✓ Created transition beeswarm plot ({from_class} → {to_class})")
    except Exception as e:
        print(f"✗ Error creating transition beeswarm: {e}")
    
    # 2. Overall beeswarm
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="overall_beeswarm",
            style=style,
            use_simple_names=use_simple_names,
            max_gene_name_length=max_gene_name_length,
            font_size_small=font_size_small,
            font_size_medium=font_size_medium,
            font_size_large=font_size_large,
            font_size_title=font_size_title
        )
        figures_created.append(fig_path)
        print("✓ Created overall beeswarm plot")
    except Exception as e:
        print(f"✗ Error creating overall beeswarm: {e}")
    
    # 3. Feature importance
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="feature_importance",
            style=style,
            use_simple_names=use_simple_names,
            max_gene_name_length=max_gene_name_length + 2,  # Slightly longer for importance plot
            font_size_small=font_size_small,
            font_size_medium=font_size_medium,
            font_size_large=font_size_large,
            font_size_title=font_size_title
        )
        figures_created.append(fig_path)
        print("✓ Created feature importance plot")
    except Exception as e:
        print(f"✗ Error creating feature importance: {e}")
    
    # 4. Prediction confidence
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="prediction_confidence",
            style=style,
            use_simple_names=use_simple_names,
            font_size_small=font_size_small,
            font_size_medium=font_size_medium,
            font_size_large=font_size_large,
            font_size_title=font_size_title
        )
        figures_created.append(fig_path)
        print("✓ Created prediction confidence plot")
    except Exception as e:
        print(f"✗ Error creating prediction confidence: {e}")
    
    # 5. Pathway heatmap (if available)
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="pathway_heatmap",
            style=style,
            use_simple_names=use_simple_names,
            font_size_small=font_size_small,
            font_size_medium=font_size_medium,
            font_size_large=font_size_large,
            font_size_title=font_size_title
        )
        figures_created.append(fig_path)
        print("✓ Created pathway heatmap")
    except Exception as e:
        print(f"✗ Error creating pathway heatmap: {e}")
    
    # 6. Comprehensive multi-panel figure
    try:
        fig_path = create_multi_panel_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            style=style,
            use_simple_names=use_simple_names,
            max_gene_name_length=max_gene_name_length - 6,  # Shorter for multi-panel
            font_size_small=font_size_small,
            font_size_medium=font_size_medium,
            font_size_large=font_size_large,
            font_size_title=font_size_title
        )
        figures_created.append(fig_path)
        print("✓ Created comprehensive multi-panel figure")
    except Exception as e:
        print(f"✗ Error creating multi-panel figure: {e}")
    
    print(f"\nSuccessfully created {len(figures_created)} conference figures!")
    print(f"All figures saved to: {output_dir}")
    print(f"Transition analysis: Class {from_class} → Class {to_class}")
    
    if use_simple_names:
        print("Gene names formatted using simple logic (cut at first '~~~')")
    else:
        print("Gene names formatted using sophisticated extraction logic")
    
    return figures_created
'''
if __name__ == "__main__":
    model_dir = "./May/ntm_pathogenicity_model"
    output_dir = "./conference_figures"
    
    # Example 1: Create publication figures with simple names and custom transition
    print("Creating publication figures with simple gene names (0 → 6 transition)...")
    create_all_conference_figures(
        model_dir=model_dir,
        output_dir=output_dir + "_simple_0to6",
        use_simple_names=True,
        style="publication",
        from_class=0,
        to_class=6,
        max_gene_name_length=18,
        font_size_small=10,
        font_size_medium=12,
        font_size_large=14,
        font_size_title=16
    )
    
    # Example 2: Create poster figures with sophisticated names and different transition
    print("\nCreating poster figures with sophisticated gene names (2 → 4 transition)...")
    create_all_conference_figures(
        model_dir=model_dir,
        output_dir=output_dir + "_poster_2to4",
        use_simple_names=False,
        style="poster",
        from_class=2,
        to_class=4,
        max_gene_name_length=20,
        font_size_small=14,
        font_size_medium=16,
        font_size_large=18,
        font_size_title=22
    )
    
    # Example 3: Create presentation figures with compact fonts
    print("\nCreating presentation figures with compact layout (4 → 6 transition)...")
    create_all_conference_figures(
        model_dir=model_dir,
        output_dir=output_dir + "_presentation_4to6",
        use_simple_names=True,
        style="presentation",
        from_class=4,
        to_class=6,
        max_gene_name_length=15,
        font_size_small=8,
        font_size_medium=10,
        font_size_large=12,
        font_size_title=14
    )
    
    print("\nAll conference figures created with customizable transitions, naming, and fonts!")
'''

'\nif __name__ == "__main__":\n    model_dir = "./May/ntm_pathogenicity_model"\n    output_dir = "./conference_figures"\n    \n    # Example 1: Create publication figures with simple names and custom transition\n    print("Creating publication figures with simple gene names (0 → 6 transition)...")\n    create_all_conference_figures(\n        model_dir=model_dir,\n        output_dir=output_dir + "_simple_0to6",\n        use_simple_names=True,\n        style="publication",\n        from_class=0,\n        to_class=6,\n        max_gene_name_length=18,\n        font_size_small=10,\n        font_size_medium=12,\n        font_size_large=14,\n        font_size_title=16\n    )\n    \n    # Example 2: Create poster figures with sophisticated names and different transition\n    print("\nCreating poster figures with sophisticated gene names (2 → 4 transition)...")\n    create_all_conference_figures(\n        model_dir=model_dir,\n        output_dir=output_dir + "_poster_2to4",\n        use_simple_

In [98]:
if __name__ == "__main__":
    model_dir = "./"
    output_dir = "./conference_figures"
    
    # Example 1: Create publication figures with simple names and custom transition
    print("Creating publication figures with simple gene names (0 → 6 transition)...")
    create_all_conference_figures(
        model_dir=model_dir,
        output_dir=output_dir + "_simple_0to6",
        use_simple_names=True,
        style="publication",
        from_class=0,
        to_class=6,
        max_gene_name_length=30,
        font_size_small=6,
        font_size_medium=8,
        font_size_large=10,
        font_size_title=16
    )

Creating publication figures with simple gene names (0 → 6 transition)...
Creating all conference figures with consistent gene naming and font control...
Loading original models, data, and CV results...
✗ Error creating transition beeswarm: [Errno 2] No such file or directory: './results.pkl'
Loading original models, data, and CV results...
✗ Error creating overall beeswarm: [Errno 2] No such file or directory: './results.pkl'
Loading original models, data, and CV results...
✗ Error creating feature importance: [Errno 2] No such file or directory: './results.pkl'
Loading original models, data, and CV results...
✗ Error creating prediction confidence: [Errno 2] No such file or directory: './results.pkl'
Loading original models, data, and CV results...
✗ Error creating pathway heatmap: [Errno 2] No such file or directory: './results.pkl'
Creating comprehensive multi-panel conference figure using PROPER CV predictions...
✗ Error creating multi-panel figure: [Errno 2] No such file or direc

In [33]:
dd = pd.read_csv('./May/ntm_pathogenicity_analysis/selected_important_features.csv')

In [34]:
dd.columns

Index(['Feature', 'Aggregated_Score', 'Iteration_1_Score', 'Iteration_2_Score',
       'Iteration_3_Score', 'Gene_Cluster', 'Interaction_Strength',
       'Top_Interaction_Partners'],
      dtype='object')

# Old Versions

## Conference Figure

### Even newer 

In [70]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import os
from sklearn.model_selection import train_test_split
import xgboost as xgb

def get_short_gene_name(gene_string):
    """
    Extract a short displayable name from the full gene string
    (Original sophisticated logic)
    """
    # Check if it's a group
    if gene_string.startswith('group_'):
        return gene_string.split('-')[0]
    
    # Check for multiple genes separated by ~~~
    if '~~~' in gene_string:
        parts = gene_string.split('~~~')
        # Take the first part
        gene_string = parts[0]
    
    # Check for species identifier pattern (sp-XXXXX-GENE_SPECIES)
    if gene_string.startswith('sp-'):
        parts = gene_string.split('-')
        if len(parts) >= 3:
            # Extract just the gene name part without species
            gene_part = parts[2].split('_')[0]
            return gene_part
        
    # If no specific pattern matched, return the first 10 characters
    return gene_string[:10]

def format_gene_names(gene_list, use_simple_names=True, max_length=18):
    """
    Format a list of gene names consistently
    
    Parameters:
    -----------
    gene_list : list
        List of gene names to format
    use_simple_names : bool
        If True, cut gene names at first '~~~'. If False, use sophisticated logic
    max_length : int
        Maximum length for displayed names (will add "..." if longer)
        
    Returns:
    --------
    list : Formatted gene names
    """
    formatted_names = []
    
    for gene in gene_list:
        if use_simple_names:
            # Cut at first ~~~
            formatted = gene.split('~~~')[0]
        else:
            # Use sophisticated logic
            formatted = get_short_gene_name(gene)
        
        # Truncate if too long
        if len(formatted) > max_length:
            formatted = formatted[:max_length-3] + "..."
            
        formatted_names.append(formatted)
    
    return formatted_names

def create_conference_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    figure_type="transition_beeswarm",
    from_class=2,
    to_class=4,
    dpi=300,
    style="publication",
    use_simple_names=True,
    max_gene_name_length=18
):
    """
    Create publication-quality figures for conference presentation
    
    Parameters:
    -----------
    model_dir : str
        Directory containing trained models
    output_dir : str
        Directory to save figures
    figure_type : str
        Type of figure to create:
        - 'transition_beeswarm': SHAP beeswarm for class transition
        - 'overall_beeswarm': SHAP beeswarm for overall pathogenicity
        - 'pathway_heatmap': Gene cluster importance heatmap
        - 'prediction_confidence': Model confidence across classes
        - 'feature_importance': Top genes driving pathogenicity
    from_class : int
        Source class for transition analysis
    to_class : int
        Target class for transition analysis
    dpi : int
        Resolution for saved figures
    style : str
        Figure style ('publication', 'poster', 'presentation')
    use_simple_names : bool
        If True, cut gene names at first '~~~'. If False, use sophisticated logic
    max_gene_name_length : int
        Maximum length for gene names in plots
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set matplotlib style based on figure purpose
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 12
        title_size = 14
        label_size = 10
        figure_size = (10, 8)
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 16
        title_size = 20
        label_size = 14
        figure_size = (12, 10)
    else:  # presentation
        plt.style.use('seaborn-v0_8-darkgrid')
        font_size = 14
        title_size = 18
        label_size = 12
        figure_size = (11, 9)
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size,
        'ytick.labelsize': label_size,
        'legend.fontsize': font_size,
        'figure.titlesize': title_size + 2
    })
    
    # Load models and data
    print("Loading models and data...")
    
    # Load the dataset
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load trained models
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load feature names and class info
    with open(os.path.join(model_dir, "feature_names.pkl"), 'rb') as f:
        feature_names = pickle.load(f)
    
    with open(os.path.join(model_dir, "class_info.pkl"), 'rb') as f:
        class_info = pickle.load(f)
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    class_names = class_info['class_names']
    
    # Convert sparse to dense if needed
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use XGBoost model for SHAP analysis
    model = models['xgb']
    
    if figure_type == "transition_beeswarm":
        print(f"Creating transition beeswarm plot for classes {from_class} → {to_class}...")
        
        # Filter data for the specific transition
        transition_mask = (y == from_class) | (y == to_class)
        X_transition = X_array[transition_mask]
        y_transition = y[transition_mask]
        
        # Convert to binary classification problem
        y_binary = (y_transition == to_class).astype(int)
        
        # Train a binary classifier for this transition
        transition_model = xgb.XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            tree_method='hist',
            device='cpu',  # Use CPU for compatibility
            objective='binary:logistic',
            random_state=42
        )
        
        transition_model.fit(X_transition, y_binary)
        
        # Create SHAP explainer
        explainer = shap.TreeExplainer(transition_model)
        
        # Calculate SHAP values (use subset for speed)
        n_samples = min(100, len(X_transition))
        sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
        X_sample = X_transition[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)
        
        # Create the beeswarm plot
        plt.figure(figsize=figure_size)
        
        # Format feature names consistently
        short_feature_names = format_gene_names(
            feature_names, 
            use_simple_names=use_simple_names, 
            max_length=max_gene_name_length
        )
        
        shap.summary_plot(
            shap_values, 
            X_sample, 
            feature_names=short_feature_names,
            max_display=20,
            show=False
        )
        
        plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=label_size)
        plt.tight_layout()
        
        # Save figure
        filename = f"transition_beeswarm_{from_class}_to_{to_class}.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "overall_beeswarm":
        print("Creating overall pathogenicity beeswarm plot...")
        
        # Create SHAP explainer
        explainer = shap.TreeExplainer(model)
        
        # Use a representative sample
        n_samples = min(150, len(X_array))
        sample_indices = np.random.choice(len(X_array), n_samples, replace=False)
        X_sample = X_array[sample_indices]
        y_sample = y[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)


        # Process SHAP values
        if isinstance(shap_values, list):
            if scoring_strategy == "high_class_emphasis":
                target_classes = high_importance_classes
            else:
                target_classes = list(range(len(shap_values)))
            
            combined_shap = np.zeros_like(shap_values[0])
            for class_idx in target_classes:
                if class_idx < len(shap_values):
                    combined_shap += shap_values[class_idx]
            shap_values_plot = combined_shap
        else:
            shap_values_plot = shap_values
        
        plt.figure(figsize=figure_size)
        
        # Format feature names consistently
        short_feature_names = format_gene_names(
            feature_names, 
            use_simple_names=use_simple_names, 
            max_length=max_gene_name_length
        )
        
        shap.summary_plot(
            shap_values_plot,
            X_sample,
            feature_names=short_feature_names,
            max_display=25,
            show=False
        )
        
        plt.title("Gene Impact on NTM Pathogenicity\n(Combined effect on pathogenic classes)", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Pathogenicity)", fontsize=label_size)
        plt.tight_layout()
        
        filename = "overall_pathogenicity_beeswarm.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "feature_importance":
        print("Creating feature importance plot...")
        
        # Get feature importances from the model
        importances = model.feature_importances_
        
        # Get top 20 features
        top_indices = np.argsort(importances)[-20:]
        top_features = [feature_names[i] for i in top_indices]
        top_importances = importances[top_indices]
        
        # Format feature names consistently
        short_names = format_gene_names(
            top_features, 
            use_simple_names=use_simple_names, 
            max_length=max_gene_name_length
        )
        
        # Create horizontal bar plot
        plt.figure(figsize=figure_size)
        bars = plt.barh(range(len(top_features)), top_importances, 
                       color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
        
        plt.yticks(range(len(top_features)), short_names)
        plt.xlabel("Feature Importance", fontsize=label_size)
        plt.title("Top Genes Driving NTM Pathogenicity Prediction", fontsize=title_size, pad=20)
        
        # Add value labels on bars
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                    f'{width:.3f}', ha='left', va='center', fontsize=font_size-2)
        
        plt.tight_layout()
        
        filename = "feature_importance_top20.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "prediction_confidence":
        print("Creating prediction confidence plot...")
        
        # Get predictions and probabilities for all samples
        y_pred = model.predict(X_array)
        y_prob = model.predict_proba(X_array)
        
        # Calculate confidence (max probability) for each prediction
        confidence = np.max(y_prob, axis=1)
        
        # Create confidence vs class plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(figure_size[0]*1.5, figure_size[1]))
        
        # Plot 1: Confidence distribution by predicted class
        for class_idx in np.unique(y_pred):
            class_confidence = confidence[y_pred == class_idx]
            ax1.hist(class_confidence, alpha=0.6, label=f'{class_names[class_idx]}', 
                    bins=20, density=True)
        
        ax1.set_xlabel("Prediction Confidence", fontsize=label_size)
        ax1.set_ylabel("Density", fontsize=label_size)
        ax1.set_title("Model Confidence by Predicted Class", fontsize=title_size)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Confusion matrix with confidence
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=[cn.replace('Class ', '') for cn in class_names],
                   yticklabels=[cn.replace('Class ', '') for cn in class_names],
                   ax=ax2)
        
        ax2.set_xlabel("Predicted Class", fontsize=label_size)
        ax2.set_ylabel("True Class", fontsize=label_size)
        ax2.set_title("Prediction Accuracy Matrix", fontsize=title_size)
        
        plt.suptitle("NTM Pathogenicity Prediction Performance", fontsize=title_size+2, y=1.02)
        plt.tight_layout()
        
        filename = "prediction_confidence_analysis.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "pathway_heatmap":
        print("Creating gene cluster/pathway heatmap...")
        
        # Load or create gene clusters
        clusters_dir = os.path.join(model_dir, "gene_clusters")
        cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
        
        if os.path.exists(cluster_file):
            with open(cluster_file, 'rb') as f:
                cluster_results = pickle.load(f)
        else:
            # Create clusters if they don't exist
            from analysis_functions import analyze_gene_clusters
            cluster_results = analyze_gene_clusters(model_dir, n_clusters=10)
        
        if cluster_results and 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Create heatmap data
            if isinstance(list(importance_data.keys())[0], int):  # Multi-class case
                n_classes = len(importance_data)
                n_clusters = len(importance_data[0])
                
                heatmap_data = np.zeros((n_clusters, n_classes))
                
                for class_idx in range(n_classes):
                    for cluster_idx in range(n_clusters):
                        if cluster_idx in importance_data[class_idx]:
                            heatmap_data[cluster_idx, class_idx] = importance_data[class_idx][cluster_idx]['mean_abs_shap']
                
                # Create the heatmap
                plt.figure(figsize=figure_size)
                
                sns.heatmap(heatmap_data, 
                           annot=True, 
                           fmt='.3f',
                           cmap='YlOrRd',
                           xticklabels=[cn.replace('Class ', '') for cn in class_names],
                           yticklabels=[f'Cluster {i}' for i in range(n_clusters)],
                           cbar_kws={'label': 'Mean |SHAP| Value'})
                
                plt.title("Gene Cluster Importance Across Pathogenicity Classes", 
                         fontsize=title_size, pad=20)
                plt.xlabel("Pathogenicity Class", fontsize=label_size)
                plt.ylabel("Gene Cluster", fontsize=label_size)
                plt.tight_layout()
                
                filename = "gene_cluster_importance_heatmap.png"
                plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
                plt.close()
        else:
            print("Warning: Gene cluster data not available. Creating feature importance instead.")
            # Fall back to feature importance
            return create_conference_figure(model_dir, output_dir, "feature_importance", 
                                          from_class, to_class, dpi, style, 
                                          use_simple_names, max_gene_name_length)
    
    print(f"Conference figure saved to {output_dir}/{filename}")
    return os.path.join(output_dir, filename)

def create_multi_panel_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    dpi=300,
    style="publication",
    use_simple_names=True,
    max_gene_name_length=12  # Shorter for multi-panel to fit better
):
    """
    Create a comprehensive multi-panel figure for conference presentation
    """
    print("Creating comprehensive multi-panel conference figure...")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set style
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 10
        title_size = 12
        label_size = 9
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 14
        title_size = 16
        label_size = 12
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size-1,
        'ytick.labelsize': label_size-1,
        'legend.fontsize': font_size-1,
    })
    
    # Load data
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    with open(os.path.join(model_dir, "feature_names.pkl"), 'rb') as f:
        feature_names = pickle.load(f)
    
    with open(os.path.join(model_dir, "class_info.pkl"), 'rb') as f:
        class_info = pickle.load(f)
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    class_names = class_info['class_names']
    
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    model = models['xgb']
    
    # Create figure with subplots (2x3 layout)
    fig = plt.figure(figsize=(18, 12))
    
    # Panel A: Overall performance
    ax1 = plt.subplot(2, 3, 1)
    y_pred = model.predict(X_array)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=[cn.replace('Class ', '') for cn in class_names],
               yticklabels=[cn.replace('Class ', '') for cn in class_names],
               ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_title("A. Model Performance", fontweight='bold')
    ax1.set_xlabel("Predicted")
    ax1.set_ylabel("Actual")
    
    # Panel B: Feature importance
    ax2 = plt.subplot(2, 3, 2)
    importances = model.feature_importances_
    top_indices = np.argsort(importances)[-15:]
    top_features = [feature_names[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    # Format names consistently
    short_names = format_gene_names(
        top_features, 
        use_simple_names=use_simple_names, 
        max_length=max_gene_name_length
    )
    
    bars = ax2.barh(range(len(top_features)), top_importances,
                   color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
    ax2.set_yticks(range(len(top_features)))
    ax2.set_yticklabels(short_names, fontsize=label_size-1)
    ax2.set_xlabel("Importance")
    ax2.set_title("B. Top Genes", fontweight='bold')
    
    # Panel C: Class distribution
    ax3 = plt.subplot(2, 3, 3)
    class_counts = np.bincount(y, minlength=len(class_names))
    bars = ax3.bar(range(len(class_names)), class_counts, 
                  color=plt.cm.RdYlBu_r(np.linspace(0, 1, len(class_names))))
    ax3.set_xticks(range(len(class_names)))
    ax3.set_xticklabels([cn.replace('Class ', '') for cn in class_names], rotation=45)
    ax3.set_ylabel("Number of Strains")
    ax3.set_title("C. Dataset Distribution", fontweight='bold')
    
    # Panel D: Prediction confidence
    ax4 = plt.subplot(2, 3, 4)
    y_prob = model.predict_proba(X_array)
    confidence = np.max(y_prob, axis=1)
    
    ax4.hist(confidence, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax4.axvline(np.mean(confidence), color='red', linestyle='--', 
               label=f'Mean: {np.mean(confidence):.2f}')
    ax4.set_xlabel("Prediction Confidence")
    ax4.set_ylabel("Frequency")
    ax4.set_title("D. Model Confidence", fontweight='bold')
    ax4.legend()
    
    # Panel E: Model comparison
    ax5 = plt.subplot(2, 3, 5)
    
    # Load results from the model training
    results_path = os.path.join(model_dir, "results.pkl")
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            results = pickle.load(f)
        
        cv_results = results['cv_results']
        model_names = list(cv_results.keys())
        accuracies = [cv_results[model]['accuracy'] for model in model_names]
        f1_scores = [cv_results[model]['f1_weighted'] for model in model_names]
        
        x = np.arange(len(model_names))
        width = 0.35
        
        ax5.bar(x - width/2, accuracies, width, label='Accuracy', alpha=0.8)
        ax5.bar(x + width/2, f1_scores, width, label='F1-Score', alpha=0.8)
        
        ax5.set_xlabel('Model')
        ax5.set_ylabel('Score')
        ax5.set_title('E. Model Comparison', fontweight='bold')
        ax5.set_xticks(x)
        ax5.set_xticklabels([m.upper() for m in model_names])
        ax5.legend()
        ax5.set_ylim(0, 1)
    else:
        # Fallback: show a simple accuracy bar
        accuracy = (y_pred == y).mean()
        ax5.bar(['Overall'], [accuracy], color='green', alpha=0.7)
        ax5.set_ylabel('Accuracy')
        ax5.set_title('E. Model Accuracy', fontweight='bold')
        ax5.set_ylim(0, 1)
    
    # Panel F: Gene cluster importance or top genes pie chart
    ax6 = plt.subplot(2, 3, 6)
    
    # Load gene clusters if available
    clusters_dir = os.path.join(model_dir, "gene_clusters")
    cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
    
    if os.path.exists(cluster_file):
        with open(cluster_file, 'rb') as f:
            cluster_results = pickle.load(f)
        
        if 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Get importance for the most pathogenic class (class 6 or 7)
            target_class = 6 if 6 in importance_data else 7 if 7 in importance_data else max(importance_data.keys())
            
            cluster_ids = list(importance_data[target_class].keys())
            cluster_importances = [importance_data[target_class][cid]['mean_abs_shap'] 
                                 for cid in cluster_ids]
            
            bars = ax6.bar([f'C{cid}' for cid in cluster_ids], cluster_importances,
                          color=plt.cm.plasma(np.linspace(0, 1, len(cluster_ids))))
            ax6.set_xlabel('Gene Cluster')
            ax6.set_ylabel('Importance')
            ax6.set_title('F. Gene Cluster Impact', fontweight='bold')
            ax6.tick_params(axis='x', rotation=45)
        else:
            # Fallback: show top feature importances in a pie chart
            top_5_indices = np.argsort(importances)[-5:]
            top_5_features = [feature_names[i] for i in top_5_indices]
            top_5_importances = importances[top_5_indices]
            
            # Format names for pie chart
            top_5_names = format_gene_names(
                top_5_features, 
                use_simple_names=use_simple_names, 
                max_length=8
            )
            
            ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
            ax6.set_title('F. Top 5 Genes', fontweight='bold')
    else:
        # Fallback: show top feature importances as pie chart
        top_5_indices = np.argsort(importances)[-5:]
        top_5_features = [feature_names[i] for i in top_5_indices]
        top_5_importances = importances[top_5_indices]
        
        # Format names for pie chart
        top_5_names = format_gene_names(
            top_5_features, 
            use_simple_names=use_simple_names, 
            max_length=8
        )
        
        ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
        ax6.set_title('F. Top 5 Genes', fontweight='bold')
    
    # Add main title
    fig.suptitle("Machine Learning-Based Prediction of NTM Pathogenicity from Genomic Data", 
                fontsize=title_size+4, fontweight='bold', y=0.96)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    
    filename = "comprehensive_conference_figure.png"
    plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    # Now create a separate SHAP transition plot
    print("Creating separate SHAP transition plot...")
    
    # Create transition data for SHAP
    from_class, to_class = 2, 4
    transition_mask = (y == from_class) | (y == to_class)
    X_transition = X_array[transition_mask]
    y_transition = y[transition_mask]
    y_binary = (y_transition == to_class).astype(int)
    
    # Train transition model
    transition_model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=5,
        tree_method='hist', device='cpu', objective='binary:logistic',
        random_state=42
    )
    transition_model.fit(X_transition, y_binary)
    
    # SHAP analysis
    explainer = shap.TreeExplainer(transition_model)
    n_samples = min(80, len(X_transition))
    sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
    X_sample = X_transition[sample_indices]
    shap_values = explainer.shap_values(X_sample)
    
    # Create standalone SHAP plot
    plt.figure(figsize=(10, 8))
    
    # Format feature names consistently
    short_feature_names = format_gene_names(
        feature_names, 
        use_simple_names=use_simple_names, 
        max_length=18
    )
    
    shap.summary_plot(shap_values, X_sample, feature_names=short_feature_names,
                     max_display=20, show=False)
    plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
             fontsize=title_size+2, pad=20)
    plt.tight_layout()
    
    shap_filename = "transition_shap_standalone.png"
    plt.savefig(os.path.join(output_dir, shap_filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"Comprehensive conference figure saved to {output_dir}/{filename}")
    print(f"Standalone SHAP plot saved to {output_dir}/{shap_filename}")
    return os.path.join(output_dir, filename)

# Example usage for different figure types
def create_all_conference_figures(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    use_simple_names=True,
    style="publication",
    from_class=0,
    to_class=6
):
    """
    Create all types of conference figures with consistent naming
    
    Parameters:
    -----------
    model_dir : str
        Directory containing trained models
    output_dir : str
        Directory to save figures
    use_simple_names : bool
        If True, cut gene names at first '~~~'. If False, use sophisticated logic
    style : str
        Figure style ('publication', 'poster', 'presentation')
    from_class : int
        Source class for transition analysis
    to_class : int
        Target class for transition analysis
    """
    
    print("Creating all conference figures with consistent gene naming...")
    
    figures_created = []
    
    # 1. Transition beeswarm (recommended for showing biological insight)
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="transition_beeswarm",
            from_class=from_class,
            to_class=to_class,
            style=style,
            use_simple_names=use_simple_names,
            max_gene_name_length=18
        )
        figures_created.append(fig_path)
        print(f"✓ Created transition beeswarm plot ({from_class} → {to_class})")
    except Exception as e:
        print(f"✗ Error creating transition beeswarm: {e}")
    
    # 2. Overall beeswarm
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="overall_beeswarm",
            style=style,
            use_simple_names=use_simple_names,
            max_gene_name_length=18
        )
        figures_created.append(fig_path)
        print("✓ Created overall beeswarm plot")
    except Exception as e:
        print(f"✗ Error creating overall beeswarm: {e}")
    
    # 3. Feature importance
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="feature_importance",
            style=style,
            use_simple_names=use_simple_names,
            max_gene_name_length=20
        )
        figures_created.append(fig_path)
        print("✓ Created feature importance plot")
    except Exception as e:
        print(f"✗ Error creating feature importance: {e}")
    
    # 4. Prediction confidence
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="prediction_confidence",
            style=style,
            use_simple_names=use_simple_names
        )
        figures_created.append(fig_path)
        print("✓ Created prediction confidence plot")
    except Exception as e:
        print(f"✗ Error creating prediction confidence: {e}")
    
    # 5. Pathway heatmap (if available)
    try:
        fig_path = create_conference_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            figure_type="pathway_heatmap",
            style=style,
            use_simple_names=use_simple_names
        )
        figures_created.append(fig_path)
        print("✓ Created pathway heatmap")
    except Exception as e:
        print(f"✗ Error creating pathway heatmap: {e}")
    
    # 6. Comprehensive multi-panel figure
    try:
        fig_path = create_multi_panel_figure(
            model_dir=model_dir,
            output_dir=output_dir,
            style=style,
            use_simple_names=use_simple_names,
            max_gene_name_length=12  # Shorter for multi-panel
        )
        figures_created.append(fig_path)
        print("✓ Created comprehensive multi-panel figure")
    except Exception as e:
        print(f"✗ Error creating multi-panel figure: {e}")
    
    print(f"\nSuccessfully created {len(figures_created)} conference figures!")
    print(f"All figures saved to: {output_dir}")
    print(f"Transition analysis: Class {from_class} → Class {to_class}")
    
    if use_simple_names:
        print("Gene names formatted using simple logic (cut at first '~~~')")
    else:
        print("Gene names formatted using sophisticated extraction logic")
    
    return figures_created

In [71]:
create_conference_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    figure_type="transition_beeswarm",
    from_class=2,
    to_class=4,
    use_simple_names=True,       # Clean names
    max_gene_name_length=15,     # Short for readability
    style="presentation"
)

Loading models and data...
Creating transition beeswarm plot for classes 2 → 4...
Conference figure saved to ./conference_figures/transition_beeswarm_2_to_4.png


'./conference_figures/transition_beeswarm_2_to_4.png'

In [72]:
# Non-pathogenic to high pathogenic
create_all_conference_figures(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./figures_0to6",
    from_class=0,
    to_class=6,
    use_simple_names=True,
    style="publication"
)

Creating all conference figures with consistent gene naming...
Loading models and data...
Creating transition beeswarm plot for classes 0 → 6...
Conference figure saved to ./figures_0to6/transition_beeswarm_0_to_6.png
✓ Created transition beeswarm plot
Loading models and data...
Creating overall pathogenicity beeswarm plot...
Conference figure saved to ./figures_0to6/overall_pathogenicity_beeswarm.png
✓ Created overall beeswarm plot
Loading models and data...
Creating feature importance plot...
Conference figure saved to ./figures_0to6/feature_importance_top20.png
✓ Created feature importance plot
Loading models and data...
Creating prediction confidence plot...
Conference figure saved to ./figures_0to6/prediction_confidence_analysis.png
✓ Created prediction confidence plot
Loading models and data...
Creating gene cluster/pathway heatmap...
Conference figure saved to ./figures_0to6/gene_cluster_importance_heatmap.png
✓ Created pathway heatmap
Creating comprehensive multi-panel conferen

['./figures_0to6/transition_beeswarm_0_to_6.png',
 './figures_0to6/overall_pathogenicity_beeswarm.png',
 './figures_0to6/feature_importance_top20.png',
 './figures_0to6/prediction_confidence_analysis.png',
 './figures_0to6/gene_cluster_importance_heatmap.png',
 './figures_0to6/comprehensive_conference_figure.png']

In [ ]:
# Example usage for different figure types
if __name__ == "__main__":
    model_dir = "./May/ntm_pathogenicity_model"
    output_dir = "./conference_figures"
    
    # Create different types of figures
    print("Creating conference figures...")
    
    # 1. Transition beeswarm (recommended for showing biological insight)
    create_original_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="transition_beeswarm",
        from_class=2,
        to_class=4,
        style="publication"
    )
    
    # 2. Overall beeswarm
    create_original_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="overall_beeswarm",
        style="publication"
    )
    
    # 3. Feature importance
    create_original_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="feature_importance",
        style="publication"
    )
    
    # 4. Comprehensive multi-panel figure
    create_original_multi_panel_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        style="publication"
    )
    
    print("All conference figures created!")

### New explainers

In [62]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import os
from sklearn.model_selection import train_test_split
import xgboost as xgb

def create_conference_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    figure_type="transition_beeswarm",
    from_class=2,
    to_class=4,
    dpi=300,
    style="publication"
):
    """
    Create publication-quality figures for conference presentation
    
    Parameters:
    -----------
    model_dir : str
        Directory containing trained models
    output_dir : str
        Directory to save figures
    figure_type : str
        Type of figure to create:
        - 'transition_beeswarm': SHAP beeswarm for class transition
        - 'overall_beeswarm': SHAP beeswarm for overall pathogenicity
        - 'pathway_heatmap': Gene cluster importance heatmap
        - 'prediction_confidence': Model confidence across classes
        - 'feature_importance': Top genes driving pathogenicity
    from_class : int
        Source class for transition analysis
    to_class : int
        Target class for transition analysis
    dpi : int
        Resolution for saved figures
    style : str
        Figure style ('publication', 'poster', 'presentation')
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set matplotlib style based on figure purpose
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 12
        title_size = 14
        label_size = 10
        figure_size = (10, 8)
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 16
        title_size = 20
        label_size = 14
        figure_size = (12, 10)
    else:  # presentation
        plt.style.use('seaborn-v0_8-darkgrid')
        font_size = 14
        title_size = 18
        label_size = 12
        figure_size = (11, 9)
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size,
        'ytick.labelsize': label_size,
        'legend.fontsize': font_size,
        'figure.titlesize': title_size + 2
    })
    
    # Load models and data
    print("Loading models and data...")
    
    # Load the dataset
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load trained models
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load feature names and class info
    with open(os.path.join(model_dir, "feature_names.pkl"), 'rb') as f:
        feature_names = pickle.load(f)
    
    with open(os.path.join(model_dir, "class_info.pkl"), 'rb') as f:
        class_info = pickle.load(f)
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    class_names = class_info['class_names']
    
    # Convert sparse to dense if needed
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use XGBoost model for SHAP analysis
    model = models['xgb']
    
    if figure_type == "transition_beeswarm":
        print(f"Creating transition beeswarm plot for classes {from_class} → {to_class}...")
        
        # Filter data for the specific transition
        transition_mask = (y == from_class) | (y == to_class)
        X_transition = X_array[transition_mask]
        y_transition = y[transition_mask]
        
        # Convert to binary classification problem
        y_binary = (y_transition == to_class).astype(int)
        
        # Train a binary classifier for this transition
        transition_model = xgb.XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            tree_method='hist',
            device='cpu',  # Use CPU for compatibility
            objective='binary:logistic',
            random_state=42
        )
        
        transition_model.fit(X_transition, y_binary)
        
        # Create SHAP explainer
        explainer = shap.TreeExplainer(transition_model)
        
        # Calculate SHAP values (use subset for speed)
        n_samples = min(100, len(X_transition))
        sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
        X_sample = X_transition[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)
        
        # Create the beeswarm plot
        plt.figure(figsize=figure_size)
        
        # Shorten feature names for better visualization
        short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                              for name in feature_names]
        
        shap.summary_plot(
            shap_values, 
            X_sample, 
            feature_names=short_feature_names,
            max_display=20,
            show=False
        )
        
        plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=label_size)
        plt.tight_layout()
        
        # Save figure
        filename = f"transition_beeswarm_{from_class}_to_{to_class}.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "overall_beeswarm":
        print("Creating overall pathogenicity beeswarm plot...")
        
        # Create SHAP explainer
        explainer = shap.TreeExplainer(model)
        
        # Use a representative sample
        n_samples = min(150, len(X_array))
        sample_indices = np.random.choice(len(X_array), n_samples, replace=False)
        X_sample = X_array[sample_indices]
        y_sample = y[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)
        
        # For multi-class, focus on high pathogenicity classes
        if isinstance(shap_values, list):
            if scoring_strategy == "high_class_emphasis":
                target_classes = high_importance_classes
            else:
                target_classes = list(range(len(shap_values)))
            
            combined_shap = np.zeros_like(shap_values[0])
            for class_idx in target_classes:
                if class_idx < len(shap_values):
                    combined_shap += shap_values[class_idx]
            shap_values_plot = combined_shap
        else:
            shap_values_plot = shap_values
        
        plt.figure(figsize=figure_size)
        
        # Shorten feature names
        short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                              for name in feature_names]
        
        shap.summary_plot(
            shap_values_plot,
            X_sample,
            feature_names=short_feature_names,
            max_display=25,
            show=False
        )
        
        plt.title("Gene Impact on NTM Pathogenicity\n(Combined effect on pathogenic classes)", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Pathogenicity)", fontsize=label_size)
        plt.tight_layout()
        
        filename = "overall_pathogenicity_beeswarm.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "feature_importance":
        print("Creating feature importance plot...")
        
        # Get feature importances from the model
        importances = model.feature_importances_
        
        # Get top 20 features
        top_indices = np.argsort(importances)[-20:]
        top_features = [feature_names[i] for i in top_indices]
        top_importances = importances[top_indices]
        
        # Shorten feature names
        short_names = [name.split('-')[0][:20] + "..." if len(name) > 23 else name 
                      for name in top_features]
        
        # Create horizontal bar plot
        plt.figure(figsize=figure_size)
        bars = plt.barh(range(len(top_features)), top_importances, 
                       color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
        
        plt.yticks(range(len(top_features)), short_names)
        plt.xlabel("Feature Importance", fontsize=label_size)
        plt.title("Top Genes Driving NTM Pathogenicity Prediction", fontsize=title_size, pad=20)
        
        # Add value labels on bars
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                    f'{width:.3f}', ha='left', va='center', fontsize=font_size-2)
        
        plt.tight_layout()
        
        filename = "feature_importance_top20.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "prediction_confidence":
        print("Creating prediction confidence plot...")
        
        # Get predictions and probabilities for all samples
        y_pred = model.predict(X_array)
        y_prob = model.predict_proba(X_array)
        
        # Calculate confidence (max probability) for each prediction
        confidence = np.max(y_prob, axis=1)
        
        # Create confidence vs class plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(figure_size[0]*1.5, figure_size[1]))
        
        # Plot 1: Confidence distribution by predicted class
        for class_idx in np.unique(y_pred):
            class_confidence = confidence[y_pred == class_idx]
            ax1.hist(class_confidence, alpha=0.6, label=f'{class_names[class_idx]}', 
                    bins=20, density=True)
        
        ax1.set_xlabel("Prediction Confidence", fontsize=label_size)
        ax1.set_ylabel("Density", fontsize=label_size)
        ax1.set_title("Model Confidence by Predicted Class", fontsize=title_size)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Confusion matrix with confidence
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=[cn.replace('Class ', '') for cn in class_names],
                   yticklabels=[cn.replace('Class ', '') for cn in class_names],
                   ax=ax2)
        
        ax2.set_xlabel("Predicted Class", fontsize=label_size)
        ax2.set_ylabel("True Class", fontsize=label_size)
        ax2.set_title("Prediction Accuracy Matrix", fontsize=title_size)
        
        plt.suptitle("NTM Pathogenicity Prediction Performance", fontsize=title_size+2, y=1.02)
        plt.tight_layout()
        
        filename = "prediction_confidence_analysis.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "pathway_heatmap":
        print("Creating gene cluster/pathway heatmap...")
        
        # Load or create gene clusters
        clusters_dir = os.path.join(model_dir, "gene_clusters")
        cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
        
        if os.path.exists(cluster_file):
            with open(cluster_file, 'rb') as f:
                cluster_results = pickle.load(f)
        else:
            # Create clusters if they don't exist
            from analysis_functions import analyze_gene_clusters
            cluster_results = analyze_gene_clusters(model_dir, n_clusters=10)
        
        if cluster_results and 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Create heatmap data
            if isinstance(list(importance_data.keys())[0], int):  # Multi-class case
                n_classes = len(importance_data)
                n_clusters = len(importance_data[0])
                
                heatmap_data = np.zeros((n_clusters, n_classes))
                
                for class_idx in range(n_classes):
                    for cluster_idx in range(n_clusters):
                        if cluster_idx in importance_data[class_idx]:
                            heatmap_data[cluster_idx, class_idx] = importance_data[class_idx][cluster_idx]['mean_abs_shap']
                
                # Create the heatmap
                plt.figure(figsize=figure_size)
                
                sns.heatmap(heatmap_data, 
                           annot=True, 
                           fmt='.3f',
                           cmap='YlOrRd',
                           xticklabels=[cn.replace('Class ', '') for cn in class_names],
                           yticklabels=[f'Cluster {i}' for i in range(n_clusters)],
                           cbar_kws={'label': 'Mean |SHAP| Value'})
                
                plt.title("Gene Cluster Importance Across Pathogenicity Classes", 
                         fontsize=title_size, pad=20)
                plt.xlabel("Pathogenicity Class", fontsize=label_size)
                plt.ylabel("Gene Cluster", fontsize=label_size)
                plt.tight_layout()
                
                filename = "gene_cluster_importance_heatmap.png"
                plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
                plt.close()
        else:
            print("Warning: Gene cluster data not available. Creating feature importance instead.")
            # Fall back to feature importance
            return create_conference_figure(model_dir, output_dir, "feature_importance", 
                                          from_class, to_class, dpi, style)
    
    print(f"Conference figure saved to {output_dir}/{filename}")
    return os.path.join(output_dir, filename)




In [63]:
def create_multi_panel_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    dpi=300,
    style="publication"
):
    """
    Create a comprehensive multi-panel figure for conference presentation
    """
    print("Creating comprehensive multi-panel conference figure...")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set style
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 10
        title_size = 12
        label_size = 9
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 14
        title_size = 16
        label_size = 12
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size-1,
        'ytick.labelsize': label_size-1,
        'legend.fontsize': font_size-1,
    })
    
    # Load data
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    with open(os.path.join(model_dir, "feature_names.pkl"), 'rb') as f:
        feature_names = pickle.load(f)
    
    with open(os.path.join(model_dir, "class_info.pkl"), 'rb') as f:
        class_info = pickle.load(f)
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    class_names = class_info['class_names']
    
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    model = models['xgb']
    
    # Create figure with subplots (2x3 layout without spanning subplot for SHAP)
    fig = plt.figure(figsize=(18, 12))
    
    # Panel A: Overall performance
    ax1 = plt.subplot(2, 3, 1)
    y_pred = model.predict(X_array)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=[cn.replace('Class ', '') for cn in class_names],
               yticklabels=[cn.replace('Class ', '') for cn in class_names],
               ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_title("A. Model Performance", fontweight='bold')
    ax1.set_xlabel("Predicted")
    ax1.set_ylabel("Actual")
    
    # Panel B: Feature importance
    ax2 = plt.subplot(2, 3, 2)
    importances = model.feature_importances_
    top_indices = np.argsort(importances)[-15:]
    top_features = [feature_names[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    short_names = [name.split('-')[0][:12] + "..." if len(name) > 15 else name 
                  for name in top_features]
    
    bars = ax2.barh(range(len(top_features)), top_importances,
                   color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
    ax2.set_yticks(range(len(top_features)))
    ax2.set_yticklabels(short_names, fontsize=label_size-1)
    ax2.set_xlabel("Importance")
    ax2.set_title("B. Top Genes", fontweight='bold')
    
    # Panel C: Class distribution
    ax3 = plt.subplot(2, 3, 3)
    class_counts = np.bincount(y, minlength=len(class_names))
    bars = ax3.bar(range(len(class_names)), class_counts, 
                  color=plt.cm.RdYlBu_r(np.linspace(0, 1, len(class_names))))
    ax3.set_xticks(range(len(class_names)))
    ax3.set_xticklabels([cn.replace('Class ', '') for cn in class_names], rotation=45)
    ax3.set_ylabel("Number of Strains")
    ax3.set_title("C. Dataset Distribution", fontweight='bold')
    
    # Panel D: Prediction confidence
    ax4 = plt.subplot(2, 3, 4)
    y_prob = model.predict_proba(X_array)
    confidence = np.max(y_prob, axis=1)
    
    ax4.hist(confidence, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax4.axvline(np.mean(confidence), color='red', linestyle='--', 
               label=f'Mean: {np.mean(confidence):.2f}')
    ax4.set_xlabel("Prediction Confidence")
    ax4.set_ylabel("Frequency")
    ax4.set_title("D. Model Confidence", fontweight='bold')
    ax4.legend()
    
    # Panel E: Model comparison
    ax5 = plt.subplot(2, 3, 5)
    
    # Load results from the model training
    results_path = os.path.join(model_dir, "results.pkl")
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            results = pickle.load(f)
        
        cv_results = results['cv_results']
        model_names = list(cv_results.keys())
        accuracies = [cv_results[model]['accuracy'] for model in model_names]
        f1_scores = [cv_results[model]['f1_weighted'] for model in model_names]
        
        x = np.arange(len(model_names))
        width = 0.35
        
        ax5.bar(x - width/2, accuracies, width, label='Accuracy', alpha=0.8)
        ax5.bar(x + width/2, f1_scores, width, label='F1-Score', alpha=0.8)
        
        ax5.set_xlabel('Model')
        ax5.set_ylabel('Score')
        ax5.set_title('E. Model Comparison', fontweight='bold')
        ax5.set_xticks(x)
        ax5.set_xticklabels([m.upper() for m in model_names])
        ax5.legend()
        ax5.set_ylim(0, 1)
    else:
        # Fallback: show a simple accuracy bar
        accuracy = (y_pred == y).mean()
        ax5.bar(['Overall'], [accuracy], color='green', alpha=0.7)
        ax5.set_ylabel('Accuracy')
        ax5.set_title('E. Model Accuracy', fontweight='bold')
        ax5.set_ylim(0, 1)
    
    # Panel F: Create a standalone SHAP plot and save it separately
    ax6 = plt.subplot(2, 3, 6)
    
    # For panel F, we'll show gene cluster importance instead of SHAP
    # Load gene clusters if available
    clusters_dir = os.path.join(model_dir, "gene_clusters")
    cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
    
    if os.path.exists(cluster_file):
        with open(cluster_file, 'rb') as f:
            cluster_results = pickle.load(f)
        
        if 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Get importance for the most pathogenic class (class 6 or 7)
            target_class = 6 if 6 in importance_data else 7 if 7 in importance_data else max(importance_data.keys())
            
            cluster_ids = list(importance_data[target_class].keys())
            cluster_importances = [importance_data[target_class][cid]['mean_abs_shap'] 
                                 for cid in cluster_ids]
            
            bars = ax6.bar([f'C{cid}' for cid in cluster_ids], cluster_importances,
                          color=plt.cm.plasma(np.linspace(0, 1, len(cluster_ids))))
            ax6.set_xlabel('Gene Cluster')
            ax6.set_ylabel('Importance')
            ax6.set_title('F. Gene Cluster Impact', fontweight='bold')
            ax6.tick_params(axis='x', rotation=45)
        else:
            # Fallback: show top feature importances in a different way
            top_5_indices = np.argsort(importances)[-5:]
            top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                          else feature_names[i] for i in top_5_indices]
            top_5_importances = importances[top_5_indices]
            
            ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
            ax6.set_title('F. Top 5 Genes', fontweight='bold')
    else:
        # Fallback: show top feature importances as pie chart
        top_5_indices = np.argsort(importances)[-5:]
        top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                      else feature_names[i] for i in top_5_indices]
        top_5_importances = importances[top_5_indices]
        
        ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
        ax6.set_title('F. Top 5 Genes', fontweight='bold')
    
    # Add main title
    fig.suptitle("Machine Learning-Based Prediction of NTM Pathogenicity from Genomic Data", 
                fontsize=title_size+4, fontweight='bold', y=0.96)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    
    filename = "comprehensive_conference_figure.png"
    plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    # Now create a separate SHAP transition plot
    print("Creating separate SHAP transition plot...")
    
    # Create transition data for SHAP
    from_class, to_class = 2, 4
    transition_mask = (y == from_class) | (y == to_class)
    X_transition = X_array[transition_mask]
    y_transition = y[transition_mask]
    y_binary = (y_transition == to_class).astype(int)
    
    # Train transition model
    transition_model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=5,
        tree_method='hist', device='cpu', objective='binary:logistic',
        random_state=42
    )
    transition_model.fit(X_transition, y_binary)
    
    # SHAP analysis
    explainer = shap.TreeExplainer(transition_model)
    n_samples = min(80, len(X_transition))
    sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
    X_sample = X_transition[sample_indices]
    shap_values = explainer.shap_values(X_sample)
    
    # Create standalone SHAP plot
    plt.figure(figsize=(10, 8))
    short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                          for name in feature_names]
    
    shap.summary_plot(shap_values, X_sample, feature_names=short_feature_names,
                     max_display=20, show=False)
    plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
             fontsize=title_size+2, pad=20)
    plt.tight_layout()
    
    shap_filename = "transition_shap_standalone.png"
    plt.savefig(os.path.join(output_dir, shap_filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"Comprehensive conference figure saved to {output_dir}/{filename}")
    print(f"Standalone SHAP plot saved to {output_dir}/{shap_filename}")
    return os.path.join(output_dir, filename)

In [64]:
# Example usage for different figure types
if __name__ == "__main__":
    model_dir = "./May/ntm_pathogenicity_model"
    output_dir = "./conference_figures"
    
    # Create different types of figures
    print("Creating conference figures...")
    
    # 1. Transition beeswarm (recommended for showing biological insight)
    create_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="transition_beeswarm",
        from_class=0,
        to_class=6,
        style="publication"
    )
    
    # 2. Overall beeswarm
    create_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="overall_beeswarm",
        style="publication"
    )
    
    # 3. Feature importance
    create_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="feature_importance",
        style="publication"
    )
    
    # 4. Comprehensive multi-panel figure
    create_multi_panel_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        style="publication"
    )
    
    print("All conference figures created!")

Creating conference figures...
Loading models and data...
Creating transition beeswarm plot for classes 0 → 6...
Conference figure saved to ./conference_figures/transition_beeswarm_0_to_6.png
Loading models and data...
Creating overall pathogenicity beeswarm plot...
Conference figure saved to ./conference_figures/overall_pathogenicity_beeswarm.png
Loading models and data...
Creating feature importance plot...
Conference figure saved to ./conference_figures/feature_importance_top20.png
Creating comprehensive multi-panel conference figure...
Creating separate SHAP transition plot...
Comprehensive conference figure saved to ./conference_figures/comprehensive_conference_figure.png
Standalone SHAP plot saved to ./conference_figures/transition_shap_standalone.png
All conference figures created!


### Original explainer

In [24]:
# Check what's in the results
results_path = "./May/ntm_pathogenicity_model/results.pkl"

if os.path.exists(results_path):
    with open(results_path, 'rb') as f:
        results = pickle.load(f)
    
    print("Keys in results:", results.keys())
    
    if 'cv_results' in results:
        print("CV results keys:", results['cv_results'].keys())
        
        if 'xgb' in results['cv_results']:
            print("XGBoost CV results keys:", results['cv_results']['xgb'].keys())
            
            # Check if we have the actual predictions
            if 'predictions' in results['cv_results']['xgb']:
                y_pred = results['cv_results']['xgb']['predictions']
                print(f"Predictions shape: {y_pred.shape}")
                print(f"Unique predictions: {np.unique(y_pred)}")
            else:
                print("No 'predictions' key found")

Keys in results: dict_keys(['cv_results', 'all_probabilities', 'all_explanations', 'models', 'explainers', 'feature_names', 'class_names', 'num_classes', 'binary_threshold'])
CV results keys: dict_keys(['xgb', 'catboost', 'rf', 'ensemble'])
XGBoost CV results keys: dict_keys(['accuracy', 'f1_macro', 'f1_weighted', 'predictions'])
Predictions shape: (174,)
Unique predictions: [0 1 2 3 4 5 6 7]


In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import os
from sklearn.model_selection import train_test_split
import xgboost as xgb

def create_original_conference_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    figure_type="transition_beeswarm",
    from_class=2,
    to_class=4,
    dpi=300,
    style="publication"
):
    """
    Create publication-quality figures for conference presentation using ORIGINAL trained models
    
    Parameters:
    -----------
    model_dir : str
        Directory containing trained models
    output_dir : str
        Directory to save figures
    figure_type : str
        Type of figure to create:
        - 'transition_beeswarm': SHAP beeswarm for class transition
        - 'overall_beeswarm': SHAP beeswarm for overall pathogenicity
        - 'pathway_heatmap': Gene cluster importance heatmap
        - 'prediction_confidence': Model confidence across classes
        - 'feature_importance': Top genes driving pathogenicity
    from_class : int
        Source class for transition analysis
    to_class : int
        Target class for transition analysis
    dpi : int
        Resolution for saved figures
    style : str
        Figure style ('publication', 'poster', 'presentation')
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set matplotlib style based on figure purpose
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 12
        title_size = 14
        label_size = 10
        figure_size = (10, 8)
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 16
        title_size = 20
        label_size = 14
        figure_size = (12, 10)
    else:  # presentation
        plt.style.use('seaborn-v0_8-darkgrid')
        font_size = 14
        title_size = 18
        label_size = 12
        figure_size = (11, 9)
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size,
        'ytick.labelsize': label_size,
        'legend.fontsize': font_size,
        'figure.titlesize': title_size + 2
    })
    
    # Load models and data
    print("Loading models and data...")
    
    # Load the dataset
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load trained models
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load the ORIGINAL explainers
    with open(os.path.join(model_dir, "shap_explainers.pkl"), 'rb') as f:
        explainers = pickle.load(f)
    
    # Load feature names and class info
    with open(os.path.join(model_dir, "feature_names.pkl"), 'rb') as f:
        feature_names = pickle.load(f)
    
    with open(os.path.join(model_dir, "class_info.pkl"), 'rb') as f:
        class_info = pickle.load(f)
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    class_names = class_info['class_names']
    
    # Convert sparse to dense if needed
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use ORIGINAL XGBoost model and explainer for consistency
    model = models['xgb']
    explainer = explainers['xgb']
    
    if figure_type == "transition_beeswarm":
        print(f"Creating transition beeswarm plot for classes {from_class} → {to_class}...")
        
        # Filter data for the specific transition
        transition_mask = (y == from_class) | (y == to_class)
        X_transition = X_array[transition_mask]
        y_transition = y[transition_mask]
        
        # Convert to binary classification problem
        y_binary = (y_transition == to_class).astype(int)
        
        # Train a binary classifier for this transition
        transition_model = xgb.XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            tree_method='hist',
            device='cpu',  # Use CPU for compatibility
            objective='binary:logistic',
            random_state=42
        )
        
        transition_model.fit(X_transition, y_binary)
        
        # Create SHAP explainer
        explainer = shap.TreeExplainer(transition_model)
        
        # Calculate SHAP values (use subset for speed)
        n_samples = min(100, len(X_transition))
        sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
        X_sample = X_transition[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)
        
        # Create the beeswarm plot
        plt.figure(figsize=figure_size)
        
        # Shorten feature names for better visualization
        short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                              for name in feature_names]
        
        shap.summary_plot(
            shap_values, 
            X_sample, 
            feature_names=short_feature_names,
            max_display=20,
            show=False
        )
        
        plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=label_size)
        plt.tight_layout()
        
        # Save figure
        filename = f"transition_beeswarm_{from_class}_to_{to_class}.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "overall_beeswarm":
        print("Creating overall pathogenicity beeswarm plot using ORIGINAL model...")
        
        # Use the ORIGINAL explainer (no need to create new one)
        # explainer is already loaded above
        
        # Use a representative sample
        n_samples = min(150, len(X_array))
        sample_indices = np.random.choice(len(X_array), n_samples, replace=False)
        X_sample = X_array[sample_indices]
        y_sample = y[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)
        
        # For multi-class, focus on high pathogenicity classes
        if isinstance(shap_values, list):
            if scoring_strategy == "high_class_emphasis":
                target_classes = high_importance_classes
            else:
                target_classes = list(range(len(shap_values)))
            
            combined_shap = np.zeros_like(shap_values[0])
            for class_idx in target_classes:
                if class_idx < len(shap_values):
                    combined_shap += shap_values[class_idx]
            shap_values_plot = combined_shap
        else:
            shap_values_plot = shap_values
        
        plt.figure(figsize=figure_size)
        
        # Shorten feature names
        short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                              for name in feature_names]
        
        shap.summary_plot(
            shap_values_plot,
            X_sample,
            feature_names=short_feature_names,
            max_display=25,
            show=False
        )
        
        plt.title("Gene Impact on NTM Pathogenicity\n(Combined effect on pathogenic classes)", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Pathogenicity)", fontsize=label_size)
        plt.tight_layout()
        
        filename = "overall_pathogenicity_beeswarm.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "feature_importance":
        print("Creating feature importance plot using ORIGINAL model...")
        
        # Get feature importances from the ORIGINAL model
        importances = model.feature_importances_
        
        # Get top 20 features
        top_indices = np.argsort(importances)[-20:]
        top_features = [feature_names[i] for i in top_indices]
        top_importances = importances[top_indices]
        
        # Shorten feature names
        short_names = [name.split('-')[0][:20] + "..." if len(name) > 23 else name 
                      for name in top_features]
        
        # Create horizontal bar plot
        plt.figure(figsize=figure_size)
        bars = plt.barh(range(len(top_features)), top_importances, 
                       color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
        
        plt.yticks(range(len(top_features)), short_names)
        plt.xlabel("Feature Importance", fontsize=label_size)
        plt.title("Top Genes Driving NTM Pathogenicity Prediction\n(From Original Model)", fontsize=title_size, pad=20)
        
        # Add value labels on bars
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                    f'{width:.3f}', ha='left', va='center', fontsize=font_size-2)
        
        plt.tight_layout()
        
        filename = "feature_importance_top20_original.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "prediction_confidence":
        print("Creating prediction confidence plot using ORIGINAL model...")
        
        # Get predictions and probabilities for all samples using ORIGINAL model
        y_pred = model.predict(X_array)
        y_prob = model.predict_proba(X_array)
        
        # Calculate confidence (max probability) for each prediction
        confidence = np.max(y_prob, axis=1)
        
        # Create confidence vs class plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(figure_size[0]*1.5, figure_size[1]))
        
        # Plot 1: Confidence distribution by predicted class
        for class_idx in np.unique(y_pred):
            class_confidence = confidence[y_pred == class_idx]
            ax1.hist(class_confidence, alpha=0.6, label=f'{class_names[class_idx]}', 
                    bins=20, density=True)
        
        ax1.set_xlabel("Prediction Confidence", fontsize=label_size)
        ax1.set_ylabel("Density", fontsize=label_size)
        ax1.set_title("Model Confidence by Predicted Class", fontsize=title_size)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Confusion matrix with confidence
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=[cn.replace('Class ', '') for cn in class_names],
                   yticklabels=[cn.replace('Class ', '') for cn in class_names],
                   ax=ax2)
        
        ax2.set_xlabel("Predicted Class", fontsize=label_size)
        ax2.set_ylabel("True Class", fontsize=label_size)
        ax2.set_title("Prediction Accuracy Matrix", fontsize=title_size)
        
        plt.suptitle("NTM Pathogenicity Prediction Performance (Original Model)", fontsize=title_size+2, y=1.02)
        plt.tight_layout()
        
        filename = "prediction_confidence_analysis_original.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "pathway_heatmap":
        print("Creating gene cluster/pathway heatmap...")
        
        # Load or create gene clusters
        clusters_dir = os.path.join(model_dir, "gene_clusters")
        cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
        
        if os.path.exists(cluster_file):
            with open(cluster_file, 'rb') as f:
                cluster_results = pickle.load(f)
        else:
            # Create clusters if they don't exist
            from analysis_functions import analyze_gene_clusters
            cluster_results = analyze_gene_clusters(model_dir, n_clusters=10)
        
        if cluster_results and 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Create heatmap data
            if isinstance(list(importance_data.keys())[0], int):  # Multi-class case
                n_classes = len(importance_data)
                n_clusters = len(importance_data[0])
                
                heatmap_data = np.zeros((n_clusters, n_classes))
                
                for class_idx in range(n_classes):
                    for cluster_idx in range(n_clusters):
                        if cluster_idx in importance_data[class_idx]:
                            heatmap_data[cluster_idx, class_idx] = importance_data[class_idx][cluster_idx]['mean_abs_shap']
                
                # Create the heatmap
                plt.figure(figsize=figure_size)
                
                sns.heatmap(heatmap_data, 
                           annot=True, 
                           fmt='.3f',
                           cmap='YlOrRd',
                           xticklabels=[cn.replace('Class ', '') for cn in class_names],
                           yticklabels=[f'Cluster {i}' for i in range(n_clusters)],
                           cbar_kws={'label': 'Mean |SHAP| Value'})
                
                plt.title("Gene Cluster Importance Across Pathogenicity Classes", 
                         fontsize=title_size, pad=20)
                plt.xlabel("Pathogenicity Class", fontsize=label_size)
                plt.ylabel("Gene Cluster", fontsize=label_size)
                plt.tight_layout()
                
                filename = "gene_cluster_importance_heatmap.png"
                plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
                plt.close()
        else:
            print("Warning: Gene cluster data not available. Creating feature importance instead.")
            # Fall back to feature importance
            return create_conference_figure(model_dir, output_dir, "feature_importance", 
                                          from_class, to_class, dpi, style)
    
    print(f"Conference figure saved to {output_dir}/{filename}")
    return os.path.join(output_dir, filename)


In [22]:
def create_original_multi_panel_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    dpi=300,
    style="publication"
):
    """
    Create a comprehensive multi-panel figure for conference presentation using ORIGINAL models
    """
    print("Creating comprehensive multi-panel conference figure using ORIGINAL models...")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set style
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 10
        title_size = 12
        label_size = 9
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 14
        title_size = 16
        label_size = 12
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size-1,
        'ytick.labelsize': label_size-1,
        'legend.fontsize': font_size-1,
    })
    
    # Load data
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load ORIGINAL trained models
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load the ORIGINAL explainers
    with open(os.path.join(model_dir, "shap_explainers.pkl"), 'rb') as f:
        explainers = pickle.load(f)
    
    with open(os.path.join(model_dir, "feature_names.pkl"), 'rb') as f:
        feature_names = pickle.load(f)
    
    with open(os.path.join(model_dir, "class_info.pkl"), 'rb') as f:
        class_info = pickle.load(f)
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    class_names = class_info['class_names']
    
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use ORIGINAL model and explainer
    model = models['xgb']
    explainer = explainers['xgb']
    
    # Create figure with subplots (2x3 layout without spanning subplot for SHAP)
    fig = plt.figure(figsize=(18, 12))
    
    # Panel A: Overall performance
    ax1 = plt.subplot(2, 3, 1)
    y_pred = model.predict(X_array)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=[cn.replace('Class ', '') for cn in class_names],
               yticklabels=[cn.replace('Class ', '') for cn in class_names],
               ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_title("A. Model Performance", fontweight='bold')
    ax1.set_xlabel("Predicted")
    ax1.set_ylabel("Actual")
    
    # Panel B: Feature importance
    ax2 = plt.subplot(2, 3, 2)
    importances = model.feature_importances_
    top_indices = np.argsort(importances)[-15:]
    top_features = [feature_names[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    short_names = [name.split('-')[0][:12] + "..." if len(name) > 15 else name 
                  for name in top_features]
    
    bars = ax2.barh(range(len(top_features)), top_importances,
                   color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
    ax2.set_yticks(range(len(top_features)))
    ax2.set_yticklabels(short_names, fontsize=label_size-1)
    ax2.set_xlabel("Importance")
    ax2.set_title("B. Top Genes", fontweight='bold')
    
    # Panel C: Class distribution
    ax3 = plt.subplot(2, 3, 3)
    class_counts = np.bincount(y, minlength=len(class_names))
    bars = ax3.bar(range(len(class_names)), class_counts, 
                  color=plt.cm.RdYlBu_r(np.linspace(0, 1, len(class_names))))
    ax3.set_xticks(range(len(class_names)))
    ax3.set_xticklabels([cn.replace('Class ', '') for cn in class_names], rotation=45)
    ax3.set_ylabel("Number of Strains")
    ax3.set_title("C. Dataset Distribution", fontweight='bold')
    
    # Panel D: Prediction confidence
    ax4 = plt.subplot(2, 3, 4)
    y_prob = model.predict_proba(X_array)
    confidence = np.max(y_prob, axis=1)
    
    ax4.hist(confidence, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax4.axvline(np.mean(confidence), color='red', linestyle='--', 
               label=f'Mean: {np.mean(confidence):.2f}')
    ax4.set_xlabel("Prediction Confidence")
    ax4.set_ylabel("Frequency")
    ax4.set_title("D. Model Confidence", fontweight='bold')
    ax4.legend()
    
    # Panel E: Model comparison
    ax5 = plt.subplot(2, 3, 5)
    
    # Load results from the model training
    results_path = os.path.join(model_dir, "results.pkl")
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            results = pickle.load(f)
        
        cv_results = results['cv_results']
        model_names = list(cv_results.keys())
        accuracies = [cv_results[model]['accuracy'] for model in model_names]
        f1_scores = [cv_results[model]['f1_weighted'] for model in model_names]
        
        x = np.arange(len(model_names))
        width = 0.35
        
        ax5.bar(x - width/2, accuracies, width, label='Accuracy', alpha=0.8)
        ax5.bar(x + width/2, f1_scores, width, label='F1-Score', alpha=0.8)
        
        ax5.set_xlabel('Model')
        ax5.set_ylabel('Score')
        ax5.set_title('E. Model Comparison', fontweight='bold')
        ax5.set_xticks(x)
        ax5.set_xticklabels([m.upper() for m in model_names])
        ax5.legend()
        ax5.set_ylim(0, 1)
    else:
        # Fallback: show a simple accuracy bar
        accuracy = (y_pred == y).mean()
        ax5.bar(['Overall'], [accuracy], color='green', alpha=0.7)
        ax5.set_ylabel('Accuracy')
        ax5.set_title('E. Model Accuracy', fontweight='bold')
        ax5.set_ylim(0, 1)
    
    # Panel F: Create a standalone SHAP plot and save it separately
    ax6 = plt.subplot(2, 3, 6)
    
    # For panel F, we'll show gene cluster importance instead of SHAP
    # Load gene clusters if available
    clusters_dir = os.path.join(model_dir, "gene_clusters")
    cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
    
    if os.path.exists(cluster_file):
        with open(cluster_file, 'rb') as f:
            cluster_results = pickle.load(f)
        
        if 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Get importance for the most pathogenic class (class 6 or 7)
            target_class = 6 if 6 in importance_data else 7 if 7 in importance_data else max(importance_data.keys())
            
            cluster_ids = list(importance_data[target_class].keys())
            cluster_importances = [importance_data[target_class][cid]['mean_abs_shap'] 
                                 for cid in cluster_ids]
            
            bars = ax6.bar([f'C{cid}' for cid in cluster_ids], cluster_importances,
                          color=plt.cm.plasma(np.linspace(0, 1, len(cluster_ids))))
            ax6.set_xlabel('Gene Cluster')
            ax6.set_ylabel('Importance')
            ax6.set_title('F. Gene Cluster Impact', fontweight='bold')
            ax6.tick_params(axis='x', rotation=45)
        else:
            # Fallback: show top feature importances in a different way
            top_5_indices = np.argsort(importances)[-5:]
            top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                          else feature_names[i] for i in top_5_indices]
            top_5_importances = importances[top_5_indices]
            
            ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
            ax6.set_title('F. Top 5 Genes', fontweight='bold')
    else:
        # Fallback: show top feature importances as pie chart
        top_5_indices = np.argsort(importances)[-5:]
        top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                      else feature_names[i] for i in top_5_indices]
        top_5_importances = importances[top_5_indices]
        
        ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
        ax6.set_title('F. Top 5 Genes', fontweight='bold')
    
    # Add main title
    fig.suptitle("Machine Learning-Based Prediction of NTM Pathogenicity from Genomic Data", 
                fontsize=title_size+4, fontweight='bold', y=0.96)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    
    filename = "comprehensive_conference_figure.png"
    plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    # Now create a separate SHAP transition plot
    print("Creating separate SHAP transition plot...")
    
    # Create transition data for SHAP
    from_class, to_class = 2, 4
    transition_mask = (y == from_class) | (y == to_class)
    X_transition = X_array[transition_mask]
    y_transition = y[transition_mask]
    y_binary = (y_transition == to_class).astype(int)
    
    # Train transition model
    transition_model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=5,
        tree_method='hist', device='cpu', objective='binary:logistic',
        random_state=42
    )
    transition_model.fit(X_transition, y_binary)
    
    # SHAP analysis
    explainer = shap.TreeExplainer(transition_model)
    n_samples = min(80, len(X_transition))
    sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
    X_sample = X_transition[sample_indices]
    shap_values = explainer.shap_values(X_sample)
    
    # Create standalone SHAP plot
    plt.figure(figsize=(10, 8))
    short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                          for name in feature_names]
    
    shap.summary_plot(shap_values, X_sample, feature_names=short_feature_names,
                     max_display=20, show=False)
    plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
             fontsize=title_size+2, pad=20)
    plt.tight_layout()
    
    shap_filename = "transition_shap_standalone.png"
    plt.savefig(os.path.join(output_dir, shap_filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"Comprehensive conference figure saved to {output_dir}/{filename}")
    print(f"Standalone SHAP plot saved to {output_dir}/{shap_filename}")
    return os.path.join(output_dir, filename)


In [23]:
# Example usage for different figure types
if __name__ == "__main__":
    model_dir = "./May/ntm_pathogenicity_model"
    output_dir = "./conference_figures"
    
    # Create different types of figures
    print("Creating conference figures...")
    
    # 1. Transition beeswarm (recommended for showing biological insight)
    create_original_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="transition_beeswarm",
        from_class=2,
        to_class=4,
        style="publication"
    )
    
    # 2. Overall beeswarm
    create_original_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="overall_beeswarm",
        style="publication"
    )
    
    # 3. Feature importance
    create_original_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="feature_importance",
        style="publication"
    )
    
    # 4. Comprehensive multi-panel figure
    create_original_multi_panel_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        style="publication"
    )
    
    print("All conference figures created!")

Creating conference figures...
Loading models and data...
Creating transition beeswarm plot for classes 2 → 4...
Conference figure saved to ./conference_figures/transition_beeswarm_2_to_4.png
Loading models and data...
Creating overall pathogenicity beeswarm plot using ORIGINAL model...
Conference figure saved to ./conference_figures/overall_pathogenicity_beeswarm.png
Loading models and data...
Creating feature importance plot using ORIGINAL model...
Conference figure saved to ./conference_figures/feature_importance_top20_original.png
Creating comprehensive multi-panel conference figure using ORIGINAL models...
Creating separate SHAP transition plot...
Comprehensive conference figure saved to ./conference_figures/comprehensive_conference_figure.png
Standalone SHAP plot saved to ./conference_figures/transition_shap_standalone.png
All conference figures created!


### Original explainer fixed cross validation

In [86]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import os
from sklearn.model_selection import train_test_split
import xgboost as xgb

def create_conference_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    figure_type="transition_beeswarm",
    from_class=2,
    to_class=4,
    dpi=300,
    style="publication"
):
    """
    Create publication-quality figures for conference presentation using PROPER CV predictions
    
    Parameters:
    -----------
    model_dir : str
        Directory containing trained models
    output_dir : str
        Directory to save figures
    figure_type : str
        Type of figure to create:
        - 'transition_beeswarm': SHAP beeswarm for class transition
        - 'overall_beeswarm': SHAP beeswarm for overall pathogenicity
        - 'pathway_heatmap': Gene cluster importance heatmap
        - 'prediction_confidence': Model confidence across classes
        - 'feature_importance': Top genes driving pathogenicity
    from_class : int
        Source class for transition analysis
    to_class : int
        Target class for transition analysis
    dpi : int
        Resolution for saved figures
    style : str
        Figure style ('publication', 'poster', 'presentation')
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set matplotlib style based on figure purpose
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 12
        title_size = 14
        label_size = 10
        figure_size = (10, 8)
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 16
        title_size = 20
        label_size = 14
        figure_size = (12, 10)
    else:  # presentation
        plt.style.use('seaborn-v0_8-darkgrid')
        font_size = 14
        title_size = 18
        label_size = 12
        figure_size = (11, 9)
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size,
        'ytick.labelsize': label_size,
        'legend.fontsize': font_size,
        'figure.titlesize': title_size + 2
    })
    
    # Load models and data
    print("Loading original models, data, and CV results...")
    
    # Load the ORIGINAL CV results (this is key!)
    results_path = os.path.join(model_dir, "results.pkl")
    with open(results_path, 'rb') as f:
        original_results = pickle.load(f)
    
    # Load the dataset
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load trained models (for SHAP and feature importance, NOT for evaluation)
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load the ORIGINAL explainers
    with open(os.path.join(model_dir, "shap_explainers.pkl"), 'rb') as f:
        explainers = pickle.load(f)
    
    # Get data from original results
    feature_names = original_results['feature_names']
    class_names = original_results['class_names']
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    
    # Convert sparse to dense if needed
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use ORIGINAL XGBoost model and explainer for SHAP/importance (NOT for evaluation)
    model = models['xgb']
    explainer = explainers['xgb']
    
    # Get the PROPER CV predictions (this is the key fix!)
    y_pred_cv = original_results['cv_results']['xgb']['predictions']
    
    print(f"Using LOOCV predictions with accuracy: {original_results['cv_results']['xgb']['accuracy']:.4f}")
    
    if figure_type == "transition_beeswarm":
        print(f"Creating transition beeswarm plot for classes {from_class} → {to_class}...")
        
        # Filter data for the specific transition
        transition_mask = (y == from_class) | (y == to_class)
        X_transition = X_array[transition_mask]
        y_transition = y[transition_mask]
        
        # Convert to binary classification problem
        y_binary = (y_transition == to_class).astype(int)
        
        # Train a binary classifier for this transition (this is still needed)
        transition_model = xgb.XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            tree_method='hist',
            device='cpu',  # Use CPU for compatibility
            objective='binary:logistic',
            random_state=42
        )
        
        transition_model.fit(X_transition, y_binary)
        
        # Create SHAP explainer for transition
        transition_explainer = shap.TreeExplainer(transition_model)
        
        # Calculate SHAP values (use subset for speed)
        n_samples = min(100, len(X_transition))
        sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
        X_sample = X_transition[sample_indices]
        
        shap_values = transition_explainer.shap_values(X_sample)
        
        # Create the beeswarm plot
        plt.figure(figsize=figure_size)
        
        # Shorten feature names for better visualization
        short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                              for name in feature_names]
        
        shap.summary_plot(
            shap_values, 
            X_sample, 
            feature_names=short_feature_names,
            max_display=20,
            show=False
        )
        
        plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=label_size)
        plt.tight_layout()
        
        # Save figure
        filename = f"transition_beeswarm_{from_class}_to_{to_class}_corrected.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "overall_beeswarm":
        print("Creating overall pathogenicity beeswarm plot using ORIGINAL explainer...")
        
        # Use the ORIGINAL explainer (no need to create new one)
        # Use a representative sample
        n_samples = min(150, len(X_array))
        sample_indices = np.random.choice(len(X_array), n_samples, replace=False)
        X_sample = X_array[sample_indices]
        y_sample = y[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)
        
        # For multi-class, focus on high pathogenicity classes
        if isinstance(shap_values, list):
            if scoring_strategy == "high_class_emphasis":
                target_classes = high_importance_classes
            else:
                target_classes = list(range(len(shap_values)))
            
            combined_shap = np.zeros_like(shap_values[0])
            for class_idx in target_classes:
                if class_idx < len(shap_values):
                    combined_shap += shap_values[class_idx]
            shap_values_plot = combined_shap
        else:
            shap_values_plot = shap_values
        
        plt.figure(figsize=figure_size)
        
        # Shorten feature names
        short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                              for name in feature_names]
        
        shap.summary_plot(
            shap_values_plot,
            X_sample,
            feature_names=short_feature_names,
            max_display=25,
            show=False
        )
        
        plt.title("Gene Impact on NTM Pathogenicity\n(Combined effect on pathogenic classes)", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Pathogenicity)", fontsize=label_size)
        plt.tight_layout()
        
        filename = "overall_pathogenicity_beeswarm_corrected.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "feature_importance":
        print("Creating feature importance plot using ORIGINAL model...")
        
        # Get feature importances from the ORIGINAL model
        importances = model.feature_importances_
        
        # Get top 20 features
        top_indices = np.argsort(importances)[-20:]
        top_features = [feature_names[i] for i in top_indices]
        top_importances = importances[top_indices]
        
        # Shorten feature names
        short_names = [name.split('-')[0][:20] + "..." if len(name) > 23 else name 
                      for name in top_features]
        
        # Create horizontal bar plot
        plt.figure(figsize=figure_size)
        bars = plt.barh(range(len(top_features)), top_importances, 
                       color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
        
        plt.yticks(range(len(top_features)), short_names)
        plt.xlabel("Feature Importance", fontsize=label_size)
        plt.title("Top Genes Driving NTM Pathogenicity Prediction\n(From Original Model)", fontsize=title_size, pad=20)
        
        # Add value labels on bars
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                    f'{width:.3f}', ha='left', va='center', fontsize=font_size-2)
        
        plt.tight_layout()
        
        filename = "feature_importance_top20_corrected.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "prediction_confidence":
        print("Creating prediction confidence plot using ORIGINAL CV results...")
        
        # Use the CV probabilities if available
        if 'all_probabilities' in original_results and 'xgb' in original_results['all_probabilities']:
            y_prob_cv = original_results['all_probabilities']['xgb']
            confidence = np.max(y_prob_cv, axis=1)
        else:
            # Fallback: calculate from final model (but this won't be as accurate)
            print("Warning: Using final model for probabilities - not ideal!")
            y_prob = model.predict_proba(X_array)
            confidence = np.max(y_prob, axis=1)
        
        # Create confidence vs class plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(figure_size[0]*1.5, figure_size[1]))
        
        # Plot 1: Confidence distribution by predicted class
        for class_idx in np.unique(y_pred_cv):
            class_confidence = confidence[y_pred_cv == class_idx]
            ax1.hist(class_confidence, alpha=0.6, label=f'{class_names[class_idx]}', 
                    bins=20, density=True)
        
        ax1.set_xlabel("Prediction Confidence", fontsize=label_size)
        ax1.set_ylabel("Density", fontsize=label_size)
        ax1.set_title("Model Confidence by Predicted Class", fontsize=title_size)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Confusion matrix with PROPER CV predictions
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y, y_pred_cv)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=[cn.replace('Class ', '') for cn in class_names],
                   yticklabels=[cn.replace('Class ', '') for cn in class_names],
                   ax=ax2)
        
        ax2.set_xlabel("Predicted Class", fontsize=label_size)
        ax2.set_ylabel("True Class", fontsize=label_size)
        ax2.set_title("LOOCV Prediction Accuracy", fontsize=title_size)
        
        plt.suptitle("NTM Pathogenicity Prediction Performance (LOOCV)", fontsize=title_size+2, y=1.02)
        plt.tight_layout()
        
        filename = "prediction_confidence_analysis_corrected.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
    
    if figure_type == "transition_beeswarm":
        print(f"Creating transition beeswarm plot for classes {from_class} → {to_class}...")
        
        # Filter data for the specific transition
        transition_mask = (y == from_class) | (y == to_class)
        X_transition = X_array[transition_mask]
        y_transition = y[transition_mask]
        
        # Convert to binary classification problem
        y_binary = (y_transition == to_class).astype(int)
        
        # Train a binary classifier for this transition
        transition_model = xgb.XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=5,
            tree_method='hist',
            device='cpu',  # Use CPU for compatibility
            objective='binary:logistic',
            random_state=42
        )
        
        transition_model.fit(X_transition, y_binary)
        
        # Create SHAP explainer
        explainer = shap.TreeExplainer(transition_model)
        
        # Calculate SHAP values (use subset for speed)
        n_samples = min(100, len(X_transition))
        sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
        X_sample = X_transition[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)
        
        # Create the beeswarm plot
        plt.figure(figsize=figure_size)
        
        # Shorten feature names for better visualization
        short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                              for name in feature_names]
        
        shap.summary_plot(
            shap_values, 
            X_sample, 
            feature_names=short_feature_names,
            max_display=20,
            show=False
        )
        
        plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Prediction)", fontsize=label_size)
        plt.tight_layout()
        
        # Save figure
        filename = f"transition_beeswarm_{from_class}_to_{to_class}.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "overall_beeswarm":
        print("Creating overall pathogenicity beeswarm plot using ORIGINAL model...")
        
        # Use the ORIGINAL explainer (no need to create new one)
        # explainer is already loaded above
        
        # Use a representative sample
        n_samples = min(150, len(X_array))
        sample_indices = np.random.choice(len(X_array), n_samples, replace=False)
        X_sample = X_array[sample_indices]
        y_sample = y[sample_indices]
        
        shap_values = explainer.shap_values(X_sample)
        
        # For multi-class, focus on high pathogenicity classes
        if isinstance(shap_values, list):
            if scoring_strategy == "high_class_emphasis":
                target_classes = high_importance_classes
            else:
                target_classes = list(range(len(shap_values)))
            
            combined_shap = np.zeros_like(shap_values[0])
            for class_idx in target_classes:
                if class_idx < len(shap_values):
                    combined_shap += shap_values[class_idx]
            shap_values_plot = combined_shap
        else:
            shap_values_plot = shap_values
        
        plt.figure(figsize=figure_size)
        
        # Shorten feature names
        short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                              for name in feature_names]
        
        shap.summary_plot(
            shap_values_plot,
            X_sample,
            feature_names=short_feature_names,
            max_display=25,
            show=False
        )
        
        plt.title("Gene Impact on NTM Pathogenicity\n(Combined effect on pathogenic classes)", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Pathogenicity)", fontsize=label_size)
        plt.tight_layout()
        
        filename = "overall_pathogenicity_beeswarm.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "feature_importance":
        print("Creating feature importance plot using ORIGINAL model...")
        
        # Get feature importances from the ORIGINAL model
        importances = model.feature_importances_
        
        # Get top 20 features
        top_indices = np.argsort(importances)[-20:]
        top_features = [feature_names[i] for i in top_indices]
        top_importances = importances[top_indices]
        
        # Shorten feature names
        short_names = [name.split('-')[0][:20] + "..." if len(name) > 23 else name 
                      for name in top_features]
        
        # Create horizontal bar plot
        plt.figure(figsize=figure_size)
        bars = plt.barh(range(len(top_features)), top_importances, 
                       color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
        
        plt.yticks(range(len(top_features)), short_names)
        plt.xlabel("Feature Importance", fontsize=label_size)
        plt.title("Top Genes Driving NTM Pathogenicity Prediction\n(From Original Model)", fontsize=title_size, pad=20)
        
        # Add value labels on bars
        for i, bar in enumerate(bars):
            width = bar.get_width()
            plt.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
                    f'{width:.3f}', ha='left', va='center', fontsize=font_size-2)
        
        plt.tight_layout()
        
        filename = "feature_importance_top20_original.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "prediction_confidence":
        print("Creating prediction confidence plot using ORIGINAL model...")
        
        # Get predictions and probabilities for all samples using ORIGINAL model
        y_pred = model.predict(X_array)
        y_prob = model.predict_proba(X_array)
        
        # Calculate confidence (max probability) for each prediction
        confidence = np.max(y_prob, axis=1)
        
        # Create confidence vs class plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(figure_size[0]*1.5, figure_size[1]))
        
        # Plot 1: Confidence distribution by predicted class
        for class_idx in np.unique(y_pred):
            class_confidence = confidence[y_pred == class_idx]
            ax1.hist(class_confidence, alpha=0.6, label=f'{class_names[class_idx]}', 
                    bins=20, density=True)
        
        ax1.set_xlabel("Prediction Confidence", fontsize=label_size)
        ax1.set_ylabel("Density", fontsize=label_size)
        ax1.set_title("Model Confidence by Predicted Class", fontsize=title_size)
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Confusion matrix with confidence
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y, y_pred)
        
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=[cn.replace('Class ', '') for cn in class_names],
                   yticklabels=[cn.replace('Class ', '') for cn in class_names],
                   ax=ax2)
        
        ax2.set_xlabel("Predicted Class", fontsize=label_size)
        ax2.set_ylabel("True Class", fontsize=label_size)
        ax2.set_title("Prediction Accuracy Matrix", fontsize=title_size)
        
        plt.suptitle("NTM Pathogenicity Prediction Performance (Original Model)", fontsize=title_size+2, y=1.02)
        plt.tight_layout()
        
        filename = "prediction_confidence_analysis_original.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
        plt.close()
        
    elif figure_type == "pathway_heatmap":
        print("Creating gene cluster/pathway heatmap...")
        
        # Load or create gene clusters
        clusters_dir = os.path.join(model_dir, "gene_clusters")
        cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
        
        if os.path.exists(cluster_file):
            with open(cluster_file, 'rb') as f:
                cluster_results = pickle.load(f)
        else:
            # Create clusters if they don't exist
            from analysis_functions import analyze_gene_clusters
            cluster_results = analyze_gene_clusters(model_dir, n_clusters=10)
        
        if cluster_results and 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Create heatmap data
            if isinstance(list(importance_data.keys())[0], int):  # Multi-class case
                n_classes = len(importance_data)
                n_clusters = len(importance_data[0])
                
                heatmap_data = np.zeros((n_clusters, n_classes))
                
                for class_idx in range(n_classes):
                    for cluster_idx in range(n_clusters):
                        if cluster_idx in importance_data[class_idx]:
                            heatmap_data[cluster_idx, class_idx] = importance_data[class_idx][cluster_idx]['mean_abs_shap']
                
                # Create the heatmap
                plt.figure(figsize=figure_size)
                
                sns.heatmap(heatmap_data, 
                           annot=True, 
                           fmt='.3f',
                           cmap='YlOrRd',
                           xticklabels=[cn.replace('Class ', '') for cn in class_names],
                           yticklabels=[f'Cluster {i}' for i in range(n_clusters)],
                           cbar_kws={'label': 'Mean |SHAP| Value'})
                
                plt.title("Gene Cluster Importance Across Pathogenicity Classes", 
                         fontsize=title_size, pad=20)
                plt.xlabel("Pathogenicity Class", fontsize=label_size)
                plt.ylabel("Gene Cluster", fontsize=label_size)
                plt.tight_layout()
                
                filename = "gene_cluster_importance_heatmap.png"
                plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
                plt.close()
        else:
            print("Warning: Gene cluster data not available. Creating feature importance instead.")
            # Fall back to feature importance
            return create_conference_figure(model_dir, output_dir, "feature_importance", 
                                          from_class, to_class, dpi, style)
    
    print(f"Conference figure saved to {output_dir}/{filename}")
    return os.path.join(output_dir, filename)


In [87]:
def create_multi_panel_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    dpi=300,
    style="publication"
):
    """
    Create a comprehensive multi-panel figure using PROPER CV predictions
    """
    print("Creating comprehensive multi-panel conference figure using PROPER CV predictions...")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set style
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 10
        title_size = 12
        label_size = 9
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 14
        title_size = 16
        label_size = 12
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size-1,
        'ytick.labelsize': label_size-1,
        'legend.fontsize': font_size-1,
    })
    
    # Load the ORIGINAL CV results (this is key!)
    results_path = os.path.join(model_dir, "results.pkl")
    with open(results_path, 'rb') as f:
        original_results = pickle.load(f)
    
    # Load data
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load ORIGINAL trained models (for feature importance only)
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load the ORIGINAL explainers
    with open(os.path.join(model_dir, "shap_explainers.pkl"), 'rb') as f:
        explainers = pickle.load(f)
    
    # Get data from original results
    feature_names = original_results['feature_names']
    class_names = original_results['class_names']
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use ORIGINAL model and explainer
    model = models['xgb']
    explainer = explainers['xgb']
    
    # Get the PROPER CV predictions
    y_pred_cv = original_results['cv_results']['xgb']['predictions']
    cv_results = original_results['cv_results']
    
    # Create figure with subplots (2x3 layout)
    fig = plt.figure(figsize=(18, 12))
    
    # Panel A: PROPER performance using CV predictions
    ax1 = plt.subplot(2, 3, 1)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y, y_pred_cv)  # Use CV predictions!
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=[cn.replace('Class ', '') for cn in class_names],
               yticklabels=[cn.replace('Class ', '') for cn in class_names],
               ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_title("A. LOOCV Performance", fontweight='bold')
    ax1.set_xlabel("Predicted")
    ax1.set_ylabel("Actual")
    
    # Panel B: Feature importance from ORIGINAL model
    ax2 = plt.subplot(2, 3, 2)
    importances = model.feature_importances_
    top_indices = np.argsort(importances)[-15:]
    top_features = [feature_names[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    short_names = [name.split('-')[0][:12] + "..." if len(name) > 15 else name 
                  for name in top_features]
    
    bars = ax2.barh(range(len(top_features)), top_importances,
                   color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
    ax2.set_yticks(range(len(top_features)))
    ax2.set_yticklabels(short_names, fontsize=label_size-1)
    ax2.set_xlabel("Importance")
    ax2.set_title("B. Top Genes", fontweight='bold')
    
    # Panel C: Class distribution
    ax3 = plt.subplot(2, 3, 3)
    class_counts = np.bincount(y, minlength=len(class_names))
    bars = ax3.bar(range(len(class_names)), class_counts, 
                  color=plt.cm.RdYlBu_r(np.linspace(0, 1, len(class_names))))
    ax3.set_xticks(range(len(class_names)))
    ax3.set_xticklabels([cn.replace('Class ', '') for cn in class_names], rotation=45)
    ax3.set_ylabel("Number of Strains")
    ax3.set_title("C. Dataset Distribution", fontweight='bold')
    
    # Panel D: Prediction confidence using CV probabilities
    ax4 = plt.subplot(2, 3, 4)
    
    if 'all_probabilities' in original_results and 'xgb' in original_results['all_probabilities']:
        y_prob_cv = original_results['all_probabilities']['xgb']
        confidence = np.max(y_prob_cv, axis=1)
    else:
        # Fallback to model predictions (not ideal)
        y_prob = model.predict_proba(X_array)
        confidence = np.max(y_prob, axis=1)
    
    ax4.hist(confidence, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax4.axvline(np.mean(confidence), color='red', linestyle='--', 
               label=f'Mean: {np.mean(confidence):.2f}')
    ax4.set_xlabel("Prediction Confidence")
    ax4.set_ylabel("Frequency")
    ax4.set_title("D. LOOCV Confidence", fontweight='bold')
    ax4.legend()
    
    # Panel E: Model comparison from ORIGINAL CV results
    ax5 = plt.subplot(2, 3, 5)
    
    model_names = list(cv_results.keys())
    accuracies = [cv_results[model]['accuracy'] for model in model_names]
    f1_scores = [cv_results[model]['f1_weighted'] for model in model_names]
    
    x = np.arange(len(model_names))
    width = 0.35
    
    ax5.bar(x - width/2, accuracies, width, label='Accuracy', alpha=0.8)
    ax5.bar(x + width/2, f1_scores, width, label='F1-Score', alpha=0.8)
    
    ax5.set_xlabel('Model')
    ax5.set_ylabel('Score')
    ax5.set_title('E. Model Comparison', fontweight='bold')
    ax5.set_xticks(x)
    ax5.set_xticklabels([m.upper() for m in model_names])
    ax5.legend()
    ax5.set_ylim(0, 1)
    
    # Panel F: Gene cluster importance or top genes from ORIGINAL analysis
    ax6 = plt.subplot(2, 3, 6)
    
    # Load gene clusters if available
    clusters_dir = os.path.join(model_dir, "gene_clusters")
    cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
    
    if os.path.exists(cluster_file):
        with open(cluster_file, 'rb') as f:
            cluster_results = pickle.load(f)
        
        if 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Get importance for the most pathogenic class (class 6 or 7)
            target_class = 6 if 6 in importance_data else 7 if 7 in importance_data else max(importance_data.keys())
            
            cluster_ids = list(importance_data[target_class].keys())
            cluster_importances = [importance_data[target_class][cid]['mean_abs_shap'] 
                                 for cid in cluster_ids]
            
            bars = ax6.bar([f'C{cid}' for cid in cluster_ids], cluster_importances,
                          color=plt.cm.plasma(np.linspace(0, 1, len(cluster_ids))))
            ax6.set_xlabel('Gene Cluster')
            ax6.set_ylabel('Importance')
            ax6.set_title('F. Gene Cluster Impact', fontweight='bold')
            ax6.tick_params(axis='x', rotation=45)
        else:
            # Fallback: show top feature importances in a different way
            top_5_indices = np.argsort(importances)[-5:]
            top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                          else feature_names[i] for i in top_5_indices]
            top_5_importances = importances[top_5_indices]
            
            ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
            ax6.set_title('F. Top 5 Genes', fontweight='bold')
    else:
        # Fallback: show top feature importances as pie chart
        top_5_indices = np.argsort(importances)[-5:]
        top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                      else feature_names[i] for i in top_5_indices]
        top_5_importances = importances[top_5_indices]
        
        ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
        ax6.set_title('F. Top 5 Genes', fontweight='bold')
    
    # Add main title
    fig.suptitle("Machine Learning-Based Prediction of NTM Pathogenicity from Genomic Data\n(Leave-One-Out Cross-Validation Results)", 
                fontsize=title_size+4, fontweight='bold', y=0.96)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    
    filename = "comprehensive_conference_figure_corrected.png"
    plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    # Create a separate SHAP transition plot (still needs new model for transition)
    print("Creating separate SHAP transition plot...")
    
    # Create transition data for SHAP (this still needs a new binary model)
    from_class, to_class = 2, 4
    transition_mask = (y == from_class) | (y == to_class)
    X_transition = X_array[transition_mask]
    y_transition = y[transition_mask]
    y_binary = (y_transition == to_class).astype(int)
    
    # Train transition model (this is still needed for binary classification)
    transition_model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=5,
        tree_method='hist', device='cpu', objective='binary:logistic',
        random_state=42
    )
    transition_model.fit(X_transition, y_binary)
    
    # SHAP analysis for transition
    transition_explainer = shap.TreeExplainer(transition_model)
    n_samples = min(80, len(X_transition))
    sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
    X_sample = X_transition[sample_indices]
    shap_values = transition_explainer.shap_values(X_sample)
    
    # Create standalone SHAP plot
    plt.figure(figsize=(10, 8))
    short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                          for name in feature_names]
    
    shap.summary_plot(shap_values, X_sample, feature_names=short_feature_names,
                     max_display=20, show=False)
    plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
             fontsize=title_size+2, pad=20)
    plt.tight_layout()
    
    shap_filename = "transition_shap_standalone_corrected.png"
    plt.savefig(os.path.join(output_dir, shap_filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"Comprehensive conference figure saved to {output_dir}/{filename}")
    print(f"Standalone SHAP plot saved to {output_dir}/{shap_filename}")
    print(f"LOOCV Accuracy used: {original_results['cv_results']['xgb']['accuracy']:.4f}")
    return os.path.join(output_dir, filename)    # Create figure with subplots (2x3 layout)
    fig = plt.figure(figsize=(18, 12))
    
    # Panel A: Overall performance using ORIGINAL model
    ax1 = plt.subplot(2, 3, 1)
    y_pred = model.predict(X_array)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=[cn.replace('Class ', '') for cn in class_names],
               yticklabels=[cn.replace('Class ', '') for cn in class_names],
               ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_title("A. Model Performance", fontweight='bold')
    ax1.set_xlabel("Predicted")
    ax1.set_ylabel("Actual")
    
    # Panel B: Feature importance from ORIGINAL model
    ax2 = plt.subplot(2, 3, 2)
    importances = model.feature_importances_
    top_indices = np.argsort(importances)[-15:]
    top_features = [feature_names[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    short_names = [name.split('-')[0][:12] + "..." if len(name) > 15 else name 
                  for name in top_features]
    
    bars = ax2.barh(range(len(top_features)), top_importances,
                   color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
    ax2.set_yticks(range(len(top_features)))
    ax2.set_yticklabels(short_names, fontsize=label_size-1)
    ax2.set_xlabel("Importance")
    ax2.set_title("B. Top Genes (Original)", fontweight='bold')
    
    # Panel C: Class distribution
    ax3 = plt.subplot(2, 3, 3)
    class_counts = np.bincount(y, minlength=len(class_names))
    bars = ax3.bar(range(len(class_names)), class_counts, 
                  color=plt.cm.RdYlBu_r(np.linspace(0, 1, len(class_names))))
    ax3.set_xticks(range(len(class_names)))
    ax3.set_xticklabels([cn.replace('Class ', '') for cn in class_names], rotation=45)
    ax3.set_ylabel("Number of Strains")
    ax3.set_title("C. Dataset Distribution", fontweight='bold')
    
    # Panel D: Prediction confidence using ORIGINAL model
    ax4 = plt.subplot(2, 3, 4)
    y_prob = model.predict_proba(X_array)
    confidence = np.max(y_prob, axis=1)
    
    ax4.hist(confidence, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax4.axvline(np.mean(confidence), color='red', linestyle='--', 
               label=f'Mean: {np.mean(confidence):.2f}')
    ax4.set_xlabel("Prediction Confidence")
    ax4.set_ylabel("Frequency")
    ax4.set_title("D. Model Confidence", fontweight='bold')
    ax4.legend()
    
    # Panel E: Model comparison from ORIGINAL results
    ax5 = plt.subplot(2, 3, 5)
    
    # Load results from the model training
    results_path = os.path.join(model_dir, "results.pkl")
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            results = pickle.load(f)
        
        cv_results = results['cv_results']
        model_names = list(cv_results.keys())
        accuracies = [cv_results[model]['accuracy'] for model in model_names]
        f1_scores = [cv_results[model]['f1_weighted'] for model in model_names]
        
        x = np.arange(len(model_names))
        width = 0.35
        
        ax5.bar(x - width/2, accuracies, width, label='Accuracy', alpha=0.8)
        ax5.bar(x + width/2, f1_scores, width, label='F1-Score', alpha=0.8)
        
        ax5.set_xlabel('Model')
        ax5.set_ylabel('Score')
        ax5.set_title('E. Model Comparison', fontweight='bold')
        ax5.set_xticks(x)
        ax5.set_xticklabels([m.upper() for m in model_names])
        ax5.legend()
        ax5.set_ylim(0, 1)
    else:
        # Fallback: show a simple accuracy bar
        accuracy = (y_pred == y).mean()
        ax5.bar(['Overall'], [accuracy], color='green', alpha=0.7)
        ax5.set_ylabel('Accuracy')
        ax5.set_title('E. Model Accuracy', fontweight='bold')
        ax5.set_ylim(0, 1)
    
    # Panel F: Gene cluster importance or top genes from ORIGINAL analysis
    ax6 = plt.subplot(2, 3, 6)
    
    # Load gene clusters if available
    clusters_dir = os.path.join(model_dir, "gene_clusters")
    cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
    
    if os.path.exists(cluster_file):
        with open(cluster_file, 'rb') as f:
            cluster_results = pickle.load(f)
        
        if 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Get importance for the most pathogenic class (class 6 or 7)
            target_class = 6 if 6 in importance_data else 7 if 7 in importance_data else max(importance_data.keys())
            
            cluster_ids = list(importance_data[target_class].keys())
            cluster_importances = [importance_data[target_class][cid]['mean_abs_shap'] 
                                 for cid in cluster_ids]
            
            bars = ax6.bar([f'C{cid}' for cid in cluster_ids], cluster_importances,
                          color=plt.cm.plasma(np.linspace(0, 1, len(cluster_ids))))
            ax6.set_xlabel('Gene Cluster')
            ax6.set_ylabel('Importance')
            ax6.set_title('F. Gene Cluster Impact', fontweight='bold')
            ax6.tick_params(axis='x', rotation=45)
        else:
            # Fallback: show top feature importances in a different way
            top_5_indices = np.argsort(importances)[-5:]
            top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                          else feature_names[i] for i in top_5_indices]
            top_5_importances = importances[top_5_indices]
            
            ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
            ax6.set_title('F. Top 5 Genes', fontweight='bold')
    else:
        # Fallback: show top feature importances as pie chart
        top_5_indices = np.argsort(importances)[-5:]
        top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                      else feature_names[i] for i in top_5_indices]
        top_5_importances = importances[top_5_indices]
        
        ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
        ax6.set_title('F. Top 5 Genes', fontweight='bold')
    
    # Add main title
    fig.suptitle("Machine Learning-Based Prediction of NTM Pathogenicity from Genomic Data\n(Using Original Trained Models)", 
                fontsize=title_size+4, fontweight='bold', y=0.96)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    
    filename = "comprehensive_conference_figure_original.png"
    plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    # Now create a separate SHAP transition plot (still needs new model for transition)
    print("Creating separate SHAP transition plot...")
    
    # Create transition data for SHAP (this still needs a new binary model)
    from_class, to_class = 2, 4
    transition_mask = (y == from_class) | (y == to_class)
    X_transition = X_array[transition_mask]
    y_transition = y[transition_mask]
    y_binary = (y_transition == to_class).astype(int)
    
    # Train transition model (this is still needed for binary classification)
    transition_model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=5,
        tree_method='hist', device='cpu', objective='binary:logistic',
        random_state=42
    )
    transition_model.fit(X_transition, y_binary)
    
    # SHAP analysis for transition
    transition_explainer = shap.TreeExplainer(transition_model)
    n_samples = min(80, len(X_transition))
    sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
    X_sample = X_transition[sample_indices]
    shap_values = transition_explainer.shap_values(X_sample)
    
    # Create standalone SHAP plot
    plt.figure(figsize=(10, 8))
    short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                          for name in feature_names]
    
    shap.summary_plot(shap_values, X_sample, feature_names=short_feature_names,
                     max_display=20, show=False)
    plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
             fontsize=title_size+2, pad=20)
    plt.tight_layout()
    
    shap_filename = "transition_shap_standalone_original.png"
    plt.savefig(os.path.join(output_dir, shap_filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"Comprehensive conference figure saved to {output_dir}/{filename}")
    print(f"Standalone SHAP plot saved to {output_dir}/{shap_filename}")
    return os.path.join(output_dir, filename)

In [88]:
'''
def create_multi_panel_figure(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    dpi=300,
    style="publication"
):
    """
    Create a comprehensive multi-panel figure for conference presentation using ORIGINAL models
    """
    print("Creating comprehensive multi-panel conference figure using ORIGINAL models...")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set style
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 10
        title_size = 12
        label_size = 9
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 14
        title_size = 16
        label_size = 12
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size-1,
        'ytick.labelsize': label_size-1,
        'legend.fontsize': font_size-1,
    })
    
    # Load data
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load ORIGINAL trained models
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load the ORIGINAL explainers
    with open(os.path.join(model_dir, "shap_explainers.pkl"), 'rb') as f:
        explainers = pickle.load(f)
    
    with open(os.path.join(model_dir, "feature_names.pkl"), 'rb') as f:
        feature_names = pickle.load(f)
    
    with open(os.path.join(model_dir, "class_info.pkl"), 'rb') as f:
        class_info = pickle.load(f)
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    class_names = class_info['class_names']
    
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use ORIGINAL model and explainer
    model = models['xgb']
    explainer = explainers['xgb']
    
    # Create figure with subplots (2x3 layout without spanning subplot for SHAP)
    fig = plt.figure(figsize=(18, 12))
    
    # Panel A: Overall performance
    ax1 = plt.subplot(2, 3, 1)
    y_pred = model.predict(X_array)
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=[cn.replace('Class ', '') for cn in class_names],
               yticklabels=[cn.replace('Class ', '') for cn in class_names],
               ax=ax1, cbar_kws={'shrink': 0.8})
    ax1.set_title("A. Model Performance", fontweight='bold')
    ax1.set_xlabel("Predicted")
    ax1.set_ylabel("Actual")
    
    # Panel B: Feature importance
    ax2 = plt.subplot(2, 3, 2)
    importances = model.feature_importances_
    top_indices = np.argsort(importances)[-15:]
    top_features = [feature_names[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    short_names = [name.split('-')[0][:12] + "..." if len(name) > 15 else name 
                  for name in top_features]
    
    bars = ax2.barh(range(len(top_features)), top_importances,
                   color=plt.cm.viridis(np.linspace(0, 1, len(top_features))))
    ax2.set_yticks(range(len(top_features)))
    ax2.set_yticklabels(short_names, fontsize=label_size-1)
    ax2.set_xlabel("Importance")
    ax2.set_title("B. Top Genes", fontweight='bold')
    
    # Panel C: Class distribution
    ax3 = plt.subplot(2, 3, 3)
    class_counts = np.bincount(y, minlength=len(class_names))
    bars = ax3.bar(range(len(class_names)), class_counts, 
                  color=plt.cm.RdYlBu_r(np.linspace(0, 1, len(class_names))))
    ax3.set_xticks(range(len(class_names)))
    ax3.set_xticklabels([cn.replace('Class ', '') for cn in class_names], rotation=45)
    ax3.set_ylabel("Number of Strains")
    ax3.set_title("C. Dataset Distribution", fontweight='bold')
    
    # Panel D: Prediction confidence
    ax4 = plt.subplot(2, 3, 4)
    y_prob = model.predict_proba(X_array)
    confidence = np.max(y_prob, axis=1)
    
    ax4.hist(confidence, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax4.axvline(np.mean(confidence), color='red', linestyle='--', 
               label=f'Mean: {np.mean(confidence):.2f}')
    ax4.set_xlabel("Prediction Confidence")
    ax4.set_ylabel("Frequency")
    ax4.set_title("D. Model Confidence", fontweight='bold')
    ax4.legend()
    
    # Panel E: Model comparison
    ax5 = plt.subplot(2, 3, 5)
    
    # Load results from the model training
    results_path = os.path.join(model_dir, "results.pkl")
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            results = pickle.load(f)
        
        cv_results = results['cv_results']
        model_names = list(cv_results.keys())
        accuracies = [cv_results[model]['accuracy'] for model in model_names]
        f1_scores = [cv_results[model]['f1_weighted'] for model in model_names]
        
        x = np.arange(len(model_names))
        width = 0.35
        
        ax5.bar(x - width/2, accuracies, width, label='Accuracy', alpha=0.8)
        ax5.bar(x + width/2, f1_scores, width, label='F1-Score', alpha=0.8)
        
        ax5.set_xlabel('Model')
        ax5.set_ylabel('Score')
        ax5.set_title('E. Model Comparison', fontweight='bold')
        ax5.set_xticks(x)
        ax5.set_xticklabels([m.upper() for m in model_names])
        ax5.legend()
        ax5.set_ylim(0, 1)
    else:
        # Fallback: show a simple accuracy bar
        accuracy = (y_pred == y).mean()
        ax5.bar(['Overall'], [accuracy], color='green', alpha=0.7)
        ax5.set_ylabel('Accuracy')
        ax5.set_title('E. Model Accuracy', fontweight='bold')
        ax5.set_ylim(0, 1)
    
    # Panel F: Create a standalone SHAP plot and save it separately
    ax6 = plt.subplot(2, 3, 6)
    
    # For panel F, we'll show gene cluster importance instead of SHAP
    # Load gene clusters if available
    clusters_dir = os.path.join(model_dir, "gene_clusters")
    cluster_file = os.path.join(clusters_dir, "clustering_results.pkl")
    
    if os.path.exists(cluster_file):
        with open(cluster_file, 'rb') as f:
            cluster_results = pickle.load(f)
        
        if 'cluster_importance' in cluster_results:
            importance_data = cluster_results['cluster_importance']
            
            # Get importance for the most pathogenic class (class 6 or 7)
            target_class = 6 if 6 in importance_data else 7 if 7 in importance_data else max(importance_data.keys())
            
            cluster_ids = list(importance_data[target_class].keys())
            cluster_importances = [importance_data[target_class][cid]['mean_abs_shap'] 
                                 for cid in cluster_ids]
            
            bars = ax6.bar([f'C{cid}' for cid in cluster_ids], cluster_importances,
                          color=plt.cm.plasma(np.linspace(0, 1, len(cluster_ids))))
            ax6.set_xlabel('Gene Cluster')
            ax6.set_ylabel('Importance')
            ax6.set_title('F. Gene Cluster Impact', fontweight='bold')
            ax6.tick_params(axis='x', rotation=45)
        else:
            # Fallback: show top feature importances in a different way
            top_5_indices = np.argsort(importances)[-5:]
            top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                          else feature_names[i] for i in top_5_indices]
            top_5_importances = importances[top_5_indices]
            
            ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
            ax6.set_title('F. Top 5 Genes', fontweight='bold')
    else:
        # Fallback: show top feature importances as pie chart
        top_5_indices = np.argsort(importances)[-5:]
        top_5_names = [feature_names[i].split('-')[0][:8] + "..." if len(feature_names[i]) > 10 
                      else feature_names[i] for i in top_5_indices]
        top_5_importances = importances[top_5_indices]
        
        ax6.pie(top_5_importances, labels=top_5_names, autopct='%1.1f%%')
        ax6.set_title('F. Top 5 Genes', fontweight='bold')
    
    # Add main title
    fig.suptitle("Machine Learning-Based Prediction of NTM Pathogenicity from Genomic Data", 
                fontsize=title_size+4, fontweight='bold', y=0.96)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])
    
    filename = "comprehensive_conference_figure.png"
    plt.savefig(os.path.join(output_dir, filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    # Now create a separate SHAP transition plot
    print("Creating separate SHAP transition plot...")
    
    # Create transition data for SHAP
    from_class, to_class = 2, 4
    transition_mask = (y == from_class) | (y == to_class)
    X_transition = X_array[transition_mask]
    y_transition = y[transition_mask]
    y_binary = (y_transition == to_class).astype(int)
    
    # Train transition model
    transition_model = xgb.XGBClassifier(
        n_estimators=100, learning_rate=0.05, max_depth=5,
        tree_method='hist', device='cpu', objective='binary:logistic',
        random_state=42
    )
    transition_model.fit(X_transition, y_binary)
    
    # SHAP analysis
    explainer = shap.TreeExplainer(transition_model)
    n_samples = min(80, len(X_transition))
    sample_indices = np.random.choice(len(X_transition), n_samples, replace=False)
    X_sample = X_transition[sample_indices]
    shap_values = explainer.shap_values(X_sample)
    
    # Create standalone SHAP plot
    plt.figure(figsize=(10, 8))
    short_feature_names = [name.split('-')[0][:15] + "..." if len(name) > 18 else name 
                          for name in feature_names]
    
    shap.summary_plot(shap_values, X_sample, feature_names=short_feature_names,
                     max_display=20, show=False)
    plt.title(f"Gene Impact on Pathogenicity Transition\n{class_names[from_class]} → {class_names[to_class]}", 
             fontsize=title_size+2, pad=20)
    plt.tight_layout()
    
    shap_filename = "transition_shap_standalone.png"
    plt.savefig(os.path.join(output_dir, shap_filename), dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"Comprehensive conference figure saved to {output_dir}/{filename}")
    print(f"Standalone SHAP plot saved to {output_dir}/{shap_filename}")
    return os.path.join(output_dir, filename)
'''

'\ndef create_multi_panel_figure(\n    model_dir="./May/ntm_pathogenicity_model",\n    output_dir="./conference_figures",\n    dpi=300,\n    style="publication"\n):\n    """\n    Create a comprehensive multi-panel figure for conference presentation using ORIGINAL models\n    """\n    print("Creating comprehensive multi-panel conference figure using ORIGINAL models...")\n    \n    os.makedirs(output_dir, exist_ok=True)\n    \n    # Set style\n    if style == "publication":\n        plt.style.use(\'seaborn-v0_8-whitegrid\')\n        font_size = 10\n        title_size = 12\n        label_size = 9\n    elif style == "poster":\n        plt.style.use(\'seaborn-v0_8-whitegrid\')\n        font_size = 14\n        title_size = 16\n        label_size = 12\n    \n    plt.rcParams.update({\n        \'font.size\': font_size,\n        \'axes.titlesize\': title_size,\n        \'axes.labelsize\': label_size,\n        \'xtick.labelsize\': label_size-1,\n        \'ytick.labelsize\': label_size-1,\n      

In [89]:
# Example usage for different figure types
if __name__ == "__main__":
    model_dir = "./May/ntm_pathogenicity_model"
    output_dir = "./conference_figures"
    
    # Create different types of figures
    print("Creating conference figures...")
    
    # 1. Transition beeswarm (recommended for showing biological insight)
    create_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="transition_beeswarm",
        from_class=1,
        to_class=6,
        style="publication"
    )
    
    # 2. Overall beeswarm
    create_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="overall_beeswarm",
        style="publication"
    )
    
    # 3. Feature importance
    create_conference_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        figure_type="feature_importance",
        style="publication"
    )
    
    # 4. Comprehensive multi-panel figure
    create_multi_panel_figure(
        model_dir=model_dir,
        output_dir=output_dir,
        style="publication"
    )
    
    print("All conference figures created!")

Creating conference figures...
Loading original models, data, and CV results...
Using LOOCV predictions with accuracy: 0.6609
Creating transition beeswarm plot for classes 1 → 6...
Creating transition beeswarm plot for classes 1 → 6...
Conference figure saved to ./conference_figures/transition_beeswarm_1_to_6.png
Loading original models, data, and CV results...
Using LOOCV predictions with accuracy: 0.6609
Creating overall pathogenicity beeswarm plot using ORIGINAL explainer...
Creating overall pathogenicity beeswarm plot using ORIGINAL model...
Conference figure saved to ./conference_figures/overall_pathogenicity_beeswarm.png
Loading original models, data, and CV results...
Using LOOCV predictions with accuracy: 0.6609
Creating feature importance plot using ORIGINAL model...
Creating feature importance plot using ORIGINAL model...
Conference figure saved to ./conference_figures/feature_importance_top20_original.png
Creating comprehensive multi-panel conference figure using PROPER CV p

### Based on original figure

In [78]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import pickle
import os

def create_simple_feature_importance_beeswarm(
    model_dir="./May/ntm_pathogenicity_model",
    output_dir="./conference_figures",
    n_top_genes=25,
    n_samples=150,
    gene_font_size=10,
    shorten_gene_names=True,
    max_gene_name_length=20,
    dpi=300,
    style="publication",
    figure_size=(12, 10),
    random_seed=42
):
    """
    Create a simple beeswarm plot showing SHAP values for the top feature importance genes
    from the original trained model.
    
    Parameters:
    -----------
    model_dir : str
        Directory containing trained models
    output_dir : str
        Directory to save figures
    n_top_genes : int
        Number of top genes to display (based on feature importance)
    n_samples : int
        Number of samples to use for SHAP calculation
    gene_font_size : int
        Font size for gene names on y-axis
    shorten_gene_names : bool
        Whether to shorten gene names (takes text before first '~~~')
    max_gene_name_length : int
        Maximum length for gene names (adds '...' if longer)
    dpi : int
        Resolution for saved figures
    style : str
        Figure style ('publication', 'poster', 'presentation')
    figure_size : tuple
        Figure size (width, height)
    random_seed : int
        Random seed for reproducible sampling
        
    Returns:
    --------
    str: Path to saved figure
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Set matplotlib style based on figure purpose
    if style == "publication":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 12
        title_size = 14
        label_size = 10
    elif style == "poster":
        plt.style.use('seaborn-v0_8-whitegrid')
        font_size = 16
        title_size = 20
        label_size = 14
    else:  # presentation
        plt.style.use('seaborn-v0_8-darkgrid')
        font_size = 14
        title_size = 18
        label_size = 12
    
    plt.rcParams.update({
        'font.size': font_size,
        'axes.titlesize': title_size,
        'axes.labelsize': label_size,
        'xtick.labelsize': label_size,
        'ytick.labelsize': label_size-1,
        'legend.fontsize': font_size,
        'figure.titlesize': title_size + 2
    })
    
    print("Loading original models, data, and results...")
    
    # Load the ORIGINAL results
    results_path = os.path.join(model_dir, "results.pkl")
    with open(results_path, 'rb') as f:
        original_results = pickle.load(f)
    
    # Load the dataset
    dataset_path = os.path.join(model_dir, "../ntm_pathogenicity_analysis/selected_genes_dataset.csv")
    df = pd.read_csv(dataset_path, index_col=0)
    
    # Load trained models
    with open(os.path.join(model_dir, "trained_models.pkl"), 'rb') as f:
        models = pickle.load(f)
    
    # Load the ORIGINAL explainers
    with open(os.path.join(model_dir, "shap_explainers.pkl"), 'rb') as f:
        explainers = pickle.load(f)
    
    # Get data from original results
    feature_names = original_results['feature_names']
    class_names = original_results['class_names']
    
    # Prepare data
    target_column = "Pathogenicity_scale"
    X = df[feature_names]
    y = df[target_column].values
    
    # Convert sparse to dense if needed
    if hasattr(X, 'sparse'):
        X_array = X.sparse.to_dense().values
    else:
        X_array = X.values
    
    # Use ORIGINAL XGBoost model and explainer
    model = models['catboost']
    explainer = explainers['catboost']
    
    print(f"Getting feature importance from original model...")
    
    # Get feature importances from the ORIGINAL model
    importances = model.feature_importances_
    
    # Get top N features based on importance
    top_indices = np.argsort(importances)[-n_top_genes:]
    top_feature_names = [feature_names[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    print(f"Top {n_top_genes} genes by feature importance:")
    for i, (name, imp) in enumerate(zip(reversed(top_feature_names), reversed(top_importances))):
        print(f"{i+1:2d}. {name[:50]}... (importance: {imp:.4f})")
    
    # Create shortened gene names if requested
    if shorten_gene_names:
        display_names = []
        for name in top_feature_names:
            # Take text before first '~~~' if it exists
            if '~~~' in name:
                short_name = name.split('~~~')[0]
            else:
                short_name = name
            
            # Further shorten if still too long
            if len(short_name) > max_gene_name_length:
                short_name = short_name[:max_gene_name_length] + "..."
            
            display_names.append(short_name)
    else:
        # Just truncate long names
        display_names = []
        for name in top_feature_names:
            if len(name) > max_gene_name_length:
                display_names.append(name[:max_gene_name_length] + "...")
            else:
                display_names.append(name)
    
    print(f"\nUsing SHAP explainer to analyze {n_samples} samples...")
    
    # Sample data for SHAP analysis (reproducible sampling)
    np.random.seed(random_seed)
    if n_samples < len(X_array):
        sample_indices = np.random.choice(len(X_array), n_samples, replace=False)
        X_sample = X_array[sample_indices]
        y_sample = y[sample_indices]
    else:
        X_sample = X_array
        y_sample = y
        sample_indices = np.arange(len(X_array))
    
    print(f"Sample class distribution: {np.bincount(y_sample, minlength=len(class_names))}")
    
    # Calculate SHAP values using the original explainer
    shap_values = explainer.shap_values(X_sample)
    
    # Create subset of SHAP values for only the top genes
    if isinstance(shap_values, list):
        # Multi-class case - we'll show all classes but only top genes
        print(f"Multi-class model detected ({len(shap_values)} classes)")
        
        # Create a figure
        plt.figure(figsize=figure_size)
        
        # For multi-class, SHAP summary_plot expects the full shap_values list
        # We need to subset the features, not the classes
        
        # Create subset data for top features only
        X_sample_subset = X_sample[:, top_indices]
        
        # Create subset of shap values for top features only
        shap_values_subset = []
        for class_shap in shap_values:
            shap_values_subset.append(class_shap[:, top_indices])
        
        # Create the beeswarm plot
        shap.summary_plot(
            shap_values_subset,
            X_sample_subset,
            feature_names=display_names,
            max_display=n_top_genes,
            show=False,
            plot_size=None  # Let matplotlib handle the size
        )
        
        # Customize the plot
        plt.title(f"Gene Impact on NTM Pathogenicity\n(Top {n_top_genes} Genes by Feature Importance)", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Model Output)", fontsize=label_size)
        
        # Adjust y-axis font size for gene names
        ax = plt.gca()
        ax.tick_params(axis='y', labelsize=gene_font_size)
        
        # Add a note about the model
        plt.figtext(0.02, 0.02, f"XGBoost model | {n_samples} samples | Original trained model", 
                   fontsize=font_size-2, style='italic', alpha=0.7)
        
    else:
        # Binary classification case
        print("Binary classification model detected")
        
        plt.figure(figsize=figure_size)
        
        # Create subset for top features only
        X_sample_subset = X_sample[:, top_indices]
        shap_values_subset = shap_values[:, top_indices]
        
        # Create the beeswarm plot
        shap.summary_plot(
            shap_values_subset,
            X_sample_subset,
            feature_names=display_names,
            max_display=n_top_genes,
            show=False,
            plot_size=None
        )
        
        # Customize the plot
        plt.title(f"Gene Impact on NTM Pathogenicity\n(Top {n_top_genes} Genes by Feature Importance)", 
                 fontsize=title_size, pad=20)
        plt.xlabel("SHAP Value (Impact on Model Output)", fontsize=label_size)
        
        # Adjust y-axis font size for gene names
        ax = plt.gca()
        ax.tick_params(axis='y', labelsize=gene_font_size)
        
        # Add a note about the model
        plt.figtext(0.02, 0.02, f"XGBoost model | {n_samples} samples | Original trained model", 
                   fontsize=font_size-2, style='italic', alpha=0.7)
    
    plt.tight_layout()
    
    # Save the figure
    filename = f"feature_importance_beeswarm_top{n_top_genes}.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=dpi, bbox_inches='tight')
    plt.close()
    
    print(f"\nBeeswarm plot saved to: {filepath}")
    print(f"Features displayed: {len(display_names)}")
    print(f"Gene name shortening: {'Enabled' if shorten_gene_names else 'Disabled'}")
    print(f"Gene font size: {gene_font_size}")
    
    # Also save a summary of the top genes
    summary_df = pd.DataFrame({
        'Rank': range(1, len(top_feature_names) + 1),
        'Full_Gene_Name': reversed(top_feature_names),
        'Display_Name': reversed(display_names),
        'Feature_Importance': reversed(top_importances)
    })
    
    summary_path = os.path.join(output_dir, f"top{n_top_genes}_genes_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"Gene summary saved to: {summary_path}")
    
    return filepath

# Example usage functions for different scenarios
def create_conference_beeswarm_publication(model_dir="./May/ntm_pathogenicity_model"):
    """Create a publication-ready beeswarm plot"""
    return create_simple_feature_importance_beeswarm(
        model_dir=model_dir,
        output_dir="./conference_figures",
        n_top_genes=20,
        n_samples=150,
        gene_font_size=11,
        shorten_gene_names=True,
        max_gene_name_length=25,
        style="publication",
        figure_size=(10, 8)
    )

def create_conference_beeswarm_poster(model_dir="./May/ntm_pathogenicity_model"):
    """Create a poster-ready beeswarm plot with larger fonts"""
    return create_simple_feature_importance_beeswarm(
        model_dir=model_dir,
        output_dir="./conference_figures",
        n_top_genes=15,
        n_samples=120,
        gene_font_size=14,
        shorten_gene_names=True,
        max_gene_name_length=20,
        style="poster",
        figure_size=(12, 10)
    )

def create_conference_beeswarm_full_names(model_dir="./May/ntm_pathogenicity_model"):
    """Create a beeswarm plot with full gene names (no shortening)"""
    return create_simple_feature_importance_beeswarm(
        model_dir=model_dir,
        output_dir="./conference_figures",
        n_top_genes=15,
        n_samples=150,
        gene_font_size=9,
        shorten_gene_names=False,
        max_gene_name_length=50,
        style="publication",
        figure_size=(14, 10)
    )

# Main execution for testing
if __name__ == "__main__":
    model_dir = "./May/ntm_pathogenicity_model"
    
    print("Creating simple feature importance beeswarm plot...")
    
    # Create a publication-ready version
    filepath = create_conference_beeswarm_publication(model_dir)
    
    print(f"\nDone! Check the output at: {filepath}")

Creating simple feature importance beeswarm plot...
Loading original models, data, and results...
Getting feature importance from original model...
Top 20 genes by feature importance:
 1. sp-P9WGH9-SIGH_MYCTU~~~sp-P66808-SIGH_MYCBO~~~sp-A... (importance: 3.2777)
 2. sp-A0R596-CBXPD_MYCS2~~~MF_00384-sp-Q744A3-TILS_MY... (importance: 3.0008)
 3. sp-Q7VEN0-Y2274_MYCBO~~~sp-P9WMC5-Y2250_MYCTU~~~sp... (importance: 2.9851)
 4. sp-P9WFP3-Y1842_MYCTU~~~sp-P9WFP2-Y1842_MYCTO-sp-P... (importance: 2.8363)
 5. sp-O69460-RECG_MYCLE~~~sp-P9WMQ7-RECG_MYCTU~~~sp-P... (importance: 2.7775)
 6. group_11817-sp-A0R4Z9-IPDE1_MYCS2~~~sp-I6Y3V5-IPDE... (importance: 2.7160)
 7. sp-A0R4C9-THTR_MYCS2~~~sp-P59989-THTR1_MYCBO~~~sp-... (importance: 2.4862)
 8. sp-P9WN80-GLPD1_MYCTO~~~sp-P64183-GLPD1_MYCBO~~~sp... (importance: 2.2890)
 9. MF_00831~~~sp-P44839-Y719_HAEIN~~~sp-Q9L6B5-Y1466_... (importance: 2.0166)
10. group_24800-sp-P63510-LAT_MYCBO~~~sp-P9WQ77-LAT_MY... (importance: 2.0138)
11. sp-Q89RB7-OAT_BRADU~~~